<a href="https://colab.research.google.com/github/chavezaltamirano-ui/Growth-Models-in-Comparative/blob/main/4Growth_Models_in_Comparative_Perspective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

os.makedirs("data_raw", exist_ok=True)
os.makedirs("data_clean", exist_ok=True)

print("data_raw:", os.listdir("data_raw"))
print("data_clean:", os.listdir("data_clean"))

data_raw: []
data_clean: []


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving generar_cuadros_apa.py to generar_cuadros_apa.py


In [ ]:
import os
import shutil

os.makedirs("data_raw", exist_ok=True)
os.makedirs("data_clean", exist_ok=True)

for fname in uploaded.keys():
    # Panel combinado lo dejamos en data_clean
    if fname == "generar_cuadros_apa.py":
        dst = os.path.join("data_clean", fname)
    else:
        # Bloques fuente en data_raw
        dst = os.path.join("data_raw", fname)

    shutil.move(fname, dst)
    print("Movido a", dst)

print("\nContenido de data_raw:", os.listdir("data_raw"))
print("Contenido de data_clean:", os.listdir("data_clean"))

FileNotFoundError: [Errno 2] No such file or directory: 'generar_cuadros_apa.py'

In [ ]:
!ls -la data_clean/china_us_super_panel_1990_2025.csv

-rw-r--r-- 1 root root 24443 Jul 30 22:15 data_clean/china_us_super_panel_1990_2025.csv


In [ ]:
!find / -name "china_us_super_panel_1990_2025.csv" 2>/dev/null

/content/data_raw/china_us_super_panel_1990_2025.csv
/content/data_clean/china_us_super_panel_1990_2025.csv


In [ ]:
!find / -name "generar_cuadros_apa.py" 2>/dev/null

/content/data_clean/generar_cuadros_apa.py


In [ ]:
!pip -q install python-docx statsmodels arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 25.3 MB/s eta 0:00:00


In [ ]:
print("Listo")

Listo


In [ ]:
import docx
import statsmodels
import arch

print("Todas las librerías están instaladas correctamente")

Todas las librerías están instaladas correctamente


In [ ]:
!python /content/generar_cuadros_apa.py

python3: can't open file '/content/generar_cuadros_apa.py': [Errno 2] No such file or directory


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving generar_cuadros_apa.py to generar_cuadros_apa.py


In [ ]:
import os

ruta = "/content/generar_cuadros_apa.py"
print(os.path.exists(ruta))

True


In [ ]:
!python /content/generar_cuadros_apa.py

Panel cargado: 68 observaciones x 36 variables (1990-2023)
Cuadro 1 listo: cobertura y disponibilidad
Cuadro 2 listo: estadistica descriptiva
Cuadro 3 listo: pruebas de raiz unitaria
Cuadros 4, 5 y 6 listos: correlaciones y VIF

Documento Word generado en: /content/outputs/cuadros_previos_apa.docx
Tablas auxiliares en CSV dentro de: /content/outputs/


In [ ]:
from google.colab import files
files.download("/content/outputs/cuadros_previos_apa.docx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# -*- coding: utf-8 -*-
"""
REPARACION Y AUDITORIA DE SERIES DEL PANEL MAESTRO
Proyecto: modelos de crecimiento comparados China - Estados Unidos, 1990-2023

NO modifica el panel maestro. Solo lo lee y escribe archivos derivados.

Salidas en /content/data_clean:
    china_us_super_panel_reparado.csv   panel con las series reconstruidas
    bitacora_reparacion.csv            registro de decisiones (cuadro 5)
    cobertura_series_nuevas.csv         cobertura de lo reconstruido

Uso en Colab:
    !python /content/reparar_series.py
"""

import json, os, time, urllib.request
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- CONFIGURACION
PANEL_MAESTRO = "/content/data_clean/china_us_super_panel_1990_2025.csv"
OUTDIR        = "/content/data_clean"
ANIO_INI, ANIO_FIN = 1990, 2024
ISO3 = {"CN": "CHN", "US": "USA"}

INDICADORES = {
    "va_agr_pct":     "NV.AGR.TOTL.ZS",   # VA agricola, % del PIB
    "va_ind_pct":     "NV.IND.TOTL.ZS",   # VA industrial, % del PIB
    "va_srv_pct":     "NV.SRV.TOTL.ZS",   # VA servicios, % del PIB
    "emp_agr_pct":    "SL.AGR.EMPL.ZS",   # empleo agricola, % del total
    "emp_ind_pct":    "SL.IND.EMPL.ZS",   # empleo industrial, % del total
    "emp_srv_pct":    "SL.SRV.EMPL.ZS",   # empleo servicios, % del total
    "fuerza_laboral": "SL.TLF.TOTL.IN",
    "desempleo_pct":  "SL.UEM.TOTL.ZS",
    "pib_real_usd":   "NY.GDP.MKTP.KD",   # PIB, USD constantes 2015
}

URL_WB = ("https://api.worldbank.org/v2/country/{iso}/indicator/{ind}"
          "?date={ini}:{fin}&format=json&per_page=500")
REINTENTOS, ESPERA = 3, 3


# ------------------------------------------------------------------- DESCARGA
def descargar_wb(iso, indicador):
    url = URL_WB.format(iso=iso, ind=indicador, ini=ANIO_INI, fin=ANIO_FIN)
    for intento in range(1, REINTENTOS + 1):
        try:
            with urllib.request.urlopen(url, timeout=60) as resp:
                datos = json.loads(resp.read().decode("utf-8"))
            if not isinstance(datos, list) or len(datos) < 2 or datos[1] is None:
                return pd.DataFrame(columns=["year", "value"])
            df = pd.DataFrame([(int(x["date"]), x["value"]) for x in datos[1]],
                              columns=["year", "value"])
            df["value"] = pd.to_numeric(df["value"], errors="coerce")
            return df.sort_values("year").reset_index(drop=True)
        except Exception as e:
            if intento == REINTENTOS:
                print("    fallo definitivo {} / {}: {}".format(iso, indicador, e))
                return pd.DataFrame(columns=["year", "value"])
            time.sleep(ESPERA)
    return pd.DataFrame(columns=["year", "value"])


def descargar_componentes():
    registros = []
    for cod, iso in ISO3.items():
        print("  Componentes de {} ({})".format(cod, iso))
        for nombre, indicador in INDICADORES.items():
            df = descargar_wb(iso, indicador)
            n = int(df["value"].notna().sum()) if len(df) else 0
            print("    {:16s} {:18s} n = {}".format(nombre, indicador, n))
            if n == 0:
                continue
            df = df.dropna(subset=["value"]).copy()
            df["country"], df["serie"] = cod, nombre
            registros.append(df[["country", "year", "serie", "value"]])
    if not registros:
        raise RuntimeError("No se descargo ningun componente. Revisa la conexion.")
    largo = pd.concat(registros, ignore_index=True)
    ancho = largo.pivot_table(index=["country", "year"], columns="serie",
                              values="value", aggfunc="first").reset_index()
    ancho.columns.name = None
    return ancho


# ------------------------------------------- RECONSTRUCCION DE PRODUCTIVIDAD
def reconstruir_productividad(comp):
    """VA por ocupado sectorial, USD constantes 2015, metodo identico CN / US."""
    df = comp.copy()
    for r in ["fuerza_laboral", "desempleo_pct", "pib_real_usd"]:
        if r not in df.columns:
            df[r] = np.nan

    df["ocupados_totales"] = df["fuerza_laboral"] * (1 - df["desempleo_pct"] / 100.0)

    for s in ["agr", "ind", "srv"]:
        cva, cemp = "va_{}_pct".format(s), "emp_{}_pct".format(s)
        for c in (cva, cemp):
            if c not in df.columns:
                df[c] = np.nan
        va_s  = (df[cva] / 100.0) * df["pib_real_usd"]
        emp_s = (df[cemp] / 100.0) * df["ocupados_totales"]
        with np.errstate(divide="ignore", invalid="ignore"):
            df["{}_va_per_worker_rec".format(s)] = np.where(
                (emp_s > 0) & np.isfinite(emp_s), va_s / emp_s, np.nan)

    # brechas relativas de productividad: indicadores de cambio estructural
    df["ratio_ind_agr_rec"] = df["ind_va_per_worker_rec"] / df["agr_va_per_worker_rec"]
    df["ratio_srv_ind_rec"] = df["srv_va_per_worker_rec"] / df["ind_va_per_worker_rec"]

    nuevas = ["agr_va_per_worker_rec", "ind_va_per_worker_rec",
              "srv_va_per_worker_rec", "ratio_ind_agr_rec",
              "ratio_srv_ind_rec", "ocupados_totales"]
    return df[["country", "year"] + nuevas], nuevas


# -------------------------------------------------------------------- AUDITORIA
def cobertura(df, variables, etiqueta=""):
    filas, anios = [], int(df["year"].max() - df["year"].min() + 1)
    for v in variables:
        if v not in df.columns:
            continue
        for pais in ["CN", "US"]:
            sub = df.loc[df["country"] == pais, ["year", v]].dropna()
            if len(sub) == 0:
                filas.append([etiqueta, v, pais, "-", "-", 0, 100.0])
            else:
                filas.append([etiqueta, v, pais, int(sub["year"].min()),
                              int(sub["year"].max()), len(sub),
                              round(100.0 * (1 - len(sub) / float(anios)), 1)])
    return pd.DataFrame(filas, columns=["Conjunto", "Variable", "Pais",
                                        "Anio_inicio", "Anio_fin", "n",
                                        "Pct_faltantes"])


# ------------------------------------------- BITACORA DE TRATAMIENTO (CUADRO 5)
BITACORA_FILAS = [
 ["inv_gfcf_const_usd", "China",
  "Serie con una sola observacion (2015).",
  "Variable excluida; se conserva inv_gfcf_gdp.",
  "El WDI no publica NE.GDI.FTOT.KD para China fuera del ano base 2015 ni "
  "NE.GDI.FTOT.KN en moneda local constante (verificado en la API). La FBCF "
  "como porcentaje del PIB cubre 1990-2023 sin interrupciones y es la medida "
  "usada en la literatura de regimenes de crecimiento."],

 ["agr/ind/srv_va_per_worker", "Estados Unidos",
  "Series con una sola observacion (2015).",
  "Series reconstruidas con metodo identico para ambos paises (sufijo _rec).",
  "NV.*.EMPL.KD solo reporta el ano base para Estados Unidos. Se reconstruye el "
  "VA por ocupado a partir de la participacion sectorial en el valor agregado, "
  "el PIB real en USD constantes de 2015, la participacion sectorial en el "
  "empleo y los ocupados estimados. El mismo procedimiento en los dos paises "
  "asegura comparabilidad. Cobertura efectiva: 1997-2021."],

 ["gov_debt_gdp", "Ambos",
  "Solo 7 observaciones (2017-2023).",
  "Variable excluida del analisis econometrico.",
  "La serie del WEO incorporada al panel no cubre el periodo. La deuda publica "
  "se mide con pub_debt_gdp de la Global Debt Database del FMI, conforme a la "
  "jerarquia de fuentes declarada."],

 ["hh_debt_gdp / corp_debt_gdp / priv_debt_gdp", "China",
  "Cero observaciones en el periodo.",
  "Bloque financiero de China operacionalizado con bis_tot_credit_gdp y "
  "bis_pvt_credit_gdp; desagregacion sectorial reservada a Estados Unidos.",
  "La Global Debt Database del FMI cubre la deuda privada sectorial de China "
  "continental solo desde 2006 y el BIS publica credito a hogares y a "
  "sociedades no financieras de China desde el primer trimestre de 2006. "
  "Limitacion de fuentes, no decision analitica."],

 ["credit_finsec_gdp", "China",
  "Cero observaciones.",
  "Variable excluida; el bloque financiero se mide con series del BIS.",
  "El indicador de activos del sistema financiero del WDI no esta disponible "
  "para China en el periodo de estudio."],

 ["bis_hh_credit_gdp / bis_nfc_credit_gdp", "China",
  "17 observaciones (desde 2005).",
  "Uso restringido a estadistica descriptiva y subperiodos; excluidas del ARDL.",
  "Con 17 observaciones anuales no es posible especificar un modelo "
  "autorregresivo con rezagos distribuidos y controles."],

 ["Series del BIS (todas)", "Ambos",
  "Cobertura que termina en 2021.",
  "Las especificaciones con credito del BIS se estiman sobre 1990-2021, y sobre "
  "1994-2021 para el credito total de China.",
  "La muestra efectiva se reporta en cada modelo para evitar comparaciones "
  "entre periodos distintos."],

 ["labour_prod; pub_debt_gdp (CN); hh_debt_gdp (US)", "Ambos",
  "Clasificadas I(2) con especificacion de solo constante.",
  "Reestimacion con constante y tendencia, logaritmos y prueba de quiebre "
  "estructural endogeno.",
  "En series con tendencia determinista la especificacion sin tendencia esta "
  "mal identificada. La discrepancia entre ADF y Phillips-Perron en la deuda "
  "publica de China sugiere quiebre estructural, no integracion de orden dos."],
]


# ------------------------------------------------------------------- EJECUCION
def main(panel_maestro=PANEL_MAESTRO, outdir=OUTDIR):
    os.makedirs(outdir, exist_ok=True)
    f_panel = os.path.join(outdir, "china_us_super_panel_reparado.csv")
    f_bit   = os.path.join(outdir, "bitacora_reparacion.csv")
    f_cob   = os.path.join(outdir, "cobertura_series_nuevas.csv")

    if not os.path.exists(panel_maestro):
        raise FileNotFoundError("No se encontro el panel: " + panel_maestro)

    print("1. Leyendo el panel maestro (solo lectura)")
    master = pd.read_csv(panel_maestro)
    master.columns = [str(c).strip() for c in master.columns]
    master["country"] = master["country"].astype(str).str.strip().str.upper()
    master["year"] = pd.to_numeric(master["year"], errors="coerce").astype(int)
    print("   {} observaciones x {} variables".format(*master.shape))

    print("\n2. Descargando componentes del Banco Mundial")
    comp = descargar_componentes()

    print("\n3. Reconstruyendo la productividad sectorial")
    rec, nuevas = reconstruir_productividad(comp)

    print("\n4. Integrando en un panel derivado")
    panel = master.merge(rec, on=["country", "year"], how="left")
    excluir = [c for c in ["gov_debt_gdp", "inv_gfcf_const_usd",
                           "credit_finsec_gdp"] if c in panel.columns]
    panel = panel.rename(columns={c: c + "_excluida" for c in excluir})
    panel.to_csv(f_panel, index=False)
    print("   {}  ->  {} obs x {} vars".format(f_panel, *panel.shape))

    print("\n5. Cobertura de las series reconstruidas")
    cob = cobertura(panel, nuevas, "Series reconstruidas")
    cob.to_csv(f_cob, index=False)
    print(cob.to_string(index=False))

    print("\n6. Bitacora de tratamiento")
    bit = pd.DataFrame(BITACORA_FILAS, columns=["Variable", "Pais", "Incidencia",
                                                "Decision", "Justificacion"])
    bit.to_csv(f_bit, index=False)
    print("   {} ({} registros)".format(f_bit, len(bit)))

    print("\nListo. El panel maestro NO fue modificado.")
    return f_panel


if __name__ == "__main__":
    main()

1. Leyendo el panel maestro (solo lectura)
   68 observaciones x 36 variables

2. Descargando componentes del Banco Mundial
  Componentes de CN (CHN)
    va_agr_pct       NV.AGR.TOTL.ZS     n = 35
    va_ind_pct       NV.IND.TOTL.ZS     n = 35
    va_srv_pct       NV.SRV.TOTL.ZS     n = 35
    emp_agr_pct      SL.AGR.EMPL.ZS     n = 34
    emp_ind_pct      SL.IND.EMPL.ZS     n = 34
    emp_srv_pct      SL.SRV.EMPL.ZS     n = 34
    fuerza_laboral   SL.TLF.TOTL.IN     n = 35
    desempleo_pct    SL.UEM.TOTL.ZS     n = 34
    pib_real_usd     NY.GDP.MKTP.KD     n = 35
  Componentes de US (USA)
    va_agr_pct       NV.AGR.TOTL.ZS     n = 25
    va_ind_pct       NV.IND.TOTL.ZS     n = 25
    va_srv_pct       NV.SRV.TOTL.ZS     n = 25
    emp_agr_pct      SL.AGR.EMPL.ZS     n = 34
    emp_ind_pct      SL.IND.EMPL.ZS     n = 34
    emp_srv_pct      SL.SRV.EMPL.ZS     n = 34
    fuerza_laboral   SL.TLF.TOTL.IN     n = 35
    desempleo_pct    SL.UEM.TOTL.ZS     n = 34
    pib_real_usd     NY.G

In [ ]:
# -*- coding: utf-8 -*-
"""
CONSTRUCCION DEL PANEL DE ANALISIS
Proyecto: modelos de crecimiento comparados China - Estados Unidos, 1990-2023

Cadena de archivos:
    china_us_super_panel_1990_2025.csv   origen inmutable  (NO se toca)
    china_us_super_panel_reparado.csv    intermedio        (entrada de este script)
    china_us_panel_analisis_v1_1990_2023.csv   salida      (base de estimacion)
    diccionario_variables_v1.csv               salida      (apendice del articulo)

Transformaciones que aplica:
    1. Recorta el periodo a 1990-2023 y elimina las columnas excluidas.
    2. Logaritmos naturales de las variables de nivel.
    3. Primeras diferencias por pais, sin contaminacion entre series.
    4. Rezagos t-1 y t-2 de los regresores de los modelos M1 a M9.
    5. Indice de Intensidad de Capital (IIC): subindice inversor menos
       subindice financiero, ambos estandarizados dentro de cada pais.
    6. Indicadoras de subperiodo: pre-2000, 2000-2008, 2009-2015, post-2015.

Uso en Colab:
    !python /content/construir_panel_analisis.py
"""

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ RUTAS
ENTRADA = "/content/data_clean/china_us_super_panel_reparado.csv"
OUTDIR  = "/content/data_clean"
VERSION = "v1"
SALIDA  = os.path.join(OUTDIR, "china_us_panel_analisis_{}_1990_2023.csv".format(VERSION))
DICCIO  = os.path.join(OUTDIR, "diccionario_variables_{}.csv".format(VERSION))

ANIO_INI, ANIO_FIN = 1990, 2023

# Variables de nivel a las que se aplica logaritmo natural
LOGARITMAR = [
    "labour_prod", "rtfpna", "rkna", "gdp_const_usd", "gdp_pc_const_usd",
    "ocupados_totales", "agr_va_per_worker_rec", "ind_va_per_worker_rec",
    "srv_va_per_worker_rec",
]

# Variables que se diferencian y se rezagan (regresores de M1 a M9)
NUCLEO = [
    "gdp_growth", "inv_gfcf_gdp", "trade_openness", "cpi_inflation",
    "urban_pop_pct", "pub_debt_gdp", "hh_debt_gdp", "corp_debt_gdp",
    "bis_tot_credit_gdp", "bis_pvt_credit_gdp", "bis_hh_credit_gdp",
    "bis_nfc_credit_gdp", "ratio_ind_agr_rec", "ratio_srv_ind_rec",
]

# Componentes del Indice de Intensidad de Capital
IIC_INVERSOR  = ["inv_gfcf_gdp", "ratio_ind_agr_rec"]
IIC_FINANCIERO = ["bis_pvt_credit_gdp", "pub_debt_gdp"]

REZAGOS = [1, 2]


def zeta_por_pais(df, col):
    """Estandariza dentro de cada pais, con desviacion muestral."""
    def z(s):
        sd = s.std(ddof=1)
        return (s - s.mean()) / sd if sd and np.isfinite(sd) and sd > 0 else np.nan
    return df.groupby("country")[col].transform(z)


def main(entrada=ENTRADA, outdir=OUTDIR):
    if not os.path.exists(entrada):
        raise FileNotFoundError("No se encontro el panel reparado: " + entrada)
    os.makedirs(outdir, exist_ok=True)

    print("1. Leyendo el panel reparado")
    df = pd.read_csv(entrada)
    df.columns = [str(c).strip() for c in df.columns]
    df["country"] = df["country"].astype(str).str.strip().str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    print("   entrada: {} obs x {} vars".format(*df.shape))

    print("\n2. Recortando periodo y eliminando columnas excluidas")
    df = df[(df["year"] >= ANIO_INI) & (df["year"] <= ANIO_FIN)].copy()
    excluidas = [c for c in df.columns if c.endswith("_excluida")]
    df = df.drop(columns=excluidas)
    df = df.sort_values(["country", "year"]).reset_index(drop=True)
    print("   periodo {}-{}; {} columnas excluidas eliminadas".format(
        ANIO_INI, ANIO_FIN, len(excluidas)))

    registro = []   # filas del diccionario de variables

    print("\n3. Logaritmos naturales")
    n_log = 0
    for c in LOGARITMAR:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        df["ln_" + c] = np.where(s > 0, np.log(s), np.nan)
        registro.append(["ln_" + c, "Logaritmo natural de " + c,
                         "Transformacion", "ln(x) para x > 0"])
        n_log += 1
    print("   {} series transformadas".format(n_log))

    print("\n4. Primeras diferencias por pais")
    n_dif = 0
    candidatas = [c for c in NUCLEO if c in df.columns]
    candidatas += [c for c in df.columns if c.startswith("ln_")]
    for c in candidatas:
        df["d_" + c] = df.groupby("country")[c].diff()
        registro.append(["d_" + c, "Primera diferencia de " + c,
                         "Transformacion", "x(t) - x(t-1), dentro de cada pais"])
        n_dif += 1
    print("   {} series diferenciadas".format(n_dif))

    print("\n5. Rezagos t-1 y t-2")
    n_rez = 0
    for c in [x for x in NUCLEO if x in df.columns]:
        for k in REZAGOS:
            nombre = "{}_lag{}".format(c, k)
            df[nombre] = df.groupby("country")[c].shift(k)
            registro.append([nombre, "Rezago de orden {} de {}".format(k, c),
                             "Transformacion", "x(t-{}), dentro de cada pais".format(k)])
            n_rez += 1
    print("   {} variables rezagadas".format(n_rez))

    print("\n6. Indice de Intensidad de Capital (IIC)")
    inv = [c for c in IIC_INVERSOR if c in df.columns]
    fin = [c for c in IIC_FINANCIERO if c in df.columns]
    if inv and fin:
        for c in inv + fin:
            df["z_" + c] = zeta_por_pais(df, c)
        df["sub_inversor"]   = df[["z_" + c for c in inv]].mean(axis=1)
        df["sub_financiero"] = df[["z_" + c for c in fin]].mean(axis=1)
        df["IIC"] = df["sub_inversor"] - df["sub_financiero"]
        for k in REZAGOS:
            df["IIC_lag{}".format(k)] = df.groupby("country")["IIC"].shift(k)
        registro += [
            ["sub_inversor", "Subindice inversor: promedio de " + ", ".join(inv),
             "Indice compuesto", "Componentes estandarizados dentro de cada pais"],
            ["sub_financiero", "Subindice financiero: promedio de " + ", ".join(fin),
             "Indice compuesto", "Componentes estandarizados dentro de cada pais"],
            ["IIC", "Indice de Intensidad de Capital",
             "Indice compuesto", "sub_inversor menos sub_financiero"],
        ]
        for pais in ["CN", "US"]:
            sub = df.loc[df["country"] == pais, ["year", "IIC"]].dropna()
            if len(sub):
                print("   {}: {}-{}, n = {}".format(pais, int(sub['year'].min()),
                                                    int(sub['year'].max()), len(sub)))
            else:
                print("   {}: sin observaciones validas".format(pais))
    else:
        print("   omitido: faltan componentes en el panel")

    print("\n7. Indicadoras de subperiodo")
    df["sp_pre2000"]   = (df["year"] <= 1999).astype(int)
    df["sp_2000_2008"] = ((df["year"] >= 2000) & (df["year"] <= 2008)).astype(int)
    df["sp_2009_2015"] = ((df["year"] >= 2009) & (df["year"] <= 2015)).astype(int)
    df["sp_post2015"]  = (df["year"] >= 2016).astype(int)
    for c, etq in [("sp_pre2000", "1990-1999"), ("sp_2000_2008", "2000-2008"),
                   ("sp_2009_2015", "2009-2015"), ("sp_post2015", "2016-2023")]:
        registro.append([c, "Indicadora de subperiodo " + etq,
                         "Indicadora", "1 si el ano pertenece al subperiodo"])

    print("\n8. Escribiendo salidas")
    df.to_csv(SALIDA, index=False)
    print("   {}".format(SALIDA))
    print("   {} obs x {} vars".format(*df.shape))

    dic = pd.DataFrame(registro, columns=["Variable", "Descripcion",
                                          "Tipo", "Construccion"])
    dic.to_csv(DICCIO, index=False)
    print("   {} ({} registros)".format(DICCIO, len(dic)))

    print("\nListo. El panel maestro y el panel reparado NO fueron modificados.")
    print("Base de estimacion: {}".format(os.path.basename(SALIDA)))
    return SALIDA


if __name__ == "__main__":
    main()

1. Leyendo el panel reparado
   entrada: 68 obs x 42 vars

2. Recortando periodo y eliminando columnas excluidas
   periodo 1990-2023; 3 columnas excluidas eliminadas

3. Logaritmos naturales
   8 series transformadas

4. Primeras diferencias por pais
   19 series diferenciadas

5. Rezagos t-1 y t-2
   22 variables rezagadas

6. Indice de Intensidad de Capital (IIC)
   CN: 1990-2023, n = 34
   US: 1990-2023, n = 34

7. Indicadoras de subperiodo

8. Escribiendo salidas
   /content/data_clean/china_us_panel_analisis_v1_1990_2023.csv
   68 obs x 101 vars
   /content/data_clean/diccionario_variables_v1.csv (56 registros)

Listo. El panel maestro y el panel reparado NO fueron modificados.
Base de estimacion: china_us_panel_analisis_v1_1990_2023.csv


In [ ]:
%%writefile /content/construir_panel_analisis.py

Writing /content/construir_panel_analisis.py


In [ ]:
# -*- coding: utf-8 -*-
"""
CONSTRUCCION DEL PANEL DE ANALISIS
Proyecto: modelos de crecimiento comparados China - Estados Unidos, 1990-2023

Cadena de archivos:
    china_us_super_panel_1990_2025.csv   origen inmutable  (NO se toca)
    china_us_super_panel_reparado.csv    intermedio        (entrada de este script)
    china_us_panel_analisis_v1_1990_2023.csv   salida      (base de estimacion)
    diccionario_variables_v1.csv               salida      (apendice del articulo)

Transformaciones que aplica:
    1. Recorta el periodo a 1990-2023 y elimina las columnas excluidas.
    2. Logaritmos naturales de las variables de nivel.
    3. Primeras diferencias por pais, sin contaminacion entre series.
    4. Rezagos t-1 y t-2 de los regresores de los modelos M1 a M9.
    5. Indice de Intensidad de Capital (IIC): subindice inversor menos
       subindice financiero, ambos estandarizados dentro de cada pais.
    6. Indicadoras de subperiodo: pre-2000, 2000-2008, 2009-2015, post-2015.

Uso en Colab:
    !python /content/construir_panel_analisis.py
"""

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ RUTAS
ENTRADA = "/content/data_clean/china_us_super_panel_reparado.csv"
OUTDIR  = "/content/data_clean"
VERSION = "v1"
SALIDA  = os.path.join(OUTDIR, "china_us_panel_analisis_{}_1990_2023.csv".format(VERSION))
DICCIO  = os.path.join(OUTDIR, "diccionario_variables_{}.csv".format(VERSION))

ANIO_INI, ANIO_FIN = 1990, 2023

# Variables de nivel a las que se aplica logaritmo natural
LOGARITMAR = [
    "labour_prod", "rtfpna", "rkna", "gdp_const_usd", "gdp_pc_const_usd",
    "ocupados_totales", "agr_va_per_worker_rec", "ind_va_per_worker_rec",
    "srv_va_per_worker_rec",
]

# Variables que se diferencian y se rezagan (regresores de M1 a M9)
NUCLEO = [
    "gdp_growth", "inv_gfcf_gdp", "trade_openness", "cpi_inflation",
    "urban_pop_pct", "pub_debt_gdp", "hh_debt_gdp", "corp_debt_gdp",
    "bis_tot_credit_gdp", "bis_pvt_credit_gdp", "bis_hh_credit_gdp",
    "bis_nfc_credit_gdp", "ratio_ind_agr_rec", "ratio_srv_ind_rec",
]

# Componentes del Indice de Intensidad de Capital
IIC_INVERSOR  = ["inv_gfcf_gdp", "ratio_ind_agr_rec"]
IIC_FINANCIERO = ["bis_pvt_credit_gdp", "pub_debt_gdp"]

REZAGOS = [1, 2]


def zeta_por_pais(df, col):
    """Estandariza dentro de cada pais, con desviacion muestral."""
    def z(s):
        sd = s.std(ddof=1)
        return (s - s.mean()) / sd if sd and np.isfinite(sd) and sd > 0 else np.nan
    return df.groupby("country")[col].transform(z)


def main(entrada=ENTRADA, outdir=OUTDIR):
    if not os.path.exists(entrada):
        raise FileNotFoundError("No se encontro el panel reparado: " + entrada)
    os.makedirs(outdir, exist_ok=True)

    print("1. Leyendo el panel reparado")
    df = pd.read_csv(entrada)
    df.columns = [str(c).strip() for c in df.columns]
    df["country"] = df["country"].astype(str).str.strip().str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    print("   entrada: {} obs x {} vars".format(*df.shape))

    print("\n2. Recortando periodo y eliminando columnas excluidas")
    df = df[(df["year"] >= ANIO_INI) & (df["year"] <= ANIO_FIN)].copy()
    excluidas = [c for c in df.columns if c.endswith("_excluida")]
    df = df.drop(columns=excluidas)
    df = df.sort_values(["country", "year"]).reset_index(drop=True)
    print("   periodo {}-{}; {} columnas excluidas eliminadas".format(
        ANIO_INI, ANIO_FIN, len(excluidas)))

    registro = []   # filas del diccionario de variables

    print("\n3. Logaritmos naturales")
    n_log = 0
    for c in LOGARITMAR:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        df["ln_" + c] = np.where(s > 0, np.log(s), np.nan)
        registro.append(["ln_" + c, "Logaritmo natural de " + c,
                         "Transformacion", "ln(x) para x > 0"])
        n_log += 1
    print("   {} series transformadas".format(n_log))

    print("\n4. Primeras diferencias por pais")
    n_dif = 0
    candidatas = [c for c in NUCLEO if c in df.columns]
    candidatas += [c for c in df.columns if c.startswith("ln_")]
    for c in candidatas:
        df["d_" + c] = df.groupby("country")[c].diff()
        registro.append(["d_" + c, "Primera diferencia de " + c,
                         "Transformacion", "x(t) - x(t-1), dentro de cada pais"])
        n_dif += 1
    print("   {} series diferenciadas".format(n_dif))

    print("\n5. Rezagos t-1 y t-2")
    n_rez = 0
    for c in [x for x in NUCLEO if x in df.columns]:
        for k in REZAGOS:
            nombre = "{}_lag{}".format(c, k)
            df[nombre] = df.groupby("country")[c].shift(k)
            registro.append([nombre, "Rezago de orden {} de {}".format(k, c),
                             "Transformacion", "x(t-{}), dentro de cada pais".format(k)])
            n_rez += 1
    print("   {} variables rezagadas".format(n_rez))

    print("\n6. Indice de Intensidad de Capital (IIC)")
    inv = [c for c in IIC_INVERSOR if c in df.columns]
    fin = [c for c in IIC_FINANCIERO if c in df.columns]
    if inv and fin:
        for c in inv + fin:
            df["z_" + c] = zeta_por_pais(df, c)
        df["sub_inversor"]   = df[["z_" + c for c in inv]].mean(axis=1)
        df["sub_financiero"] = df[["z_" + c for c in fin]].mean(axis=1)
        df["IIC"] = df["sub_inversor"] - df["sub_financiero"]
        for k in REZAGOS:
            df["IIC_lag{}".format(k)] = df.groupby("country")["IIC"].shift(k)
        registro += [
            ["sub_inversor", "Subindice inversor: promedio de " + ", ".join(inv),
             "Indice compuesto", "Componentes estandarizados dentro de cada pais"],
            ["sub_financiero", "Subindice financiero: promedio de " + ", ".join(fin),
             "Indice compuesto", "Componentes estandarizados dentro de cada pais"],
            ["IIC", "Indice de Intensidad de Capital",
             "Indice compuesto", "sub_inversor menos sub_financiero"],
        ]
        for pais in ["CN", "US"]:
            sub = df.loc[df["country"] == pais, ["year", "IIC"]].dropna()
            if len(sub):
                print("   {}: {}-{}, n = {}".format(pais, int(sub['year'].min()),
                                                    int(sub['year'].max()), len(sub)))
            else:
                print("   {}: sin observaciones validas".format(pais))
    else:
        print("   omitido: faltan componentes en el panel")

    print("\n7. Indicadoras de subperiodo")
    df["sp_pre2000"]   = (df["year"] <= 1999).astype(int)
    df["sp_2000_2008"] = ((df["year"] >= 2000) & (df["year"] <= 2008)).astype(int)
    df["sp_2009_2015"] = ((df["year"] >= 2009) & (df["year"] <= 2015)).astype(int)
    df["sp_post2015"]  = (df["year"] >= 2016).astype(int)
    for c, etq in [("sp_pre2000", "1990-1999"), ("sp_2000_2008", "2000-2008"),
                   ("sp_2009_2015", "2009-2015"), ("sp_post2015", "2016-2023")]:
        registro.append([c, "Indicadora de subperiodo " + etq,
                         "Indicadora", "1 si el ano pertenece al subperiodo"])

    print("\n8. Escribiendo salidas")
    df.to_csv(SALIDA, index=False)
    print("   {}".format(SALIDA))
    print("   {} obs x {} vars".format(*df.shape))

    dic = pd.DataFrame(registro, columns=["Variable", "Descripcion",
                                          "Tipo", "Construccion"])
    dic.to_csv(DICCIO, index=False)
    print("   {} ({} registros)".format(DICCIO, len(dic)))

    print("\nListo. El panel maestro y el panel reparado NO fueron modificados.")
    print("Base de estimacion: {}".format(os.path.basename(SALIDA)))
    return SALIDA


if __name__ == "__main__":
    main()

1. Leyendo el panel reparado
   entrada: 68 obs x 42 vars

2. Recortando periodo y eliminando columnas excluidas
   periodo 1990-2023; 3 columnas excluidas eliminadas

3. Logaritmos naturales
   8 series transformadas

4. Primeras diferencias por pais
   19 series diferenciadas

5. Rezagos t-1 y t-2
   22 variables rezagadas

6. Indice de Intensidad de Capital (IIC)
   CN: 1990-2023, n = 34
   US: 1990-2023, n = 34

7. Indicadoras de subperiodo

8. Escribiendo salidas
   /content/data_clean/china_us_panel_analisis_v1_1990_2023.csv
   68 obs x 101 vars
   /content/data_clean/diccionario_variables_v1.csv (56 registros)

Listo. El panel maestro y el panel reparado NO fueron modificados.
Base de estimacion: china_us_panel_analisis_v1_1990_2023.csv


In [ ]:
!python /content/construir_panel_analisis.py

In [ ]:
import numpy as np, pandas as pd

RUTA = "/content/data_clean/china_us_panel_analisis_v1_1990_2023.csv"
df = pd.read_csv(RUTA)

# ---- DIAGNOSTICO 1: que series de nivel existen realmente
print("Columnas logaritmadas presentes:")
print(sorted([c for c in df.columns if c.startswith("ln_")]))
print("\nBusqueda de PWT y capital:")
print(sorted([c for c in df.columns if any(k in c.lower()
      for k in ["tfp", "rtfp", "rkna", "capital", "prod"])]))

# ---- DIAGNOSTICO 2: cuales del nucleo faltaron
NUCLEO = ["gdp_growth","inv_gfcf_gdp","trade_openness","cpi_inflation",
          "urban_pop_pct","pub_debt_gdp","hh_debt_gdp","corp_debt_gdp",
          "bis_tot_credit_gdp","bis_pvt_credit_gdp","bis_hh_credit_gdp",
          "bis_nfc_credit_gdp","ratio_ind_agr_rec","ratio_srv_ind_rec"]
print("\nDel nucleo NO estan en el panel:")
print([c for c in NUCLEO if c not in df.columns])

# ---- RECONSTRUCCION DEL IIC CON COMPOSICION FIJA  (Opcion B)
INVERSOR   = ["inv_gfcf_gdp"]
FINANCIERO = ["bis_pvt_credit_gdp", "pub_debt_gdp"]

def z_pais(d, col):
    def z(s):
        sd = s.std(ddof=1)
        return (s - s.mean()) / sd if sd and np.isfinite(sd) and sd > 0 else np.nan
    return d.groupby("country")[col].transform(z)

faltan = [c for c in INVERSOR + FINANCIERO if c not in df.columns]
if faltan:
    raise KeyError("Faltan componentes en el panel: {}".format(faltan))

for c in INVERSOR + FINANCIERO:
    df["z_" + c] = z_pais(df, c)

# skipna=False: si falta un componente, el subindice es faltante
df["sub_inversor"]   = df[["z_" + c for c in INVERSOR]].mean(axis=1, skipna=False)
df["sub_financiero"] = df[["z_" + c for c in FINANCIERO]].mean(axis=1, skipna=False)
df["IIC"] = df["sub_inversor"] - df["sub_financiero"]

df = df.sort_values(["country", "year"]).reset_index(drop=True)
for k in (1, 2):
    df["IIC_lag{}".format(k)] = df.groupby("country")["IIC"].shift(k)
df["d_IIC"] = df.groupby("country")["IIC"].diff()

print("\nIIC con composicion fija:")
print("  inversor  :", INVERSOR)
print("  financiero:", FINANCIERO)
for p in ["CN", "US"]:
    s = df.loc[df["country"] == p, ["year", "IIC"]].dropna()
    if len(s):
        print("  {}: {}-{}, n = {}, media = {:.3f}, de = {:.3f}".format(
            p, int(s["year"].min()), int(s["year"].max()), len(s),
            s["IIC"].mean(), s["IIC"].std(ddof=1)))
    else:
        print("  {}: sin observaciones validas".format(p))

SALIDA = "/content/data_clean/china_us_panel_analisis_v2_1990_2023.csv"
df.to_csv(SALIDA, index=False)
print("\nEscrito: {}\n{} obs x {} vars".format(SALIDA, *df.shape))

Columnas logaritmadas presentes:
['ln_agr_va_per_worker_rec', 'ln_gdp_const_usd', 'ln_ind_va_per_worker_rec', 'ln_labour_prod', 'ln_ocupados_totales', 'ln_rkna', 'ln_rtfpna', 'ln_srv_va_per_worker_rec']

Busqueda de PWT y capital:
['d_ln_labour_prod', 'd_ln_rkna', 'd_ln_rtfpna', 'labour_prod', 'ln_labour_prod', 'ln_rkna', 'ln_rtfpna', 'rkna', 'rtfpna']

Del nucleo NO estan en el panel:
['gdp_growth', 'trade_openness', 'cpi_inflation']

IIC con composicion fija:
  inversor  : ['inv_gfcf_gdp']
  financiero: ['bis_pvt_credit_gdp', 'pub_debt_gdp']
  CN: 1995-2021, n = 27, media = 0.182, de = 0.620
  US: 1990-2021, n = 32, media = 0.033, de = 1.411

Escrito: /content/data_clean/china_us_panel_analisis_v2_1990_2023.csv
68 obs x 102 vars


In [ ]:
!ls -la /content/*.py
!head -3 /content/construir_panel_analisis.py

-rw-r--r-- 1 root root     1 Jul 31 00:56 /content/construir_panel_analisis.py
-rw-r--r-- 1 root root 33918 Jul 30 23:15 /content/generar_cuadros_apa.py
 

In [ ]:
import pandas as pd
df = pd.read_csv("/content/data_clean/china_us_panel_analisis_v1_1990_2023.csv")
print(sorted([c for c in df.columns if c.startswith("ln_")]))

['ln_agr_va_per_worker_rec', 'ln_gdp_const_usd', 'ln_ind_va_per_worker_rec', 'ln_labour_prod', 'ln_ocupados_totales', 'ln_rkna', 'ln_rtfpna', 'ln_srv_va_per_worker_rec']


In [ ]:
import numpy as np, pandas as pd

RUTA = "/content/data_clean/china_us_panel_analisis_v1_1990_2023.csv"
df = pd.read_csv(RUTA)

# ---------- DIAGNOSTICO: nombres reales y cobertura de PWT
print("PIB per capita, candidatos:")
print(sorted([c for c in df.columns if "pc" in c.lower() or "capita" in c.lower()]))

NUCLEO = ["gdp_growth","inv_gfcf_gdp","trade_openness","cpi_inflation",
          "urban_pop_pct","pub_debt_gdp","hh_debt_gdp","corp_debt_gdp",
          "bis_tot_credit_gdp","bis_pvt_credit_gdp","bis_hh_credit_gdp",
          "bis_nfc_credit_gdp","ratio_ind_agr_rec","ratio_srv_ind_rec"]
print("\nDel nucleo NO estan en el panel:")
print([c for c in NUCLEO if c not in df.columns])

print("\nCobertura de Penn World Table (define el periodo de M2):")
for v in ["rtfpna", "rkna", "labour_prod"]:
    if v in df.columns:
        for p in ["CN", "US"]:
            s = df.loc[df["country"] == p, ["year", v]].dropna()
            if len(s):
                print("  {:12s} {}: {}-{}, n = {}".format(
                    v, p, int(s["year"].min()), int(s["year"].max()), len(s)))

# ---------- RECONSTRUCCION DEL IIC CON COMPOSICION FIJA (Opcion B)
INVERSOR   = ["inv_gfcf_gdp"]
FINANCIERO = ["bis_pvt_credit_gdp", "pub_debt_gdp"]

def z_pais(d, col):
    def z(s):
        sd = s.std(ddof=1)
        return (s - s.mean()) / sd if sd and np.isfinite(sd) and sd > 0 else np.nan
    return d.groupby("country")[col].transform(z)

faltan = [c for c in INVERSOR + FINANCIERO if c not in df.columns]
if faltan:
    raise KeyError("Faltan componentes: {}".format(faltan))

for c in INVERSOR + FINANCIERO:
    df["z_" + c] = z_pais(df, c)

df["sub_inversor"]   = df[["z_" + c for c in INVERSOR]].mean(axis=1, skipna=False)
df["sub_financiero"] = df[["z_" + c for c in FINANCIERO]].mean(axis=1, skipna=False)
df["IIC"] = df["sub_inversor"] - df["sub_financiero"]

df = df.sort_values(["country", "year"]).reset_index(drop=True)
for k in (1, 2):
    df["IIC_lag{}".format(k)] = df.groupby("country")["IIC"].shift(k)
df["d_IIC"] = df.groupby("country")["IIC"].diff()

print("\nIIC con composicion fija:")
for p in ["CN", "US"]:
    s = df.loc[df["country"] == p, ["year", "IIC"]].dropna()
    if len(s):
        print("  {}: {}-{}, n = {}, media = {:.3f}, de = {:.3f}".format(
            p, int(s["year"].min()), int(s["year"].max()), len(s),
            s["IIC"].mean(), s["IIC"].std(ddof=1)))
    else:
        print("  {}: sin observaciones validas".format(p))

SALIDA = "/content/data_clean/china_us_panel_analisis_v2_1990_2023.csv"
df.to_csv(SALIDA, index=False)
print("\nEscrito: {}\n{} obs x {} vars".format(SALIDA, *df.shape))

PIB per capita, candidatos:
['d_urban_pop_pct', 'gdp_growth_pct', 'gdp_pc_ppp_current', 'gni_pc_ppp_current', 'inflation_cpi_pct', 'unemployment_pct', 'urban_pop_pct', 'urban_pop_pct_lag1', 'urban_pop_pct_lag2']

Del nucleo NO estan en el panel:
['gdp_growth', 'trade_openness', 'cpi_inflation']

Cobertura de Penn World Table (define el periodo de M2):
  rtfpna       CN: 1990-2023, n = 34
  rtfpna       US: 1990-2023, n = 34
  rkna         CN: 1990-2023, n = 34
  rkna         US: 1990-2023, n = 34
  labour_prod  CN: 1991-2023, n = 33
  labour_prod  US: 1991-2023, n = 33

IIC con composicion fija:
  CN: 1995-2021, n = 27, media = 0.182, de = 0.620
  US: 1990-2021, n = 32, media = 0.033, de = 1.411

Escrito: /content/data_clean/china_us_panel_analisis_v2_1990_2023.csv
68 obs x 102 vars


In [ ]:
import numpy as np, pandas as pd

RUTA = "/content/data_clean/china_us_panel_analisis_v2_1990_2023.csv"
df = pd.read_csv(RUTA).sort_values(["country", "year"]).reset_index(drop=True)

# ---------- LOCALIZAR LA APERTURA COMERCIAL
print("Candidatos a apertura comercial:")
print(sorted([c for c in df.columns
              if any(k in c.lower() for k in
                     ["trade", "open", "exp", "imp", "comer"])
              and not c.startswith(("d_", "ln_", "z_"))
              and "_lag" not in c]))

print("\nTodas las columnas de origen (sin transformadas):")
base = sorted([c for c in df.columns
               if not c.startswith(("d_", "ln_", "z_", "sp_", "sub_", "IIC"))
               and "_lag" not in c])
for i in range(0, len(base), 4):
    print("   " + "  ".join("{:26s}".format(x) for x in base[i:i+4]))

# ---------- TRANSFORMAR LAS VARIABLES CON NOMBRE CORREGIDO
CORREGIDAS = ["gdp_growth_pct", "inflation_cpi_pct", "unemployment_pct",
              "gdp_pc_ppp_current"]
# agrega aqui el nombre real de la apertura comercial cuando lo veas arriba,
# por ejemplo:  CORREGIDAS.append("trade_pct_gdp")

nuevas = []
for c in CORREGIDAS:
    if c not in df.columns:
        print("\n  aviso: no existe la columna {}".format(c))
        continue
    if "d_" + c not in df.columns:
        df["d_" + c] = df.groupby("country")[c].diff()
        nuevas.append("d_" + c)
    for k in (1, 2):
        nom = "{}_lag{}".format(c, k)
        if nom not in df.columns:
            df[nom] = df.groupby("country")[c].shift(k)
            nuevas.append(nom)

print("\nVariables creadas ({}):".format(len(nuevas)))
print(nuevas)

# ---------- VERIFICACION: el termino autorregresivo debe existir
for p in ["CN", "US"]:
    s = df.loc[df["country"] == p, ["year", "gdp_growth_pct_lag1"]].dropna()
    print("  gdp_growth_pct_lag1 {}: {}-{}, n = {}".format(
        p, int(s["year"].min()), int(s["year"].max()), len(s)))

SALIDA = "/content/data_clean/china_us_panel_analisis_v3_1990_2023.csv"
df.to_csv(SALIDA, index=False)
print("\nEscrito: {}\n{} obs x {} vars".format(SALIDA, *df.shape))

Candidatos a apertura comercial:
['trade_gdp']

Todas las columnas de origen (sin transformadas):
   agr_va_per_worker           agr_va_per_worker_rec       bis_gov_credit_gdp          bis_hh_credit_gdp         
   bis_nfc_credit_gdp          bis_pvt_banks_credit_gdp    bis_pvt_credit_gdp          bis_tot_credit_gdp        
   corp_debt_gdp               country                     credit_priv_gdp             gdp_const_usd             
   gdp_growth_pct              gdp_pc_ppp_current          gni_pc_ppp_current          gov_cons_gdp              
   hc                          hh_debt_gdp                 ind_va_per_worker           ind_va_per_worker_rec     
   inflation_cpi_pct           inv_gfcf_gdp                labour_prod                 ocupados_totales          
   pop                         priv_debt_gdp               pub_debt_gdp                ratio_ind_agr_rec         
   ratio_srv_ind_rec           remittances_gdp             rgdpna                      rkna             

In [ ]:
import numpy as np, pandas as pd

RUTA = "/content/data_clean/china_us_panel_analisis_v3_1990_2023.csv"
df = pd.read_csv(RUTA).sort_values(["country", "year"]).reset_index(drop=True)

# ---------- 1. COBERTURA DE LAS VARIABLES RECIEN IDENTIFICADAS
print("COBERTURA")
for v in ["credit_priv_gdp", "trade_gdp", "gov_cons_gdp", "hc", "pop",
          "rgdpna", "bis_pvt_credit_gdp", "pub_debt_gdp"]:
    if v not in df.columns:
        print("  {:22s} ausente".format(v)); continue
    for p in ["CN", "US"]:
        s = df.loc[df["country"] == p, ["year", v]].dropna()
        if len(s):
            print("  {:22s} {}: {}-{}, n = {}".format(
                v, p, int(s["year"].min()), int(s["year"].max()), len(s)))
        else:
            print("  {:22s} {}: 0 observaciones".format(v, p))

# ---------- 2. PIB REAL PER CAPITA CON PENN WORLD TABLE
if {"rgdpna", "pop"}.issubset(df.columns):
    pc = df["rgdpna"] / df["pop"]
    df["gdp_pc_pwt"] = pc
    df["ln_gdp_pc_pwt"] = np.where(pc > 0, np.log(pc), np.nan)
    df["growth_pc_pwt"] = 100 * df.groupby("country")["gdp_pc_pwt"].pct_change()
    print("\nPIB per capita de PWT construido (gdp_pc_pwt, ln_gdp_pc_pwt, "
          "growth_pc_pwt)")

# ---------- 3. DIFERENCIAS Y REZAGOS DE LO QUE FALTA
PENDIENTES = ["trade_gdp", "credit_priv_gdp", "gov_cons_gdp", "hc",
              "ln_gdp_pc_pwt", "growth_pc_pwt"]
nuevas = []
for c in PENDIENTES:
    if c not in df.columns:
        print("  aviso: no existe {}".format(c)); continue
    if "d_" + c not in df.columns:
        df["d_" + c] = df.groupby("country")[c].diff(); nuevas.append("d_" + c)
    for k in (1, 2):
        nom = "{}_lag{}".format(c, k)
        if nom not in df.columns:
            df[nom] = df.groupby("country")[c].shift(k); nuevas.append(nom)
print("\nVariables creadas ({}): {}".format(len(nuevas), nuevas))

# ---------- 4. IIC ALTERNATIVO CON CREDITO DEL WDI
def z_pais(d, col):
    def z(s):
        sd = s.std(ddof=1)
        return (s - s.mean()) / sd if sd and np.isfinite(sd) and sd > 0 else np.nan
    return d.groupby("country")[col].transform(z)

INV_ALT = ["inv_gfcf_gdp"]
FIN_ALT = ["credit_priv_gdp", "pub_debt_gdp"]
if set(INV_ALT + FIN_ALT).issubset(df.columns):
    for c in set(INV_ALT + FIN_ALT):
        if "z_" + c not in df.columns:
            df["z_" + c] = z_pais(df, c)
    df["sub_inversor_alt"]   = df[["z_" + c for c in INV_ALT]].mean(axis=1, skipna=False)
    df["sub_financiero_alt"] = df[["z_" + c for c in FIN_ALT]].mean(axis=1, skipna=False)
    df["IIC_alt"] = df["sub_inversor_alt"] - df["sub_financiero_alt"]
    for k in (1, 2):
        df["IIC_alt_lag{}".format(k)] = df.groupby("country")["IIC_alt"].shift(k)
    df["d_IIC_alt"] = df.groupby("country")["IIC_alt"].diff()
    print("\nIIC alternativo (credito del WDI en lugar del BIS):")
    for p in ["CN", "US"]:
        s = df.loc[df["country"] == p, ["year", "IIC_alt"]].dropna()
        if len(s):
            print("  {}: {}-{}, n = {}, de = {:.3f}".format(
                p, int(s["year"].min()), int(s["year"].max()), len(s),
                s["IIC_alt"].std(ddof=1)))

# ---------- 5. CORRELACION ENTRE LAS DOS MEDIDAS DE CREDITO
if {"credit_priv_gdp", "bis_pvt_credit_gdp"}.issubset(df.columns):
    print("\nCorrelacion entre credito del WDI y del BIS:")
    for p in ["CN", "US"]:
        s = df.loc[df["country"] == p, ["credit_priv_gdp",
                                        "bis_pvt_credit_gdp"]].dropna()
        if len(s) > 3:
            print("  {}: r = {:.3f}  (n = {})".format(
                p, s.corr().iloc[0, 1], len(s)))

SALIDA = "/content/data_clean/china_us_panel_analisis_v4_1990_2023.csv"
df.to_csv(SALIDA, index=False)
print("\nEscrito: {}\n{} obs x {} vars".format(SALIDA, *df.shape))

COBERTURA
  credit_priv_gdp        CN: 1990-2023, n = 34
  credit_priv_gdp        US: 1990-2023, n = 34
  trade_gdp              CN: 1990-2023, n = 34
  trade_gdp              US: 1990-2023, n = 34
  gov_cons_gdp           CN: 1990-2023, n = 34
  gov_cons_gdp           US: 1990-2023, n = 34
  hc                     CN: 1990-2023, n = 34
  hc                     US: 1990-2023, n = 34
  pop                    CN: 1990-2023, n = 34
  pop                    US: 1990-2023, n = 34
  rgdpna                 CN: 1990-2023, n = 34
  rgdpna                 US: 1990-2023, n = 34
  bis_pvt_credit_gdp     CN: 1990-2021, n = 32
  bis_pvt_credit_gdp     US: 1990-2021, n = 32
  pub_debt_gdp           CN: 1995-2023, n = 29
  pub_debt_gdp           US: 1990-2023, n = 34

PIB per capita de PWT construido (gdp_pc_pwt, ln_gdp_pc_pwt, growth_pc_pwt)

Variables creadas (18): ['d_trade_gdp', 'trade_gdp_lag1', 'trade_gdp_lag2', 'd_credit_priv_gdp', 'credit_priv_gdp_lag1', 'credit_priv_gdp_lag2', 'd_gov_cons_gdp

In [ ]:
# -*- coding: utf-8 -*-
"""
PRUEBAS DE DIAGNOSTICO v2
Cuadro 3 revisado : raiz unitaria con constante y con tendencia, mas
                    quiebre estructural endogeno (Zivot-Andrews).
Cuadro 5 revisado : correlaciones en PRIMERAS DIFERENCIAS.
Cuadro 6 revisado : VIF con conjuntos de regresores por pais.

Salida: /content/outputs/Cuadros_diagnostico_v2_APA.docx
"""

import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

from statsmodels.tsa.stattools import adfuller, zivot_andrews
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from arch.unitroot import PhillipsPerron

from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_ORIENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

PANEL  = "/content/data_clean/china_us_panel_analisis_v4_1990_2023.csv"
OUTDIR = "/content/outputs"
SALIDA = os.path.join(OUTDIR, "Cuadros_diagnostico_v2_APA.docx")

PAISES = {"CN": "China", "US": "Estados Unidos"}

# Series a evaluar, con su etiqueta y si llevan tendencia deterministica
SERIES = [
    ("gdp_growth_pct",     "Crecimiento del PIB (%)",            False),
    ("inv_gfcf_gdp",       "FBCF (% del PIB)",                   False),
    ("trade_gdp",          "Apertura comercial (% del PIB)",     True),
    ("inflation_cpi_pct",  "Inflacion IPC (%)",                  False),
    ("ln_rtfpna",          "log PTF (PWT)",                      True),
    ("ln_labour_prod",     "log productividad laboral",          True),
    ("ln_gdp_pc_pwt",      "log PIB per capita real (PWT)",      True),
    ("credit_priv_gdp",    "Credito privado, WDI (% del PIB)",   True),
    ("bis_pvt_credit_gdp", "Credito privado, BIS (% del PIB)",   True),
    ("pub_debt_gdp",       "Deuda publica (% del PIB)",          True),
    ("hh_debt_gdp",        "Deuda de hogares (% del PIB)",       True),
    ("corp_debt_gdp",      "Deuda corporativa (% del PIB)",      True),
    ("urban_pop_pct",      "Poblacion urbana (%)",               True),
    ("gov_cons_gdp",       "Consumo de gobierno (% del PIB)",    False),
    ("hc",                 "Indice de capital humano",           True),
    ("ratio_ind_agr_rec",  "Brecha productividad ind./agr.",     True),
    ("IIC_alt",            "IIC (indice compuesto)",             True),
]

# Conjuntos de regresores para el VIF, diferenciados por pais
VIF_SET = {
    "CN": ["inv_gfcf_gdp", "credit_priv_gdp", "pub_debt_gdp", "trade_gdp",
           "inflation_cpi_pct", "ln_rtfpna"],
    "US": ["inv_gfcf_gdp", "credit_priv_gdp", "pub_debt_gdp", "trade_gdp",
           "inflation_cpi_pct", "ln_rtfpna", "hh_debt_gdp", "corp_debt_gdp"],
}

# Variables para la matriz de correlaciones en diferencias
CORR_SET = ["gdp_growth_pct", "inv_gfcf_gdp", "credit_priv_gdp",
            "pub_debt_gdp", "trade_gdp", "inflation_cpi_pct",
            "ln_rtfpna", "IIC_alt"]

MIN_OBS = 15   # minimo de observaciones para intentar una prueba


# ============================================================ UTILIDADES
def estrellas(p):
    if p is None or not np.isfinite(p): return ""
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""


def fmt(v, p=None, dec=2):
    if v is None or not np.isfinite(v): return "—"
    return "{:.{d}f}{}".format(v, estrellas(p), d=dec)


def adf(y, reg):
    try:
        r = adfuller(y, regression=reg, autolag="AIC")
        return r[0], r[1]
    except Exception:
        return np.nan, np.nan


def pp(y, trend):
    try:
        r = PhillipsPerron(y, trend=trend)
        return r.stat, r.pvalue
    except Exception:
        return np.nan, np.nan


def za(y, reg="ct"):
    """Zivot-Andrews: estadistico, p, y posicion del quiebre."""
    try:
        r = zivot_andrews(np.asarray(y, dtype=float), regression=reg,
                          autolag="AIC")
        return r[0], r[1], r[4]
    except Exception:
        return np.nan, np.nan, None


def orden_integracion(p_niv, p_dif, p_za):
    """Regla: rechazo en niveles -> I(0); si no, rechazo en diferencias -> I(1).
    Si ZA rechaza en niveles, se reporta I(0) con quiebre."""
    if p_za is not None and np.isfinite(p_za) and p_za < .05:
        return "I(0) con quiebre"
    if np.isfinite(p_niv) and p_niv < .05:
        return "I(0)"
    if np.isfinite(p_dif) and p_dif < .05:
        return "I(1)"
    return "no concluyente"


# ============================================================ CUADRO 3
def cuadro_raiz_unitaria(df):
    filas = []
    for cod, pais in PAISES.items():
        sub = df[df["country"] == cod].sort_values("year")
        for var, etq, tend in SERIES:
            if var not in sub.columns: continue
            s = sub[["year", var]].dropna()
            if len(s) < MIN_OBS:
                filas.append([pais, etq, "—", "—", "—", "—", "—", "—", "—",
                              "muestra insuficiente"])
                continue
            y = s[var].astype(float).values
            dy = np.diff(y)

            a_c,  pa_c  = adf(y, "c")
            a_ct, pa_ct = adf(y, "ct")
            p_c,  pp_c  = pp(y, "c")
            p_ct, pp_ct = pp(y, "ct")
            ad_,  pad_  = adf(dy, "c")
            pd_,  ppd_  = pp(dy, "c")
            z_, pz_, bidx = za(y, "ct" if tend else "c")

            anio_q = "—"
            if bidx is not None:
                try: anio_q = str(int(s["year"].values[int(bidx)]))
                except Exception: pass

            # p de referencia en niveles: con tendencia si la serie la tiene
            p_niv = pa_ct if tend else pa_c
            p_dif = pad_ if np.isfinite(pad_) else ppd_

            filas.append([
                pais, etq,
                fmt(a_c, pa_c), fmt(a_ct, pa_ct),
                fmt(p_c, pp_c), fmt(p_ct, pp_ct),
                fmt(ad_, pad_), fmt(pd_, ppd_),
                "{} [{}]".format(fmt(z_, pz_), anio_q),
                orden_integracion(p_niv, p_dif, pz_),
            ])
    cols = ["Pais", "Serie", "ADF (c)", "ADF (c+t)", "PP (c)", "PP (c+t)",
            "ADF Δ", "PP Δ", "Zivot-Andrews [año]", "Orden"]
    return pd.DataFrame(filas, columns=cols)


# ============================================================ CUADRO 5
def cuadro_correlaciones_dif(df):
    salidas = {}
    for cod, pais in PAISES.items():
        sub = df[df["country"] == cod].sort_values("year")
        cols = [c for c in CORR_SET if c in sub.columns]
        d = sub[cols].diff()
        d = d.dropna(how="all")
        M = pd.DataFrame(index=cols, columns=cols, dtype=object)
        for i in cols:
            for j in cols:
                par = d[[i, j]].dropna()
                if i == j:
                    M.loc[i, j] = "1.00"
                elif len(par) < 10:
                    M.loc[i, j] = "—"
                else:
                    from scipy.stats import pearsonr
                    r, p = pearsonr(par[i], par[j])
                    M.loc[i, j] = "{:.2f}{}".format(r, estrellas(p))
        M.insert(0, "Variable", cols)
        salidas[pais] = M.reset_index(drop=True)
    return salidas


# ============================================================ CUADRO 6
def cuadro_vif(df):
    filas = []
    for cod, pais in PAISES.items():
        sub = df[df["country"] == cod].sort_values("year")
        cols = [c for c in VIF_SET[cod] if c in sub.columns]
        X = sub[cols].dropna()
        if len(X) <= len(cols) + 2:
            for c in cols:
                filas.append([pais, c, "—", "muestra insuficiente"])
            continue
        Xc = sm.add_constant(X.astype(float).values)
        for k, c in enumerate(cols, start=1):
            try:
                v = variance_inflation_factor(Xc, k)
                diag = ("severa" if v >= 10 else
                        "revisar" if v >= 5 else "aceptable")
                filas.append([pais, c, "{:.2f}".format(v), diag])
            except Exception:
                filas.append([pais, c, "—", "no estimable"])
        filas.append([pais, "n (casos completos)", str(len(X)), ""])
    return pd.DataFrame(filas, columns=["Pais", "Variable", "VIF",
                                        "Diagnostico"])


# ============================================================ WORD APA
def sombrear(celda, hexc):
    tc = celda._tc.get_or_add_tcPr()
    sh = OxmlElement("w:shd"); sh.set(qn("w:val"), "clear")
    sh.set(qn("w:fill"), hexc); tc.append(sh)


def borde(celda, lado, sz=8):
    tc = celda._tc.get_or_add_tcPr()
    bd = tc.find(qn("w:tcBorders"))
    if bd is None:
        bd = OxmlElement("w:tcBorders"); tc.append(bd)
    e = OxmlElement("w:" + lado)
    e.set(qn("w:val"), "single"); e.set(qn("w:sz"), str(sz))
    e.set(qn("w:color"), "000000"); bd.append(e)


def layout_fijo(tabla):
    tblPr = tabla._tbl.tblPr
    el = OxmlElement("w:tblLayout"); el.set(qn("w:type"), "fixed")
    tblPr.append(el)


def repetir_encabezado(fila):
    trPr = fila._tr.get_or_add_trPr()
    el = OxmlElement("w:tblHeader"); el.set(qn("w:val"), "true")
    trPr.append(el)


def poner(celda, texto, negrita=False, size=8, centro=False):
    celda.text = ""
    p = celda.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER if centro else WD_ALIGN_PARAGRAPH.LEFT
    p.paragraph_format.space_before = Pt(1)
    p.paragraph_format.space_after = Pt(1)
    r = p.add_run(str(texto))
    r.font.name = "Arial"; r.font.size = Pt(size); r.bold = negrita


def agregar_cuadro(doc, num, titulo, dframe, nota, anchos, size=8):
    p = doc.add_paragraph(); p.paragraph_format.keep_with_next = True
    r = p.add_run("Cuadro {}".format(num)); r.bold = True
    r.font.name = "Arial"; r.font.size = Pt(10)

    p = doc.add_paragraph(); p.paragraph_format.keep_with_next = True
    r = p.add_run(titulo); r.italic = True
    r.font.name = "Arial"; r.font.size = Pt(10)

    ncol = len(dframe.columns)
    t = doc.add_table(rows=1, cols=ncol)
    t.autofit = False
    layout_fijo(t)

    total = sum(anchos)
    util = 23.94
    anchos_cm = [util * a / total for a in anchos]

    hdr = t.rows[0]
    repetir_encabezado(hdr)
    for k, col in enumerate(dframe.columns):
        c = hdr.cells[k]
        c.width = Cm(anchos_cm[k])
        poner(c, col, negrita=True, size=size, centro=True)
        sombrear(c, "F0EFEA"); borde(c, "top"); borde(c, "bottom")

    for _, row in dframe.iterrows():
        cells = t.add_row().cells
        for k, col in enumerate(dframe.columns):
            cells[k].width = Cm(anchos_cm[k])
            poner(cells[k], row[col], size=size, centro=(k > 1))

    for c in t.rows[-1].cells:
        borde(c, "bottom")

    p = doc.add_paragraph()
    r = p.add_run("Nota. "); r.italic = True
    r.font.name = "Arial"; r.font.size = Pt(8)
    r = p.add_run(nota); r.font.name = "Arial"; r.font.size = Pt(8)
    doc.add_paragraph()


def documento_horizontal():
    doc = Document()
    s = doc.sections[0]
    s.orientation = WD_ORIENT.LANDSCAPE
    s.page_width, s.page_height = Cm(27.94), Cm(21.59)
    for m in ("top_margin", "bottom_margin", "left_margin", "right_margin"):
        setattr(s, m, Cm(2))
    est = doc.styles["Normal"]
    est.font.name = "Arial"; est.font.size = Pt(10)
    return doc


# ============================================================ MAIN
def main(panel=PANEL, outdir=OUTDIR):
    os.makedirs(outdir, exist_ok=True)
    df = pd.read_csv(panel)
    df["country"] = df["country"].astype(str).str.strip().str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    print("Panel: {} obs x {} vars".format(*df.shape))

    print("\n1. Cuadro 3: raiz unitaria con tendencia y quiebre")
    c3 = cuadro_raiz_unitaria(df)
    print(c3.to_string(index=False))

    print("\n2. Cuadro 5: correlaciones en primeras diferencias")
    c5 = cuadro_correlaciones_dif(df)

    print("\n3. Cuadro 6: VIF por pais")
    c6 = cuadro_vif(df)
    print(c6.to_string(index=False))

    print("\n4. Maquetando el documento")
    doc = documento_horizontal()

    agregar_cuadro(
        doc, "3",
        "Pruebas de raiz unitaria con especificacion alternativa y quiebre "
        "estructural endogeno, China y Estados Unidos, 1990-2023",
        c3,
        "c = constante; c+t = constante y tendencia; Δ = primera diferencia. "
        "Zivot-Andrews con quiebre endogeno en intercepto y tendencia; entre "
        "corchetes, el año del quiebre estimado. El orden de integracion se "
        "determina con la especificacion pertinente a cada serie: con tendencia "
        "para las series tendenciales y solo con constante para las "
        "estacionarias en media. *p < .10. **p < .05. ***p < .01.",
        anchos=[9, 20, 9, 9, 9, 9, 9, 9, 12, 12], size=7)

    for pais, M in c5.items():
        n = len(M.columns) - 1
        agregar_cuadro(
            doc, "5",
            "Correlaciones de Pearson en primeras diferencias, {}, 1990-2023"
            .format(pais),
            M,
            "Las correlaciones se calculan sobre las primeras diferencias para "
            "evitar la asociacion espuria entre series con tendencia comun. "
            "*p < .10. **p < .05. ***p < .01.",
            anchos=[22] + [8] * n, size=7)

    agregar_cuadro(
        doc, "6",
        "Factores de inflacion de la varianza con conjuntos de regresores "
        "diferenciados por pais",
        c6,
        "VIF ≥ 10 indica multicolinealidad severa; entre 5 y 10, revisar. Los "
        "conjuntos difieren entre paises porque la desagregacion sectorial de "
        "la deuda privada no esta disponible para China en el periodo de "
        "estudio. *n* corresponde a los casos completos de cada conjunto.",
        anchos=[14, 30, 10, 14], size=8)

    doc.save(SALIDA)
    print("   Guardado: {}".format(SALIDA))

    c3.to_csv(os.path.join(outdir, "cuadro3_raiz_unitaria_v2.csv"), index=False)
    c6.to_csv(os.path.join(outdir, "cuadro6_vif_v2.csv"), index=False)
    print("\nListo.")


if __name__ == "__main__":
    main()

Panel: 68 obs x 142 vars

1. Cuadro 3: raiz unitaria con tendencia y quiebre
          Pais                            Serie  ADF (c) ADF (c+t)   PP (c) PP (c+t)    ADF Δ      PP Δ Zivot-Andrews [año]                Orden
         China          Crecimiento del PIB (%)    -1.72    -3.33*  -3.23** -4.67*** -7.31***  -9.97***      -5.25** [2003]     I(0) con quiebre
         China                 FBCF (% del PIB)   -2.64*    -3.25*  -2.98**    -2.36 -4.60***  -3.73***      -4.83** [2002]     I(0) con quiebre
         China   Apertura comercial (% del PIB)    -2.01     -1.76    -1.89    -1.35 -3.88***  -3.76***        -4.83 [2002]                 I(1)
         China                Inflacion IPC (%)    -2.28     -2.86    -1.75    -2.17 -4.54***  -4.05***     -6.42*** [1996]     I(0) con quiebre
         China                    log PTF (PWT)    -1.73     -2.75   -2.72*    -1.56  -3.11**    -2.70*     -7.12*** [2005]     I(0) con quiebre
         China        log productividad laboral    -1

Script v3
Resuelve los tres problemas. Guárdalo con %%writefile /content/pruebas_v3.py.

In [ ]:
# -*- coding: utf-8 -*-
"""
PRUEBAS DE DIAGNOSTICO v3
1. Segunda ronda de raiz unitaria para las series no concluyentes:
   ADF y PP con tendencia sobre la primera diferencia, y Zivot-Andrews
   sobre la primera diferencia.
2. VIF por especificacion de modelo (M1 a M9), en niveles y en diferencias.
3. Indicadoras de quiebre estructural derivadas de las fechas estimadas.

Salidas: /content/outputs/Cuadros_diagnostico_v3_APA.docx
         /content/data_clean/china_us_panel_analisis_v5_1990_2023.csv
"""

import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

from statsmodels.tsa.stattools import adfuller, zivot_andrews
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from arch.unitroot import PhillipsPerron

PANEL  = "/content/data_clean/china_us_panel_analisis_v4_1990_2023.csv"
OUTDIR = "/content/outputs"
DATADIR = "/content/data_clean"

PAISES = {"CN": "China", "US": "Estados Unidos"}

# Series que quedaron no concluyentes en la primera ronda
NO_CONCLUYENTES = {
    "CN": ["ln_labour_prod", "ln_gdp_pc_pwt", "pub_debt_gdp",
           "urban_pop_pct", "hc", "IIC_alt"],
    "US": ["hh_debt_gdp", "urban_pop_pct", "hc"],
}

# Especificaciones del cuadro 4
MODELOS = {
    "M1": ["inv_gfcf_gdp", "trade_gdp", "inflation_cpi_pct"],
    "M2": ["inv_gfcf_gdp", "trade_gdp"],
    "M3": ["credit_priv_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M4": ["bis_pvt_credit_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M5": ["pub_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M6": ["hh_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M7": ["corp_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M8": ["ratio_ind_agr_rec", "inv_gfcf_gdp", "trade_gdp"],
    "M9": ["IIC_alt", "trade_gdp"],
}
SOLO_US = {"M6", "M7"}

# Fechas de quiebre para las indicadoras (de la corrida v2)
QUIEBRES = {"CN": [2002, 2014], "US": [2008]}


def estrellas(p):
    if p is None or not np.isfinite(p): return ""
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""


def fmt(v, p=None, d=2):
    if v is None or not np.isfinite(v): return "—"
    return "{:.{k}f}{}".format(v, estrellas(p), k=d)


def adf(y, reg):
    try:
        r = adfuller(y, regression=reg, autolag="AIC"); return r[0], r[1]
    except Exception:
        return np.nan, np.nan


def pp_(y, trend):
    try:
        r = PhillipsPerron(y, trend=trend); return r.stat, r.pvalue
    except Exception:
        return np.nan, np.nan


def za(y, reg="ct"):
    try:
        r = zivot_andrews(np.asarray(y, float), regression=reg, autolag="AIC")
        return r[0], r[1], r[4]
    except Exception:
        return np.nan, np.nan, None


# ============================================== 1. SEGUNDA RONDA
def segunda_ronda(df):
    filas = []
    for cod, pais in PAISES.items():
        sub = df[df["country"] == cod].sort_values("year")
        for var in NO_CONCLUYENTES.get(cod, []):
            if var not in sub.columns: continue
            s = sub[["year", var]].dropna()
            if len(s) < 15:
                filas.append([pais, var, "—", "—", "—", "—", "—", "insuf."])
                continue
            y = s[var].astype(float).values
            dy = np.diff(y); d2y = np.diff(dy)

            a_ct, pa_ct = adf(dy, "ct")      # diferencia CON tendencia
            p_ct, pp_ct = pp_(dy, "ct")
            z_,  pz_, bidx = za(dy, "ct")    # quiebre sobre la diferencia
            a2, pa2 = adf(d2y, "c")          # segunda diferencia

            anio_q = "—"
            if bidx is not None:
                try: anio_q = str(int(s["year"].values[1:][int(bidx)]))
                except Exception: pass

            if np.isfinite(pa_ct) and pa_ct < .05:
                concl = "I(1)"
            elif np.isfinite(pp_ct) and pp_ct < .05:
                concl = "I(1) (PP)"
            elif np.isfinite(pz_) and pz_ < .05:
                concl = "I(1) con quiebre"
            elif np.isfinite(pa2) and pa2 < .05:
                concl = "I(2): excluir de niveles"
            else:
                concl = "tendencia determinista"

            filas.append([pais, var, fmt(a_ct, pa_ct), fmt(p_ct, pp_ct),
                          "{} [{}]".format(fmt(z_, pz_), anio_q),
                          fmt(a2, pa2), concl])
    return pd.DataFrame(filas, columns=[
        "Pais", "Serie", "ADF Δ (c+t)", "PP Δ (c+t)",
        "ZA sobre Δ [año]", "ADF Δ²", "Conclusion"])


# ============================================== 2. VIF POR MODELO
def vif_por_modelo(df):
    filas = []
    for cod, pais in PAISES.items():
        sub = df[df["country"] == cod].sort_values("year")
        for mid, regs in MODELOS.items():
            if mid in SOLO_US and cod != "US":
                continue
            cols = [c for c in regs if c in sub.columns]
            if len(cols) < 2:
                filas.append([pais, mid, "—", "—", "—", "columnas ausentes"])
                continue
            for etiqueta, base in [("niveles", sub[cols]),
                                   ("diferencias", sub[cols].diff())]:
                X = base.dropna().astype(float)
                if len(X) <= len(cols) + 2:
                    filas.append([pais, mid, etiqueta, "—", str(len(X)),
                                  "muestra insuficiente"])
                    continue
                Xc = sm.add_constant(X.values)
                vifs = []
                for k in range(1, len(cols) + 1):
                    try: vifs.append(variance_inflation_factor(Xc, k))
                    except Exception: vifs.append(np.nan)
                vmax = np.nanmax(vifs)
                detalle = "; ".join("{} = {:.2f}".format(c, v)
                                    for c, v in zip(cols, vifs))
                diag = ("severa" if vmax >= 10 else
                        "revisar" if vmax >= 5 else "aceptable")
                filas.append([pais, mid, etiqueta, "{:.2f}".format(vmax),
                              str(len(X)), "{} ({})".format(diag, detalle)])
    return pd.DataFrame(filas, columns=[
        "Pais", "Modelo", "Forma", "VIF max", "n", "Diagnostico"])


# ============================================== 3. INDICADORAS DE QUIEBRE
def indicadoras_quiebre(df):
    df = df.copy()
    creadas = []
    for cod, anios in QUIEBRES.items():
        for a in anios:
            col = "q{}".format(a)
            if col not in df.columns:
                df[col] = 0
            df.loc[(df["country"] == cod) & (df["year"] >= a), col] = 1
            if col not in creadas: creadas.append(col)
    # indicadora comun de crisis financiera global
    df["q_cfg"] = ((df["year"] >= 2008) & (df["year"] <= 2009)).astype(int)
    creadas.append("q_cfg")
    return df, creadas


def main():
    os.makedirs(OUTDIR, exist_ok=True)
    df = pd.read_csv(PANEL)
    df["country"] = df["country"].astype(str).str.strip().str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    print("Panel: {} obs x {} vars".format(*df.shape))

    print("\n1. Segunda ronda de raiz unitaria")
    c3b = segunda_ronda(df)
    print(c3b.to_string(index=False))

    print("\n2. VIF por especificacion de modelo")
    c6b = vif_por_modelo(df)
    print(c6b.to_string(index=False))

    print("\n3. Indicadoras de quiebre")
    df, creadas = indicadoras_quiebre(df)
    print("   creadas: {}".format(creadas))

    c3b.to_csv(os.path.join(OUTDIR, "cuadro3b_segunda_ronda.csv"), index=False)
    c6b.to_csv(os.path.join(OUTDIR, "cuadro6b_vif_por_modelo.csv"), index=False)
    salida = os.path.join(DATADIR, "china_us_panel_analisis_v5_1990_2023.csv")
    df.to_csv(salida, index=False)
    print("\nEscrito: {}\n{} obs x {} vars".format(salida, *df.shape))
    print("\nListo.")


if __name__ == "__main__":
    main()

Panel: 68 obs x 142 vars

1. Segunda ronda de raiz unitaria
          Pais          Serie ADF Δ (c+t) PP Δ (c+t) ZA sobre Δ [año]   ADF Δ²               Conclusion
         China ln_labour_prod       -1.94      -2.00     -2.91 [2004] -4.98*** I(2): excluir de niveles
         China  ln_gdp_pc_pwt       -2.08   -3.99***     -3.57 [2002] -9.63***                I(1) (PP)
         China   pub_debt_gdp       -1.17   -7.39***            — [—]    -1.51                I(1) (PP)
         China  urban_pop_pct      -3.15*      -3.11   -5.29** [2010] -9.93***         I(1) con quiebre
         China             hc       -0.99      -1.88            — [—] -4.38*** I(2): excluir de niveles
         China        IIC_alt     -3.87**     -3.40*    -4.84* [2002] -5.64***                     I(1)
Estados Unidos    hh_debt_gdp       -2.63      -2.28            — [—] -7.72*** I(2): excluir de niveles
Estados Unidos  urban_pop_pct        2.13      -2.25            — [—]  -3.28** I(2): excluir de niveles
Esta

Script de estimación
Este es el primero que produce resultados sustantivos. Guárdalo con %%writefile /content/estimar_ardl.py. Imprime todo en consola sin maquetar: primero validamos los números y después los llevamos a Word.

In [ ]:
# -*- coding: utf-8 -*-
"""
ESTIMACION ARDL
Familia A: ln_gdp_pc_pwt (I(1))  -> ARDL con pruebas de limites y ECM
Familia B: gdp_growth_pct (I(0)) -> regresion dinamica con errores HAC

Salidas en /content/outputs:
    resultados_familiaA_bounds.csv
    resultados_familiaA_largo_plazo.csv
    resultados_familiaB_corto_plazo.csv
    diagnosticos_residuales.csv
"""

import os, warnings, itertools
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

import statsmodels.api as sm
from statsmodels.tsa.ardl import ARDL, UECM, ardl_select_order
from statsmodels.stats.diagnostic import (acorr_breusch_godfrey,
                                          het_breuschpagan, acorr_ljungbox)
from statsmodels.stats.stattools import jarque_bera

PANEL  = "/content/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
OUTDIR = "/content/outputs"
PAISES = {"CN": "China", "US": "Estados Unidos"}

MODELOS = {
    "M1": ["inv_gfcf_gdp", "trade_gdp", "inflation_cpi_pct"],
    "M3": ["credit_priv_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M4": ["bis_pvt_credit_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M5": ["pub_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M6": ["d_hh_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M7": ["corp_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M8": ["ratio_ind_agr_rec", "inv_gfcf_gdp", "trade_gdp"],
    "M9": ["IIC_alt", "trade_gdp"],
}
SOLO_US = {"M6", "M7"}
DUMMY = {"CN": ["q2002", "q2014"], "US": ["q2008"]}

MAXLAG, MAXORDER = 2, 2


def est(p):
    if p is None or not np.isfinite(p): return ""
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""


def preparar(df, cod, dep, regs, con_dummies=True):
    sub = df[df["country"] == cod].sort_values("year").set_index("year")
    cols = [dep] + [c for c in regs if c in sub.columns]
    faltan = [c for c in [dep] + regs if c not in sub.columns]
    d = sub[cols].dropna()
    fixed = None
    if con_dummies:
        dcols = [c for c in DUMMY.get(cod, []) if c in sub.columns]
        if dcols:
            fixed = sub.loc[d.index, dcols]
            fixed = fixed.loc[:, fixed.std() > 0]
            if fixed.shape[1] == 0: fixed = None
    y = d[dep].astype(float)
    X = d[[c for c in cols if c != dep]].astype(float)
    return y, X, fixed, faltan


# ==================================================== FAMILIA A
def familia_A(df):
    fb, fl, diag = [], [], []
    for cod, pais in PAISES.items():
        for mid, regs in MODELOS.items():
            if mid in SOLO_US and cod != "US": continue
            y, X, fixed, faltan = preparar(df, cod, "ln_gdp_pc_pwt", regs)
            if faltan or len(y) < 20:
                fb.append([pais, mid, "—", "—", "—", "—",
                           "n insuficiente o columnas ausentes ({})".format(len(y))])
                continue
            try:
                sel = ardl_select_order(y, MAXLAG, X, MAXORDER, trend="c",
                                        ic="aic", fixed=fixed)
                orden_y = sel.model.ardl_order[0]
                orden_x = sel.model.ardl_order[1:]
                u = UECM(y, orden_y, X, orden_x, trend="c", fixed=fixed).fit()
                bt = u.bounds_test(case=3)
                fstat = float(np.asarray(bt.stat).ravel()[0])
                cv = bt.crit_vals
                try:
                    i5 = float(cv.loc["5%", "lower"]); s5 = float(cv.loc["5%", "upper"])
                except Exception:
                    i5, s5 = np.nan, np.nan
                veredicto = ("cointegracion" if np.isfinite(s5) and fstat > s5 else
                             "no cointegracion" if np.isfinite(i5) and fstat < i5 else
                             "zona no concluyente")
                fb.append([pais, mid, "({}, {})".format(orden_y, list(orden_x)),
                           "{:.3f}".format(fstat),
                           "{:.2f}".format(i5) if np.isfinite(i5) else "—",
                           "{:.2f}".format(s5) if np.isfinite(s5) else "—",
                           veredicto])

                # coeficientes de largo plazo (relacion cointegrante)
                cip, cise = u.ci_params, u.ci_bse
                for nom in cip.index:
                    b, se = float(cip[nom]), float(cise[nom])
                    t = b / se if se else np.nan
                    from scipy.stats import t as tdist
                    p = 2 * (1 - tdist.cdf(abs(t), max(u.df_resid, 1))) if np.isfinite(t) else np.nan
                    fl.append([pais, mid, nom, "{:.4f}{}".format(b, est(p)),
                               "{:.4f}".format(se), "{:.2f}".format(t) if np.isfinite(t) else "—"])

                # coeficiente de correccion de error
                ect = [c for c in u.params.index if "ln_gdp_pc_pwt.L1" in c
                       or c.endswith(".L1") and "ln_gdp_pc_pwt" in c]
                r = u.resid
                lb = acorr_ljungbox(r, lags=[2], return_df=True)
                bg = acorr_breusch_godfrey(u, nlags=2)
                jb = jarque_bera(r)
                diag.append([pais, mid, "{}".format(len(y)),
                             "{:.3f}".format(float(lb["lb_pvalue"].iloc[0])),
                             "{:.3f}".format(bg[3]),
                             "{:.3f}".format(jb[1]),
                             "{:.3f}".format(u.rsquared) if hasattr(u, "rsquared") else "—"])
            except Exception as e:
                fb.append([pais, mid, "—", "—", "—", "—", "error: {}".format(e)[:60]])
    return (pd.DataFrame(fb, columns=["Pais", "Modelo", "Orden (p, q)",
                                      "F de limites", "I(0) 5%", "I(1) 5%",
                                      "Veredicto"]),
            pd.DataFrame(fl, columns=["Pais", "Modelo", "Variable",
                                      "Coef. largo plazo", "EE", "t"]),
            pd.DataFrame(diag, columns=["Pais", "Modelo", "n",
                                        "Ljung-Box p", "Breusch-Godfrey p",
                                        "Jarque-Bera p", "R2"]))


# ==================================================== FAMILIA B
def familia_B(df):
    filas = []
    for cod, pais in PAISES.items():
        for mid, regs in MODELOS.items():
            if mid in SOLO_US and cod != "US": continue
            y, X, fixed, faltan = preparar(df, cod, "gdp_growth_pct", regs)
            if faltan or len(y) < 20: continue
            Z = X.copy()
            Z["y_lag1"] = y.shift(1)
            for c in X.columns:
                Z[c + "_lag1"] = X[c].shift(1)
            if fixed is not None:
                for c in fixed.columns: Z[c] = fixed[c]
            dat = pd.concat([y, Z], axis=1).dropna()
            yy = dat.iloc[:, 0]
            XX = sm.add_constant(dat.iloc[:, 1:])
            try:
                m = sm.OLS(yy, XX).fit(cov_type="HAC",
                                       cov_kwds={"maxlags": 2, "use_correction": True})
                for nom in m.params.index:
                    if nom == "const": continue
                    filas.append([pais, mid, nom,
                                  "{:.4f}{}".format(m.params[nom], est(m.pvalues[nom])),
                                  "{:.4f}".format(m.bse[nom]),
                                  "{:.3f}".format(m.rsquared_adj), str(int(m.nobs))])
            except Exception as e:
                filas.append([pais, mid, "error", str(e)[:50], "—", "—", "—"])
    return pd.DataFrame(filas, columns=["Pais", "Modelo", "Variable",
                                        "Coeficiente", "EE (HAC)", "R2 aj.", "n"])


def main():
    os.makedirs(OUTDIR, exist_ok=True)
    df = pd.read_csv(PANEL)
    df["country"] = df["country"].astype(str).str.strip().str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    print("Panel: {} obs x {} vars\n".format(*df.shape))

    print("FAMILIA A: cointegracion con ln_gdp_pc_pwt")
    fb, fl, dg = familia_A(df)
    print(fb.to_string(index=False))
    print("\nCoeficientes de largo plazo")
    print(fl.to_string(index=False))
    print("\nDiagnosticos residuales")
    print(dg.to_string(index=False))

    print("\n\nFAMILIA B: dinamica de corto plazo con gdp_growth_pct")
    fbb = familia_B(df)
    print(fbb.to_string(index=False))

    fb.to_csv(os.path.join(OUTDIR, "resultados_familiaA_bounds.csv"), index=False)
    fl.to_csv(os.path.join(OUTDIR, "resultados_familiaA_largo_plazo.csv"), index=False)
    dg.to_csv(os.path.join(OUTDIR, "diagnosticos_residuales.csv"), index=False)
    fbb.to_csv(os.path.join(OUTDIR, "resultados_familiaB_corto_plazo.csv"), index=False)
    print("\nListo. Cuatro archivos escritos en {}".format(OUTDIR))


if __name__ == "__main__":
    main()

In [ ]:
import sys, os, subprocess
print("Python:", sys.version.split()[0])
print("Existe el archivo:", os.path.exists("/content/estimar_ardl.py"))

import statsmodels, numpy, pandas
print("statsmodels:", statsmodels.__version__)

try:
    from statsmodels.tsa.ardl import ARDL, UECM, ardl_select_order
    print("Importacion de ARDL: correcta")
except Exception as e:
    print("Importacion de ARDL: FALLO ->", e)

import pandas as pd, numpy as np
df = pd.read_csv("/content/data_clean/china_us_panel_analisis_v5_1990_2023.csv")
df["country"] = df["country"].astype(str).str.upper()
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
print("\nPanel:", df.shape)

# prueba minima: M3 para China
sub = df[df["country"] == "CN"].sort_values("year").set_index("year")
cols = ["ln_gdp_pc_pwt", "credit_priv_gdp", "inv_gfcf_gdp", "trade_gdp"]
print("Columnas presentes:", [c for c in cols if c in sub.columns])
d = sub[cols].dropna()
print("Observaciones utilizables:", len(d), "periodo:", d.index.min(), "-", d.index.max())

y = d["ln_gdp_pc_pwt"].astype(float)
X = d[["credit_priv_gdp", "inv_gfcf_gdp", "trade_gdp"]].astype(float)

try:
    from statsmodels.tsa.ardl import UECM, ardl_select_order
    sel = ardl_select_order(y, 2, X, 2, trend="c", ic="aic")
    print("Orden seleccionado:", sel.model.ardl_order)
    u = UECM(y, sel.model.ardl_order[0], X, sel.model.ardl_order[1:],
             trend="c").fit()
    bt = u.bounds_test(case=3)
    print("\nF de limites:", float(np.asarray(bt.stat).ravel()[0]))
    print("Valores criticos:\n", bt.crit_vals)
    print("\nCoeficientes de largo plazo:\n", u.ci_params)
except Exception as e:
    import traceback; traceback.print_exc()
    print("\nFALLO EN LA ESTIMACION ->", type(e).__name__, e)

Python: 3.12.13
Existe el archivo: False
statsmodels: 0.14.6
Importacion de ARDL: correcta

Panel: (68, 146)
Columnas presentes: ['ln_gdp_pc_pwt', 'credit_priv_gdp', 'inv_gfcf_gdp', 'trade_gdp']
Observaciones utilizables: 34 periodo: 1990 - 2023
Orden seleccionado: (1, 1, 1, 2)

FALLO EN LA ESTIMACION -> TypeError order must be None, a positive integer, or a dict containing positive integers or None


Traceback (most recent call last):
  File "/tmp/ipykernel_3569/4188493767.py", line 34, in <cell line: 0>
    u = UECM(y, sel.model.ardl_order[0], X, sel.model.ardl_order[1:],
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/ardl/model.py", line 1762, in __init__
    super().__init__(
  File "/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/ardl/model.py", line 377, in __init__
    self._order = self._check_order(order)
                  ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/ardl/model.py", line 1796, in _check_order
    raise TypeError(
TypeError: order must be None, a positive integer, or a dict containing positive integers or None


In [ ]:
import numpy as np, pandas as pd
from statsmodels.tsa.ardl import UECM, ardl_select_order

df = pd.read_csv("/content/data_clean/china_us_panel_analisis_v5_1990_2023.csv")
df["country"] = df["country"].astype(str).str.upper()
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)

sub = df[df["country"] == "CN"].sort_values("year").set_index("year")
d = sub[["ln_gdp_pc_pwt", "credit_priv_gdp", "inv_gfcf_gdp", "trade_gdp"]].dropna()
y = d["ln_gdp_pc_pwt"].astype(float)
X = d[["credit_priv_gdp", "inv_gfcf_gdp", "trade_gdp"]].astype(float)

sel = ardl_select_order(y, 2, X, 2, trend="c", ic="aic")
print("Orden ARDL seleccionado:", sel.model.ardl_order)

u = None

# --- VIA 1: conversion directa desde el ARDL ya seleccionado
try:
    u = UECM.from_ardl(sel.model).fit()
    print("Via 1 (from_ardl): correcta")
except Exception as e:
    print("Via 1 fallo ->", type(e).__name__, e)

# --- VIA 2: diccionario de ordenes, uno por regresor
if u is None:
    try:
        ordenes = {c: max(1, int(k))
                   for c, k in zip(X.columns, sel.model.ardl_order[1:])}
        print("Diccionario de ordenes:", ordenes)
        u = UECM(y, int(sel.model.ardl_order[0]), X, ordenes, trend="c").fit()
        print("Via 2 (diccionario): correcta")
    except Exception as e:
        import traceback; traceback.print_exc()
        print("Via 2 fallo ->", type(e).__name__, e)

# --- RESULTADOS
if u is not None:
    bt = u.bounds_test(case=3)
    print("\nF de limites: {:.4f}".format(float(np.asarray(bt.stat).ravel()[0])))
    print("\nValores criticos:")
    print(bt.crit_vals)
    print("\nCoeficientes de largo plazo (relacion cointegrante):")
    print(u.ci_params.round(4))
    print("\nErrores estandar:")
    print(u.ci_bse.round(4))
    print("\nCoeficientes del modelo completo:")
    print(u.params.round(4))
    print("\nn = {}, gl residuales = {}".format(int(u.nobs), int(u.df_resid)))

Orden ARDL seleccionado: (1, 1, 1, 2)
Via 1 (from_ardl): correcta

F de limites: 9.9809

Valores criticos:
               lower     upper
percentile                    
90.0        2.456553  3.516144
95.0        2.877209  4.010500
99.0        3.779853  5.050468
99.9        4.985268  6.402243

Coeficientes de largo plazo (relacion cointegrante):
const             -6.5413
ln_gdp_pc_pwt      1.0000
credit_priv_gdp   -0.0103
inv_gfcf_gdp      -0.0442
trade_gdp         -0.0221
Name: ci_params, dtype: float64

Errores estandar:
const              0.9657
ln_gdp_pc_pwt      0.0000
credit_priv_gdp    0.0046
inv_gfcf_gdp       0.0200
trade_gdp          0.0076
Name: ci_bse, dtype: float64

Coeficientes del modelo completo:
const                   0.3409
ln_gdp_pc_pwt.L1       -0.0521
credit_priv_gdp.L1      0.0005
inv_gfcf_gdp.L1         0.0023
trade_gdp.L1            0.0012
D.credit_priv_gdp.L0   -0.0014
D.inv_gfcf_gdp.L0       0.0049
D.trade_gdp.L0         -0.0005
D.trade_gdp.L1         -0.0016

In [ ]:
# -*- coding: utf-8 -*-
"""
ESTIMACION ARDL v2  (interfaz corregida)
Familia A: ln_gdp_pc_pwt (I(1)) -> UECM.from_ardl, pruebas de limites, ECM.
Familia B: gdp_growth_pct (I(0)) -> regresion dinamica con errores HAC.

IMPORTANTE: los coeficientes de largo plazo se reportan como -ci_params,
porque statsmodels normaliza la relacion cointegrante en la forma y + b'x + c = 0.
"""

import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

import statsmodels.api as sm
from statsmodels.tsa.ardl import UECM, ardl_select_order
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_breuschpagan
from statsmodels.stats.stattools import jarque_bera
from scipy.stats import t as tdist

PANEL  = "/content/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
OUTDIR = "/content/outputs"
DEP_A, DEP_B = "ln_gdp_pc_pwt", "gdp_growth_pct"
PAISES = {"CN": "China", "US": "Estados Unidos"}

MODELOS = {
    "M1": ["inv_gfcf_gdp", "trade_gdp", "inflation_cpi_pct"],
    "M3": ["credit_priv_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M4": ["bis_pvt_credit_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M5": ["pub_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M6": ["d_hh_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M7": ["corp_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M8": ["ratio_ind_agr_rec", "inv_gfcf_gdp", "trade_gdp"],
    "M9": ["IIC_alt", "trade_gdp"],
}
SOLO_US = {"M6", "M7"}
MAXLAG = MAXORDER = 2


def est(p):
    if p is None or not np.isfinite(p): return ""
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""


def pval(t_, gl):
    if not np.isfinite(t_) or gl <= 0: return np.nan
    return 2 * (1 - tdist.cdf(abs(t_), gl))


def datos(df, cod, dep, regs):
    sub = df[df["country"] == cod].sort_values("year").set_index("year")
    faltan = [c for c in [dep] + regs if c not in sub.columns]
    if faltan: return None, None, faltan
    d = sub[[dep] + regs].dropna()
    return d[dep].astype(float), d[regs].astype(float), []


def familia_A(df):
    bnd, lrg, dgn = [], [], []
    for cod, pais in PAISES.items():
        for mid, regs in MODELOS.items():
            if mid in SOLO_US and cod != "US": continue
            y, X, faltan = datos(df, cod, DEP_A, regs)
            if faltan:
                bnd.append([pais, mid, "—", "—", "—", "—", "—",
                            "ausentes: {}".format(faltan)]); continue
            if len(y) < 20:
                bnd.append([pais, mid, "—", "—", "—", "—", str(len(y)),
                            "n insuficiente"]); continue
            try:
                sel = ardl_select_order(y, MAXLAG, X, MAXORDER,
                                        trend="c", ic="aic")
                u = UECM.from_ardl(sel.model).fit()

                bt = u.bounds_test(case=3)
                F = float(np.asarray(bt.stat).ravel()[0])
                cv = bt.crit_vals
                lo5, up5 = float(cv.loc[95.0, "lower"]), float(cv.loc[95.0, "upper"])
                ver = ("cointegracion" if F > up5 else
                       "no cointegracion" if F < lo5 else "no concluyente")

                # ---- termino de correccion de error
                nom_ect = DEP_A + ".L1"
                lam = float(u.params.get(nom_ect, np.nan))
                se_l = float(u.bse.get(nom_ect, np.nan))
                t_l = lam / se_l if se_l else np.nan
                p_l = pval(t_l, u.df_resid)
                vm = (np.log(0.5) / np.log(1 + lam)
                      if -1 < lam < 0 else np.nan)

                bnd.append([pais, mid, str(sel.model.ardl_order),
                            "{:.3f}".format(F),
                            "{:.2f} / {:.2f}".format(lo5, up5),
                            "{:.4f}{}".format(lam, est(p_l)),
                            "{:.1f}".format(vm) if np.isfinite(vm) else "—",
                            ver])

                # ---- largo plazo, con el signo invertido
                ci, se = u.ci_params, u.ci_bse
                for nom in ci.index:
                    if nom == DEP_A: continue
                    b = -float(ci[nom])
                    s = float(se[nom])
                    t_ = b / s if s else np.nan
                    p_ = pval(t_, u.df_resid)
                    lrg.append([pais, mid, nom, "{:.4f}{}".format(b, est(p_)),
                                "{:.4f}".format(s),
                                "{:.2f}".format(t_) if np.isfinite(t_) else "—"])

                # ---- diagnosticos
                r = np.asarray(u.resid, float)
                try: bg = acorr_breusch_godfrey(u, nlags=2)[3]
                except Exception: bg = np.nan
                try:
                    ex = sm.add_constant(np.arange(len(r), dtype=float))
                    bp = het_breuschpagan(r, ex)[1]
                except Exception: bp = np.nan
                jb = jarque_bera(r)[1]
                dgn.append([pais, mid, str(int(u.nobs)), str(int(u.df_resid)),
                            "{:.3f}".format(bg) if np.isfinite(bg) else "—",
                            "{:.3f}".format(bp) if np.isfinite(bp) else "—",
                            "{:.3f}".format(jb) if np.isfinite(jb) else "—"])
            except Exception as e:
                bnd.append([pais, mid, "—", "—", "—", "—", "—",
                            "error: {}: {}".format(type(e).__name__, e)[:70]])
    return (pd.DataFrame(bnd, columns=["Pais", "Modelo", "Orden ARDL",
                                       "F limites", "I(0)/I(1) 5%", "ECT",
                                       "Vida media", "Veredicto"]),
            pd.DataFrame(lrg, columns=["Pais", "Modelo", "Variable",
                                       "Largo plazo", "EE", "t"]),
            pd.DataFrame(dgn, columns=["Pais", "Modelo", "n", "gl",
                                       "Breusch-Godfrey p", "Breusch-Pagan p",
                                       "Jarque-Bera p"]))


def familia_B(df):
    filas = []
    for cod, pais in PAISES.items():
        for mid, regs in MODELOS.items():
            if mid in SOLO_US and cod != "US": continue
            y, X, faltan = datos(df, cod, DEP_B, regs)
            if faltan or len(y) < 20: continue
            Z = X.copy()
            Z[DEP_B + "_L1"] = y.shift(1)
            for c in X.columns: Z[c + "_L1"] = X[c].shift(1)
            dat = pd.concat([y, Z], axis=1).dropna()
            try:
                m = sm.OLS(dat.iloc[:, 0], sm.add_constant(dat.iloc[:, 1:])).fit(
                    cov_type="HAC", cov_kwds={"maxlags": 2, "use_correction": True})
                for nom in m.params.index:
                    if nom == "const": continue
                    filas.append([pais, mid, nom,
                                  "{:.4f}{}".format(m.params[nom], est(m.pvalues[nom])),
                                  "{:.4f}".format(m.bse[nom]),
                                  "{:.3f}".format(m.rsquared_adj),
                                  str(int(m.nobs))])
            except Exception as e:
                filas.append([pais, mid, "error", str(e)[:50], "—", "—", "—"])
    return pd.DataFrame(filas, columns=["Pais", "Modelo", "Variable",
                                        "Coeficiente", "EE (HAC)",
                                        "R2 aj.", "n"])


def main():
    os.makedirs(OUTDIR, exist_ok=True)
    df = pd.read_csv(PANEL)
    df["country"] = df["country"].astype(str).str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    print("Panel: {} obs x {} vars\n".format(*df.shape))

    print("=" * 78)
    print("FAMILIA A. Cointegracion, dependiente = ln PIB per capita real")
    print("=" * 78)
    bnd, lrg, dgn = familia_A(df)
    print(bnd.to_string(index=False))
    print("\nCoeficientes de largo plazo (signo ya corregido)")
    print(lrg.to_string(index=False))
    print("\nDiagnosticos residuales")
    print(dgn.to_string(index=False))

    print("\n" + "=" * 78)
    print("FAMILIA B. Corto plazo, dependiente = tasa de crecimiento del PIB")
    print("=" * 78)
    fb = familia_B(df)
    print(fb.to_string(index=False))

    bnd.to_csv(os.path.join(OUTDIR, "A_bounds_ect.csv"), index=False)
    lrg.to_csv(os.path.join(OUTDIR, "A_largo_plazo.csv"), index=False)
    dgn.to_csv(os.path.join(OUTDIR, "A_diagnosticos.csv"), index=False)
    fb.to_csv(os.path.join(OUTDIR, "B_corto_plazo.csv"), index=False)
    print("\nListo. Cuatro archivos en {}".format(OUTDIR))


if __name__ == "__main__":
    main()

Panel: 68 obs x 146 vars

FAMILIA A. Cointegracion, dependiente = ln PIB per capita real
          Pais Modelo   Orden ARDL F limites I(0)/I(1) 5%        ECT Vida media                                                              Veredicto
         China     M1 (2, 2, 2, 2)    14.896  2.88 / 4.01  -0.0258**       26.6                                                          cointegracion
         China     M3 (1, 1, 1, 2)     9.981  2.88 / 4.01  -0.0521**       13.0                                                          cointegracion
         China     M4            —         —            —          —          — error: ValueError: All included exog variables must have a lag length 
         China     M5       (2, 2)    21.770  3.80 / 4.81 -0.0209***       32.9                                                          cointegracion
         China     M8            —         —            —          —          — error: ValueError: All included exog variables must have a lag length 
     

Script corregido
Fuerza orden mínimo uno en todos los regresores, exige ECT negativo y significativo para declarar cointegración, arregla los dos diagnósticos y calcula los multiplicadores netos automáticamente.

In [43]:
# -*- coding: utf-8 -*-
"""
ESTIMACION ARDL v3
Correcciones respecto de v2:
 1. Orden minimo 1 para TODOS los regresores: la seleccion por AIC no puede
    eliminar la variable focal de cada hipotesis.
 2. El veredicto de cointegracion exige F por encima del limite superior Y
    un ECT negativo y significativo al 10 %.
 3. Breusch-Pagan sobre los regresores del modelo (antes: sobre una tendencia).
 4. Ljung-Box como respaldo cuando Breusch-Godfrey falla.
 5. Multiplicadores netos de la familia B: (suma de coeficientes)/(1 - rho).
"""

import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

import statsmodels.api as sm
from statsmodels.tsa.ardl import UECM, ardl_select_order
from statsmodels.stats.diagnostic import (acorr_breusch_godfrey,
                                          het_breuschpagan, acorr_ljungbox)
from statsmodels.stats.stattools import jarque_bera
from scipy.stats import t as tdist

PANEL  = "/content/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
OUTDIR = "/content/outputs"
DEP_A, DEP_B = "ln_gdp_pc_pwt", "gdp_growth_pct"
PAISES = {"CN": "China", "US": "Estados Unidos"}

MODELOS = {
    "M1": ["inv_gfcf_gdp", "trade_gdp", "inflation_cpi_pct"],
    "M3": ["credit_priv_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M4": ["bis_pvt_credit_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M5": ["pub_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M6": ["d_hh_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M7": ["corp_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M8": ["ratio_ind_agr_rec", "inv_gfcf_gdp", "trade_gdp"],
    "M9": ["IIC_alt", "trade_gdp"],
}
SOLO_US = {"M6", "M7"}
MAXLAG = MAXORDER = 2


def est(p):
    if p is None or not np.isfinite(p): return ""
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""


def pval(t_, gl):
    if not np.isfinite(t_) or gl <= 0: return np.nan
    return 2 * (1 - tdist.cdf(abs(t_), gl))


def datos(df, cod, dep, regs):
    sub = df[df["country"] == cod].sort_values("year").set_index("year")
    faltan = [c for c in [dep] + regs if c not in sub.columns]
    if faltan: return None, None, faltan
    d = sub[[dep] + regs].dropna()
    return d[dep].astype(float), d[regs].astype(float), []


def familia_A(df):
    bnd, lrg, dgn = [], [], []
    for cod, pais in PAISES.items():
        for mid, regs in MODELOS.items():
            if mid in SOLO_US and cod != "US": continue
            y, X, faltan = datos(df, cod, DEP_A, regs)
            if faltan or len(y) < 20:
                bnd.append([pais, mid, "—", "—", "—", "—", "—",
                            "ausentes {}".format(faltan) if faltan
                            else "n insuficiente"]); continue
            try:
                # ---- seleccion con PISO 1 en todos los regresores
                sel = ardl_select_order(y, MAXLAG, X, MAXORDER,
                                        trend="c", ic="aic")
                ord_y = max(1, int(sel.model.ardl_order[0]))
                brutos = list(sel.model.ardl_order[1:])
                ordenes = {}
                for i, c in enumerate(X.columns):
                    k = int(brutos[i]) if i < len(brutos) else 1
                    ordenes[c] = max(1, k)      # nunca se elimina un regresor

                u = UECM(y, ord_y, X, ordenes, trend="c").fit()

                bt = u.bounds_test(case=3)
                F = float(np.asarray(bt.stat).ravel()[0])
                cv = bt.crit_vals
                lo5, up5 = float(cv.loc[95.0, "lower"]), float(cv.loc[95.0, "upper"])

                nom_ect = DEP_A + ".L1"
                lam = float(u.params.get(nom_ect, np.nan))
                se_l = float(u.bse.get(nom_ect, np.nan))
                t_l = lam / se_l if se_l else np.nan
                p_l = pval(t_l, u.df_resid)
                vm = np.log(0.5) / np.log(1 + lam) if -1 < lam < 0 else np.nan

                ect_valido = (np.isfinite(lam) and -1 < lam < 0
                              and np.isfinite(p_l) and p_l < .10)
                if F > up5 and ect_valido:
                    ver = "cointegracion"
                elif F > up5 and not ect_valido:
                    ver = "F alto pero ECT invalido"
                elif F < lo5:
                    ver = "sin cointegracion"
                else:
                    ver = "zona no concluyente"

                bnd.append([pais, mid,
                            "y={} | {}".format(ord_y, ordenes),
                            "{:.3f}".format(F),
                            "{:.2f}/{:.2f}".format(lo5, up5),
                            "{:.4f}{}".format(lam, est(p_l)),
                            "{:.1f}".format(vm) if np.isfinite(vm) else "—",
                            ver])

                ci, se = u.ci_params, u.ci_bse
                for nom in ci.index:
                    if nom == DEP_A: continue
                    b, s = -float(ci[nom]), float(se[nom])
                    t_ = b / s if s else np.nan
                    lrg.append([pais, mid, nom,
                                "{:.4f}{}".format(b, est(pval(t_, u.df_resid))),
                                "{:.4f}".format(s),
                                "{:.2f}".format(t_) if np.isfinite(t_) else "—"])

                # ---- diagnosticos corregidos
                r = np.asarray(u.resid, float)
                try: bg = float(acorr_breusch_godfrey(u, nlags=2)[3])
                except Exception: bg = np.nan
                try: lb = float(acorr_ljungbox(r, lags=[2],
                                               return_df=True)["lb_pvalue"].iloc[0])
                except Exception: lb = np.nan
                try:
                    ex = np.asarray(u.model.exog, float)
                    if ex.ndim == 1: ex = ex.reshape(-1, 1)
                    if not np.allclose(ex[:, 0], 1): ex = sm.add_constant(ex)
                    bp = float(het_breuschpagan(r, ex)[1])
                except Exception: bp = np.nan
                jb = float(jarque_bera(r)[1])
                dgn.append([pais, mid, str(int(u.nobs)), str(int(u.df_resid)),
                            "{:.3f}".format(bg) if np.isfinite(bg) else "—",
                            "{:.3f}".format(lb) if np.isfinite(lb) else "—",
                            "{:.3f}".format(bp) if np.isfinite(bp) else "—",
                            "{:.3f}".format(jb)])
            except Exception as e:
                bnd.append([pais, mid, "—", "—", "—", "—", "—",
                            "error {}: {}".format(type(e).__name__, e)[:70]])
    return (pd.DataFrame(bnd, columns=["Pais", "Modelo", "Ordenes", "F limites",
                                       "I(0)/I(1) 5%", "ECT", "Vida media",
                                       "Veredicto"]),
            pd.DataFrame(lrg, columns=["Pais", "Modelo", "Variable",
                                       "Largo plazo", "EE", "t"]),
            pd.DataFrame(dgn, columns=["Pais", "Modelo", "n", "gl",
                                       "Breusch-Godfrey p", "Ljung-Box p",
                                       "Breusch-Pagan p", "Jarque-Bera p"]))


def familia_B(df):
    coef, mult = [], []
    for cod, pais in PAISES.items():
        for mid, regs in MODELOS.items():
            if mid in SOLO_US and cod != "US": continue
            y, X, faltan = datos(df, cod, DEP_B, regs)
            if faltan or len(y) < 20: continue
            Z = X.copy()
            Z["rho"] = y.shift(1)
            for c in X.columns: Z[c + "_L1"] = X[c].shift(1)
            dat = pd.concat([y, Z], axis=1).dropna()
            try:
                m = sm.OLS(dat.iloc[:, 0],
                           sm.add_constant(dat.iloc[:, 1:])).fit(
                    cov_type="HAC", cov_kwds={"maxlags": 2, "use_correction": True})
                rho = float(m.params.get("rho", 0.0))
                for nom in m.params.index:
                    if nom == "const": continue
                    coef.append([pais, mid, nom,
                                 "{:.4f}{}".format(m.params[nom],
                                                   est(m.pvalues[nom])),
                                 "{:.4f}".format(m.bse[nom]),
                                 "{:.3f}".format(m.rsquared_adj),
                                 str(int(m.nobs))])
                # multiplicadores netos
                den = 1.0 - rho
                for c in X.columns:
                    b0 = float(m.params.get(c, 0.0))
                    b1 = float(m.params.get(c + "_L1", 0.0))
                    net = (b0 + b1) / den if abs(den) > 1e-8 else np.nan
                    mult.append([pais, mid, c, "{:.4f}".format(b0),
                                 "{:.4f}".format(b1), "{:.4f}".format(rho),
                                 "{:.4f}".format(net) if np.isfinite(net) else "—"])
            except Exception as e:
                coef.append([pais, mid, "error", str(e)[:50], "—", "—", "—"])
    return (pd.DataFrame(coef, columns=["Pais", "Modelo", "Variable",
                                        "Coeficiente", "EE (HAC)", "R2 aj.", "n"]),
            pd.DataFrame(mult, columns=["Pais", "Modelo", "Variable",
                                        "Contemporaneo", "Rezagado", "rho",
                                        "Multiplicador neto"]))


def main():
    os.makedirs(OUTDIR, exist_ok=True)
    df = pd.read_csv(PANEL)
    df["country"] = df["country"].astype(str).str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    print("Panel: {} obs x {} vars\n".format(*df.shape))

    print("=" * 76)
    print("FAMILIA A. Cointegracion (dependiente: ln PIB per capita real)")
    print("=" * 76)
    bnd, lrg, dgn = familia_A(df)
    print(bnd.to_string(index=False))
    print("\nLargo plazo (signo corregido)")
    print(lrg.to_string(index=False))
    print("\nDiagnosticos")
    print(dgn.to_string(index=False))

    print("\n" + "=" * 76)
    print("FAMILIA B. Corto plazo (dependiente: tasa de crecimiento)")
    print("=" * 76)
    coef, mult = familia_B(df)
    print("\nMultiplicadores netos")
    print(mult.to_string(index=False))

    for nombre, obj in [("A_bounds_ect_v3", bnd), ("A_largo_plazo_v3", lrg),
                        ("A_diagnosticos_v3", dgn), ("B_coeficientes_v3", coef),
                        ("B_multiplicadores_v3", mult)]:
        obj.to_csv(os.path.join(OUTDIR, nombre + ".csv"), index=False)
    print("\nListo. Cinco archivos en {}".format(OUTDIR))


if __name__ == "__main__":
    main()

Panel: 68 obs x 146 vars

FAMILIA A. Cointegracion (dependiente: ln PIB per capita real)
          Pais Modelo                                                            Ordenes F limites I(0)/I(1) 5%        ECT Vida media                Veredicto
         China     M1  y=2 | {'inv_gfcf_gdp': 2, 'trade_gdp': 2, 'inflation_cpi_pct': 2}    14.896    2.88/4.01  -0.0258**       26.6            cointegracion
         China     M3    y=1 | {'credit_priv_gdp': 1, 'inv_gfcf_gdp': 1, 'trade_gdp': 2}     9.981    2.88/4.01  -0.0521**       13.0            cointegracion
         China     M4 y=1 | {'bis_pvt_credit_gdp': 2, 'inv_gfcf_gdp': 1, 'trade_gdp': 2}    14.410    2.88/4.01  -0.0718**        9.3            cointegracion
         China     M5       y=2 | {'pub_debt_gdp': 2, 'inv_gfcf_gdp': 1, 'trade_gdp': 1}     8.363    2.88/4.01    -0.0004     1802.2 F alto pero ECT invalido
         China     M8  y=1 | {'ratio_ind_agr_rec': 1, 'inv_gfcf_gdp': 1, 'trade_gdp': 1}    15.746    2.88/4.01 -0.0

La solución correcta, y que además nos da una validación cruzada gratuita, es reconstruir la ecuación del UECM a mano y estimarla por mínimos cuadrados ordinarios. Si los coeficientes replicados coinciden con los de statsmodels, la especificación es correcta y los diagnósticos calculados sobre ese objeto OLS sí son válidos.

In [44]:
# -*- coding: utf-8 -*-
"""
DIAGNOSTICOS VALIDOS POR REPLICACION EN MCO
La ecuacion del UECM se reconstruye a mano:
   d_y_t = c + lambda*y_{t-1} + sum_i beta_i*x_{i,t-1}
           + sum_j sum_k gamma*d_x_{j,t-k} + sum_m phi*d_y_{t-m}
Si lambda replicado == ECT de statsmodels, la especificacion es correcta y
los diagnosticos sobre el objeto MCO son confiables.
"""
import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import statsmodels.api as sm
from statsmodels.tsa.ardl import UECM, ardl_select_order
from statsmodels.stats.diagnostic import (acorr_breusch_godfrey,
                                          het_breuschpagan, acorr_ljungbox)
from statsmodels.stats.stattools import jarque_bera, durbin_watson

PANEL  = "/content/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
OUTDIR = "/content/outputs"
DEP    = "ln_gdp_pc_pwt"
PAISES = {"CN": "China", "US": "Estados Unidos"}

# M5 de China reespecificado con DOS regresores (n=27 no admite tres)
MODELOS = {
    "M1": ["inv_gfcf_gdp", "trade_gdp", "inflation_cpi_pct"],
    "M3": ["credit_priv_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M4": ["bis_pvt_credit_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M5": ["pub_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M5b": ["pub_debt_gdp", "trade_gdp"],
    "M6": ["d_hh_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M7": ["corp_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M8": ["ratio_ind_agr_rec", "inv_gfcf_gdp", "trade_gdp"],
    "M9": ["IIC_alt", "trade_gdp"],
}
SOLO_US = {"M6", "M7"}


def est(p):
    if p is None or not np.isfinite(p): return ""
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""


def construir(y, X, ord_y, ordenes):
    """Matriz de diseno de la ecuacion en forma de correccion de error."""
    Z = pd.DataFrame(index=y.index)
    Z["y_L1"] = y.shift(1)
    for c in X.columns:
        Z[c + "_L1"] = X[c].shift(1)
    dy = y.diff()
    for m in range(1, ord_y):
        Z["d_y_L{}".format(m)] = dy.shift(m)
    for c in X.columns:
        dx = X[c].diff()
        for k in range(0, ordenes[c]):
            Z["d_{}_L{}".format(c, k)] = dx.shift(k)
    dat = pd.concat([dy.rename("d_y"), Z], axis=1).dropna()
    return dat


def main():
    os.makedirs(OUTDIR, exist_ok=True)
    df = pd.read_csv(PANEL)
    df["country"] = df["country"].astype(str).str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)

    filas, val = [], []
    for cod, pais in PAISES.items():
        for mid, regs in MODELOS.items():
            if mid in SOLO_US and cod != "US": continue
            if mid == "M5b" and cod != "CN": continue
            sub = df[df["country"] == cod].sort_values("year").set_index("year")
            if any(c not in sub.columns for c in [DEP] + regs): continue
            d = sub[[DEP] + regs].dropna()
            if len(d) < 20: continue
            y, X = d[DEP].astype(float), d[regs].astype(float)
            try:
                sel = ardl_select_order(y, 2, X, 2, trend="c", ic="aic")
                ord_y = max(1, int(sel.model.ardl_order[0]))
                brutos = list(sel.model.ardl_order[1:])
                ordenes = {c: max(1, int(brutos[i]) if i < len(brutos) else 1)
                           for i, c in enumerate(X.columns)}
                u = UECM(y, ord_y, X, ordenes, trend="c").fit()
                lam_sm = float(u.params.get(DEP + ".L1", np.nan))

                dat = construir(y, X, ord_y, ordenes)
                m = sm.OLS(dat["d_y"],
                           sm.add_constant(dat.drop(columns="d_y"))).fit()
                lam_ols = float(m.params.get("y_L1", np.nan))

                val.append([pais, mid, "{:.6f}".format(lam_sm),
                            "{:.6f}".format(lam_ols),
                            "{:.2e}".format(abs(lam_sm - lam_ols)),
                            "SI" if abs(lam_sm - lam_ols) < 1e-6 else "NO",
                            str(int(u.nobs)), str(int(m.nobs))])

                r = np.asarray(m.resid, float)
                bg = float(acorr_breusch_godfrey(m, nlags=2)[3])
                lb = float(acorr_ljungbox(r, lags=[2],
                                          return_df=True)["lb_pvalue"].iloc[0])
                bp = float(het_breuschpagan(r, m.model.exog)[1])
                jb = float(jarque_bera(r)[1])
                dw = float(durbin_watson(r))
                # Ljung-Box tambien sobre el residuo de statsmodels, para comparar
                r_sm = np.asarray(u.resid, float)
                r_sm = r_sm[np.isfinite(r_sm)]
                lb_sm = float(acorr_ljungbox(r_sm, lags=[2],
                                             return_df=True)["lb_pvalue"].iloc[0])

                filas.append([pais, mid, str(int(m.nobs)), str(int(m.df_resid)),
                              "{:.4f}{}".format(lam_ols,
                                                est(m.pvalues.get("y_L1", np.nan))),
                              "{:.3f}".format(bg), "{:.3f}".format(lb),
                              "{:.3f}".format(bp), "{:.3f}".format(jb),
                              "{:.2f}".format(dw), "{:.3f}".format(lb_sm)])
            except Exception as e:
                filas.append([pais, mid, "—", "—", "—", "—", "—", "—", "—", "—",
                              "error {}: {}".format(type(e).__name__, e)[:40]])

    dfv = pd.DataFrame(val, columns=["Pais", "Modelo", "ECT statsmodels",
                                     "ECT replicado MCO", "Diferencia",
                                     "Coincide", "n sm", "n MCO"])
    dfd = pd.DataFrame(filas, columns=["Pais", "Modelo", "n", "gl", "ECT",
                                       "Breusch-Godfrey p", "Ljung-Box p",
                                       "Breusch-Pagan p", "Jarque-Bera p",
                                       "Durbin-Watson", "LB sobre resid sm"])
    print("=" * 72)
    print("VALIDACION: replicacion del termino de correccion de error")
    print("=" * 72)
    print(dfv.to_string(index=False))
    print("\n" + "=" * 72)
    print("DIAGNOSTICOS SOBRE LA ECUACION REPLICADA (validos)")
    print("=" * 72)
    print(dfd.to_string(index=False))

    dfv.to_csv(os.path.join(OUTDIR, "A_validacion_ect.csv"), index=False)
    dfd.to_csv(os.path.join(OUTDIR, "A_diagnosticos_validos.csv"), index=False)
    print("\nDos archivos en {}".format(OUTDIR))


if __name__ == "__main__":
    main()

VALIDACION: replicacion del termino de correccion de error
          Pais Modelo ECT statsmodels ECT replicado MCO Diferencia Coincide n sm n MCO
         China     M1       -0.025767         -0.025767   0.00e+00       SI   32    32
         China     M3       -0.052108         -0.052108   0.00e+00       SI   33    32
         China     M4       -0.071774         -0.071774   0.00e+00       SI   31    30
         China     M5       -0.000385         -0.000385   0.00e+00       SI   27    27
         China    M5b       -0.018486         -0.018486   0.00e+00       SI   27    27
         China     M8       -0.042568         -0.042568   0.00e+00       SI   32    32
         China     M9       -0.019069         -0.019069   0.00e+00       SI   27    27
Estados Unidos     M1        0.013108          0.013108   0.00e+00       SI   32    32
Estados Unidos     M3       -0.124444         -0.124444   0.00e+00       SI   32    32
Estados Unidos     M4       -0.035520         -0.035520   0.00e+00     

========================================================================
VALIDACION: replicacion del termino de correccion de error
========================================================================
          Pais Modelo ECT statsmodels ECT replicado MCO Diferencia Coincide n sm n MCO
         China     M1       -0.025767         -0.025767   0.00e+00       SI   32    32
         China     M3       -0.052108         -0.052108   0.00e+00       SI   33    32
         China     M4       -0.071774         -0.071774   0.00e+00       SI   31    30
         China     M5       -0.000385         -0.000385   0.00e+00       SI   27    27
         China    M5b       -0.018486         -0.018486   0.00e+00       SI   27    27
         China     M8       -0.042568         -0.042568   0.00e+00       SI   32    32
         China     M9       -0.019069         -0.019069   0.00e+00       SI   27    27
Estados Unidos     M1        0.013108          0.013108   0.00e+00       SI   32    32
Estados Unidos     M3       -0.124444         -0.124444   0.00e+00       SI   32    32
Estados Unidos     M4       -0.035520         -0.035520   0.00e+00       SI   30    30
Estados Unidos     M5        0.003674          0.003674   0.00e+00       SI   32    32
Estados Unidos     M6        0.009438          0.009438   0.00e+00       SI   31    31
Estados Unidos     M7       -0.059267         -0.059267   0.00e+00       SI   32    32
Estados Unidos     M8       -0.045666         -0.045666   0.00e+00       SI   23    23
Estados Unidos     M9        0.033430          0.033430   0.00e+00       SI   32    32

========================================================================
DIAGNOSTICOS SOBRE LA ECUACION REPLICADA (validos)
========================================================================
          Pais Modelo  n gl        ECT Breusch-Godfrey p Ljung-Box p Breusch-Pagan p Jarque-Bera p Durbin-Watson LB sobre resid sm
         China     M1 32 20  -0.0258**             0.268       0.214           0.944         0.035          2.31             0.000
         China     M3 32 23  -0.0521**             0.639       0.701           0.132         0.756          2.08             0.000
         China     M4 30 20  -0.0718**             0.184       0.232           0.073         0.781          2.34             0.000
         China     M5 27 17    -0.0004             0.258       0.406           0.175         0.935          2.39             0.000
         China    M5b 27 19   -0.0185*             0.158       0.317           0.263         0.736          2.43             0.000
         China     M8 32 24 -0.0426***             0.411       0.254           0.192         0.879          2.35             0.000
         China     M9 27 19 -0.0191***             0.579       0.604           0.636         0.880          2.22             0.000
Estados Unidos     M1 32 22     0.0131             0.434       0.750           0.327         0.005          1.77             0.000
Estados Unidos     M3 32 20    -0.1244             0.643       0.779           0.261         0.881          1.79             0.000
Estados Unidos     M4 30 18    -0.0355             0.356       0.390           0.359         0.377          2.18             0.000
Estados Unidos     M5 32 23     0.0037             0.399       0.267           0.779         0.781          1.91             0.000
Estados Unidos     M6 31 21     0.0094             0.894       0.895           0.479         0.000          2.03             0.000
Estados Unidos     M7 32 22    -0.0593             0.606       0.655           0.067         0.144          2.02             0.000
Estados Unidos     M8 23 13    -0.0457             0.044       0.229           0.194         0.942          2.61             0.000
Estados Unidos     M9 32 24    0.0334*             0.871       0.938           0.541         0.959          1.89             0.000

Dos archivos en /content/outputs


In [45]:
# -*- coding: utf-8 -*-
"""
ESTIMACION FINAL. Anade ficticias como regresores fijos (sin rezagos):
  d2020 en todos los modelos (la muestra llega a 2023 y contiene la pandemia)
  q2002 y q2014 en China; q2008 en Estados Unidos, segun Zivot-Andrews.
Los diagnosticos se calculan sobre la ecuacion replicada en MCO, validada
previamente con diferencia 0.00e+00 en el ECT.
"""
import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import statsmodels.api as sm
from statsmodels.tsa.ardl import UECM, ardl_select_order
from statsmodels.stats.diagnostic import (acorr_breusch_godfrey,
                                          het_breuschpagan, acorr_ljungbox)
from statsmodels.stats.stattools import jarque_bera, durbin_watson
from scipy.stats import t as tdist

PANEL  = "/content/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
OUTDIR = "/content/outputs"
DEP_A, DEP_B = "ln_gdp_pc_pwt", "gdp_growth_pct"
PAISES = {"CN": "China", "US": "Estados Unidos"}

MODELOS = {
    "M1":  ["inv_gfcf_gdp", "trade_gdp", "inflation_cpi_pct"],
    "M3":  ["credit_priv_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M4":  ["bis_pvt_credit_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M5":  ["pub_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M5b": ["pub_debt_gdp", "trade_gdp"],
    "M6":  ["d_hh_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M7":  ["corp_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M8":  ["ratio_ind_agr_rec", "inv_gfcf_gdp", "trade_gdp"],
    "M9":  ["IIC_alt", "trade_gdp"],
}
SOLO_US, SOLO_CN = {"M6", "M7"}, {"M5b"}
OMITIR = {("CN", "M5")}          # singular: reemplazado por M5b


def est(p):
    if p is None or not np.isfinite(p): return ""
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""


def pval(t_, gl):
    return np.nan if (not np.isfinite(t_) or gl <= 0) else 2*(1-tdist.cdf(abs(t_), gl))


def ficticias(idx, cod):
    """Regresores fijos: pandemia y quiebres de Zivot-Andrews."""
    f = pd.DataFrame(index=idx)
    f["d2020"] = (idx == 2020).astype(float)
    if cod == "CN":
        f["q2002"] = (idx >= 2002).astype(float)
        f["q2014"] = (idx >= 2014).astype(float)
    else:
        f["q2008"] = (idx >= 2008).astype(float)
    return f.loc[:, f.std() > 0]


def construir(y, X, F, ord_y, ordenes):
    Z = pd.DataFrame(index=y.index)
    Z["y_L1"] = y.shift(1)
    for c in X.columns: Z[c + "_L1"] = X[c].shift(1)
    dy = y.diff()
    for m in range(1, ord_y): Z["d_y_L{}".format(m)] = dy.shift(m)
    for c in X.columns:
        dx = X[c].diff()
        for k in range(0, ordenes[c]): Z["d_{}_L{}".format(c, k)] = dx.shift(k)
    for c in F.columns: Z[c] = F[c]
    return pd.concat([dy.rename("d_y"), Z], axis=1).dropna()


def main():
    os.makedirs(OUTDIR, exist_ok=True)
    df = pd.read_csv(PANEL)
    df["country"] = df["country"].astype(str).str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)

    c7, c8, c10 = [], [], []
    for cod, pais in PAISES.items():
        for mid, regs in MODELOS.items():
            if (cod, mid) in OMITIR: continue
            if mid in SOLO_US and cod != "US": continue
            if mid in SOLO_CN and cod != "CN": continue
            sub = df[df["country"] == cod].sort_values("year").set_index("year")
            if any(c not in sub.columns for c in [DEP_A] + regs): continue
            d = sub[[DEP_A] + regs].dropna()
            if len(d) < 20: continue
            y, X = d[DEP_A].astype(float), d[regs].astype(float)
            F = ficticias(d.index, cod)
            try:
                sel = ardl_select_order(y, 2, X, 2, fixed=F, trend="c", ic="aic")
                ord_y = max(1, int(sel.model.ardl_order[0]))
                br = list(sel.model.ardl_order[1:])
                ordenes = {c: max(1, int(br[i]) if i < len(br) else 1)
                           for i, c in enumerate(X.columns)}
                u = UECM(y, ord_y, X, ordenes, fixed=F, trend="c").fit()

                bt = u.bounds_test(case=3)
                Fst = float(np.asarray(bt.stat).ravel()[0])
                cv = bt.crit_vals
                lo, up = float(cv.loc[95.0, "lower"]), float(cv.loc[95.0, "upper"])

                dat = construir(y, X, F, ord_y, ordenes)
                m = sm.OLS(dat["d_y"], sm.add_constant(dat.drop(columns="d_y"))).fit()
                lam = float(m.params["y_L1"]); p_l = float(m.pvalues["y_L1"])
                vm = np.log(0.5)/np.log(1+lam) if -1 < lam < 0 else np.nan
                ok = (-1 < lam < 0) and p_l < .10
                ver = ("Cointegracion" if (Fst > up and ok) else
                       "F alto, ECT invalido" if Fst > up else
                       "Sin cointegracion" if Fst < lo else "No concluyente")

                c7.append([pais, mid, "{}, {}".format(ord_y, list(ordenes.values())),
                           "{:.3f}".format(Fst), "{:.2f} / {:.2f}".format(lo, up),
                           "{:.4f}{}".format(lam, est(p_l)),
                           "{:.1f}".format(vm) if np.isfinite(vm) else "—",
                           str(int(m.nobs)), ver])

                ci, se = u.ci_params, u.ci_bse
                for nom in ci.index:
                    if nom == DEP_A: continue
                    b, s = -float(ci[nom]), float(se[nom])
                    t_ = b/s if s else np.nan
                    c8.append([pais, mid, nom, "{:.4f}{}".format(b, est(pval(t_, m.df_resid))),
                               "{:.4f}".format(s),
                               "{:.2f}".format(t_) if np.isfinite(t_) else "—"])

                r = np.asarray(m.resid, float)
                c10.append([pais, mid, str(int(m.nobs)), str(int(m.df_resid)),
                            "{:.3f}".format(float(acorr_breusch_godfrey(m, nlags=2)[3])),
                            "{:.3f}".format(float(acorr_ljungbox(r, lags=[2], return_df=True)["lb_pvalue"].iloc[0])),
                            "{:.3f}".format(float(het_breuschpagan(r, m.model.exog)[1])),
                            "{:.3f}".format(float(jarque_bera(r)[1])),
                            "{:.2f}".format(float(durbin_watson(r))),
                            "{:.3f}".format(float(m.rsquared_adj))])
            except Exception as e:
                c7.append([pais, mid, "—", "—", "—", "—", "—", "—",
                           "error {}: {}".format(type(e).__name__, e)[:45]])

    # ---------- familia B con las mismas ficticias
    c9 = []
    for cod, pais in PAISES.items():
        for mid, regs in MODELOS.items():
            if (cod, mid) in OMITIR: continue
            if mid in SOLO_US and cod != "US": continue
            if mid in SOLO_CN and cod != "CN": continue
            sub = df[df["country"] == cod].sort_values("year").set_index("year")
            if any(c not in sub.columns for c in [DEP_B] + regs): continue
            d = sub[[DEP_B] + regs].dropna()
            if len(d) < 20: continue
            y, X = d[DEP_B].astype(float), d[regs].astype(float)
            Z = X.copy(); Z["rho"] = y.shift(1)
            for c in X.columns: Z[c + "_L1"] = X[c].shift(1)
            Z = pd.concat([Z, ficticias(d.index, cod)], axis=1)
            dat = pd.concat([y, Z], axis=1).dropna()
            try:
                m = sm.OLS(dat.iloc[:, 0], sm.add_constant(dat.iloc[:, 1:])).fit(
                    cov_type="HAC", cov_kwds={"maxlags": 2, "use_correction": True})
                rho = float(m.params.get("rho", 0.0)); den = 1.0 - rho
                for c in X.columns:
                    b0, b1 = float(m.params.get(c, 0.0)), float(m.params.get(c+"_L1", 0.0))
                    c9.append([pais, mid, c,
                               "{:.4f}{}".format(b0, est(m.pvalues.get(c, np.nan))),
                               "{:.4f}{}".format(b1, est(m.pvalues.get(c+"_L1", np.nan))),
                               "{:.4f}".format(rho),
                               "{:.4f}".format((b0+b1)/den) if abs(den) > 1e-8 else "—",
                               "{:.3f}".format(m.rsquared_adj), str(int(m.nobs))])
            except Exception as e:
                c9.append([pais, mid, "error", str(e)[:40], "—", "—", "—", "—", "—"])

    t7 = pd.DataFrame(c7, columns=["Pais", "Modelo", "Ordenes", "F limites",
                                   "I(0)/I(1) 5%", "ECT", "Vida media", "n", "Veredicto"])
    t8 = pd.DataFrame(c8, columns=["Pais", "Modelo", "Variable", "Coeficiente", "EE", "t"])
    t9 = pd.DataFrame(c9, columns=["Pais", "Modelo", "Variable", "Contemporaneo",
                                   "Rezagado", "rho", "Multiplicador neto", "R2 aj.", "n"])
    t10 = pd.DataFrame(c10, columns=["Pais", "Modelo", "n", "gl", "Breusch-Godfrey p",
                                     "Ljung-Box p", "Breusch-Pagan p", "Jarque-Bera p",
                                     "Durbin-Watson", "R2 aj."])
    for nom, obj in [("cuadro7_cointegracion", t7), ("cuadro8_largo_plazo", t8),
                     ("cuadro9_corto_plazo", t9), ("cuadro10_diagnosticos", t10)]:
        obj.to_csv(os.path.join(OUTDIR, nom + ".csv"), index=False)
        print("=" * 74); print(nom.upper()); print("=" * 74)
        print(obj.to_string(index=False)); print()
    print("Cuatro cuadros guardados en {}".format(OUTDIR))


if __name__ == "__main__":
    main()

CUADRO7_COINTEGRACION
          Pais Modelo      Ordenes F limites I(0)/I(1) 5%        ECT Vida media  n            Veredicto
         China     M1 2, [2, 2, 2]    14.896  2.88 / 4.01  -0.0280**       24.4 32        Cointegracion
         China     M3 1, [1, 1, 2]     9.981  2.88 / 4.01  -0.0591**       11.4 32        Cointegracion
         China     M4 1, [2, 1, 2]    14.410  2.88 / 4.01    -0.0514       13.1 30 F alto, ECT invalido
         China    M5b    2, [2, 1]    11.988  3.23 / 4.32    -0.0194       35.4 27 F alto, ECT invalido
         China     M8 1, [1, 1, 1]    15.746  2.88 / 4.01 -0.0543***       12.4 32        Cointegracion
         China     M9    2, [2, 1]    12.020  3.23 / 4.32   -0.0192*       35.8 27        Cointegracion
Estados Unidos     M1 2, [2, 1, 1]     2.625  2.88 / 4.01     0.0102          — 32    Sin cointegracion
Estados Unidos     M3 2, [1, 2, 1]     3.175  2.88 / 4.01    -0.0528       12.8 32       No concluyente
Estados Unidos     M4 2, [2, 2, 2]     7.5

Especificación preferida y prueba de sensibilidad
Propongo fixed = d2020 únicamente. Y como ya tenemos las tres corridas, conviene documentar la sensibilidad de forma explícita: se convierte en una tabla de anexo que blinda el artículo. Este script estima las tres especificaciones de un tirón y las compara.

In [46]:
# -*- coding: utf-8 -*-
"""
SENSIBILIDAD A LOS REGRESORES DETERMINISTAS
S1  sin ficticias
S2  solo d2020 (impulso, corrige el valor extremo de la pandemia)  <- preferida
S3  d2020 + ficticias de escalon de Zivot-Andersen
Reporta F de limites, ECT y el coeficiente de largo plazo de la variable focal
(el primer regresor de cada modelo) en las tres especificaciones.
"""
import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import statsmodels.api as sm
from statsmodels.tsa.ardl import UECM, ardl_select_order
from statsmodels.stats.diagnostic import acorr_breusch_godfrey
from statsmodels.stats.stattools import jarque_bera
from scipy.stats import t as tdist

PANEL  = "/content/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
OUTDIR = "/content/outputs"
DEP    = "ln_gdp_pc_pwt"
PAISES = {"CN": "China", "US": "Estados Unidos"}
MODELOS = {
    "M1":  ["inv_gfcf_gdp", "trade_gdp", "inflation_cpi_pct"],
    "M3":  ["credit_priv_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M4":  ["bis_pvt_credit_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M5b": ["pub_debt_gdp", "trade_gdp"],
    "M5":  ["pub_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M6":  ["d_hh_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M7":  ["corp_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M8":  ["ratio_ind_agr_rec", "inv_gfcf_gdp", "trade_gdp"],
    "M9":  ["IIC_alt", "trade_gdp"],
}
SOLO_US, SOLO_CN, OMITIR = {"M6", "M7"}, {"M5b"}, {("CN", "M5")}


def est(p):
    if p is None or not np.isfinite(p): return ""
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""


def pval(t_, gl):
    return np.nan if (not np.isfinite(t_) or gl <= 0) else 2*(1-tdist.cdf(abs(t_), gl))


def fict(idx, cod, spec):
    if spec == "S1": return None
    f = pd.DataFrame(index=idx)
    f["d2020"] = (idx == 2020).astype(float)
    if spec == "S3":
        if cod == "CN":
            f["q2002"] = (idx >= 2002).astype(float)
            f["q2014"] = (idx >= 2014).astype(float)
        else:
            f["q2008"] = (idx >= 2008).astype(float)
    f = f.loc[:, f.std() > 0]
    return f if f.shape[1] else None


def replicar(y, X, F, ord_y, ordenes):
    Z = pd.DataFrame(index=y.index)
    Z["y_L1"] = y.shift(1)
    for c in X.columns: Z[c + "_L1"] = X[c].shift(1)
    dy = y.diff()
    for m in range(1, ord_y): Z["d_y_L{}".format(m)] = dy.shift(m)
    for c in X.columns:
        dx = X[c].diff()
        for k in range(0, ordenes[c]): Z["d_{}_L{}".format(c, k)] = dx.shift(k)
    if F is not None:
        for c in F.columns: Z[c] = F[c]
    dat = pd.concat([dy.rename("d_y"), Z], axis=1).dropna()
    return sm.OLS(dat["d_y"], sm.add_constant(dat.drop(columns="d_y"))).fit()


def main():
    os.makedirs(OUTDIR, exist_ok=True)
    df = pd.read_csv(PANEL)
    df["country"] = df["country"].astype(str).str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)

    filas = []
    for cod, pais in PAISES.items():
        for mid, regs in MODELOS.items():
            if (cod, mid) in OMITIR: continue
            if mid in SOLO_US and cod != "US": continue
            if mid in SOLO_CN and cod != "CN": continue
            sub = df[df["country"] == cod].sort_values("year").set_index("year")
            if any(c not in sub.columns for c in [DEP] + regs): continue
            d = sub[[DEP] + regs].dropna()
            if len(d) < 20: continue
            y, X = d[DEP].astype(float), d[regs].astype(float)
            focal = regs[0]
            for spec in ["S1", "S2", "S3"]:
                F = fict(d.index, cod, spec)
                try:
                    sel = ardl_select_order(y, 2, X, 2, fixed=F, trend="c", ic="aic")
                    ord_y = max(1, int(sel.model.ardl_order[0]))
                    br = list(sel.model.ardl_order[1:])
                    ordenes = {c: max(1, int(br[i]) if i < len(br) else 1)
                               for i, c in enumerate(X.columns)}
                    u = UECM(y, ord_y, X, ordenes, fixed=F, trend="c").fit()
                    bt = u.bounds_test(case=3)
                    Fst = float(np.asarray(bt.stat).ravel()[0])
                    up = float(bt.crit_vals.loc[95.0, "upper"])

                    m = replicar(y, X, F, ord_y, ordenes)
                    lam = float(m.params["y_L1"]); p_l = float(m.pvalues["y_L1"])
                    gl = int(m.df_resid)
                    b = -float(u.ci_params[focal]); s = float(u.ci_bse[focal])
                    t_ = b/s if s else np.nan
                    r = np.asarray(m.resid, float)
                    try: bg = float(acorr_breusch_godfrey(m, nlags=2)[3])
                    except Exception: bg = np.nan
                    ok = (-1 < lam < 0) and p_l < .10
                    filas.append([pais, mid, spec, "{:.3f}".format(Fst),
                                  "si" if Fst > up else "no",
                                  "{:.4f}{}".format(lam, est(p_l)),
                                  "{:.4f}{}".format(b, est(pval(t_, gl))),
                                  "{:.3f}".format(bg) if np.isfinite(bg) else "—",
                                  "{:.3f}".format(float(jarque_bera(r)[1])),
                                  str(gl), "VALIDO" if (Fst > up and ok) else ""])
                except Exception as e:
                    filas.append([pais, mid, spec, "—", "—", "—", "—", "—", "—",
                                  "—", "error {}".format(type(e).__name__)])

    t = pd.DataFrame(filas, columns=["Pais", "Modelo", "Espec", "F", "F>lim sup",
                                     "ECT", "LP variable focal", "BG p", "JB p",
                                     "gl", "Estatus"])
    print(t.to_string(index=False))
    t.to_csv(os.path.join(OUTDIR, "anexo_sensibilidad_deterministas.csv"),
             index=False)
    print("\nGuardado: {}/anexo_sensibilidad_deterministas.csv".format(OUTDIR))

    print("\nModelos validos por especificacion:")
    print(t[t["Estatus"] == "VALIDO"].groupby(["Espec", "Pais"]).size()
          .to_string())


if __name__ == "__main__":
    main()

          Pais Modelo Espec      F F>lim sup        ECT LP variable focal  BG p  JB p gl Estatus
         China     M1    S1 14.896        si  -0.0258**           -0.0370 0.268 0.035 20  VALIDO
         China     M1    S2 14.896        si  -0.0212**           -0.0616 0.345 0.559 19  VALIDO
         China     M1    S3 14.896        si  -0.0280**           -0.0378 0.388 0.778 17  VALIDO
         China     M3    S1  9.981        si  -0.0521**          0.0103** 0.639 0.756 23  VALIDO
         China     M3    S2  9.981        si  -0.0540**          0.0110** 0.665 0.548 22  VALIDO
         China     M3    S3  9.981        si  -0.0591**            0.0072 0.826 0.510 20  VALIDO
         China     M4    S1 14.410        si  -0.0718**          0.0077** 0.184 0.781 20  VALIDO
         China     M4    S2 14.410        si  -0.0605**          0.0075** 0.495 0.632 19  VALIDO
         China     M4    S3 14.410        si    -0.0514            0.0024 0.281 0.687 17        
         China    M5b    S1 11

Script 1: estimación definitiva bajo S2
Celda 1, con %%writefile /content/estimacion_final_S2.py como primera línea:

In [47]:
# -*- coding: utf-8 -*-
"""
ESTIMACION DEFINITIVA. Especificacion S2.
Ficticias: solo d2020 (impulso). Se excluyen las de escalon porque, al entrar
en niveles, absorben la variacion de largo plazo de los regresores y alteran
los valores criticos tabulados por Pesaran, Shin y Smith (2001).
Diagnosticos calculados sobre la ecuacion de correccion de error replicada en
MCO, validada con diferencia 0.00e+00 en el termino de correccion de error.
Genera: cuadro7, cuadro8, cuadro9, cuadro10 en /content/outputs
"""
import os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import statsmodels.api as sm
from statsmodels.tsa.ardl import UECM, ardl_select_order
from statsmodels.stats.diagnostic import (acorr_breusch_godfrey,
                                          het_breuschpagan, acorr_ljungbox)
from statsmodels.stats.stattools import jarque_bera, durbin_watson
from scipy.stats import t as tdist

PANEL  = "/content/data_clean/china_us_panel_analisis_v5_1990_2023.csv"
OUTDIR = "/content/outputs"
DEP_A, DEP_B = "ln_gdp_pc_pwt", "gdp_growth_pct"
PAISES = {"CN": "China", "US": "Estados Unidos"}

MODELOS = {
    "M1":  ["inv_gfcf_gdp", "trade_gdp", "inflation_cpi_pct"],
    "M3":  ["credit_priv_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M4":  ["bis_pvt_credit_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M5":  ["pub_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M5b": ["pub_debt_gdp", "trade_gdp"],
    "M6":  ["d_hh_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M7":  ["corp_debt_gdp", "inv_gfcf_gdp", "trade_gdp"],
    "M8":  ["ratio_ind_agr_rec", "inv_gfcf_gdp", "trade_gdp"],
    "M9":  ["IIC_alt", "trade_gdp"],
}
SOLO_US = {"M6", "M7"}
SOLO_CN = {"M5b"}
OMITIR  = {("CN", "M5")}     # singular con n=27; se reemplaza por M5b


def est(p):
    if p is None or not np.isfinite(p): return ""
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""


def pval(t_, gl):
    if not np.isfinite(t_) or gl <= 0: return np.nan
    return 2 * (1 - tdist.cdf(abs(t_), gl))


def ficticias(idx):
    """S2: unicamente ficticia de impulso para el ano 2020."""
    f = pd.DataFrame(index=idx)
    f["d2020"] = (np.asarray(idx) == 2020).astype(float)
    f = f.loc[:, f.std() > 0]
    return f if f.shape[1] else None


def replicar(y, X, F, ord_y, ordenes):
    """Reconstruye la ecuacion del UECM y la estima por MCO."""
    Z = pd.DataFrame(index=y.index)
    Z["y_L1"] = y.shift(1)
    for c in X.columns:
        Z[c + "_L1"] = X[c].shift(1)
    dy = y.diff()
    for m in range(1, ord_y):
        Z["d_y_L{}".format(m)] = dy.shift(m)
    for c in X.columns:
        dx = X[c].diff()
        for k in range(0, ordenes[c]):
            Z["d_{}_L{}".format(c, k)] = dx.shift(k)
    if F is not None:
        for c in F.columns:
            Z[c] = F[c]
    dat = pd.concat([dy.rename("d_y"), Z], axis=1).dropna()
    return sm.OLS(dat["d_y"], sm.add_constant(dat.drop(columns="d_y"))).fit()


def combinaciones():
    for cod, pais in PAISES.items():
        for mid, regs in MODELOS.items():
            if (cod, mid) in OMITIR: continue
            if mid in SOLO_US and cod != "US": continue
            if mid in SOLO_CN and cod != "CN": continue
            yield cod, pais, mid, regs


def cargar(df, cod, dep, regs):
    sub = df[df["country"] == cod].sort_values("year").set_index("year")
    if any(c not in sub.columns for c in [dep] + regs):
        return None, None
    d = sub[[dep] + regs].dropna()
    if len(d) < 20:
        return None, None
    return d[dep].astype(float), d[regs].astype(float)


def familia_A(df):
    c7, c8, c10 = [], [], []
    for cod, pais, mid, regs in combinaciones():
        y, X = cargar(df, cod, DEP_A, regs)
        if y is None: continue
        F = ficticias(y.index)
        try:
            sel = ardl_select_order(y, 2, X, 2, fixed=F, trend="c", ic="aic")
            ord_y = max(1, int(sel.model.ardl_order[0]))
            br = list(sel.model.ardl_order[1:])
            ordenes = {c: max(1, int(br[i]) if i < len(br) else 1)
                       for i, c in enumerate(X.columns)}
            u = UECM(y, ord_y, X, ordenes, fixed=F, trend="c").fit()

            bt = u.bounds_test(case=3)
            Fst = float(np.asarray(bt.stat).ravel()[0])
            cv = bt.crit_vals
            lo = float(cv.loc[95.0, "lower"]); up = float(cv.loc[95.0, "upper"])

            m = replicar(y, X, F, ord_y, ordenes)
            lam = float(m.params["y_L1"]); p_l = float(m.pvalues["y_L1"])
            gl = int(m.df_resid)
            vm = np.log(0.5) / np.log(1 + lam) if -1 < lam < 0 else np.nan
            ok = (-1 < lam < 0) and p_l < .10
            if Fst > up and ok:      ver = "Cointegracion"
            elif Fst > up:           ver = "F alto, ECT invalido"
            elif Fst < lo:           ver = "Sin cointegracion"
            else:                    ver = "No concluyente"

            c7.append([pais, mid,
                       "{}, {}".format(ord_y, list(ordenes.values())),
                       "{:.3f}".format(Fst),
                       "{:.2f} / {:.2f}".format(lo, up),
                       "{:.4f}{}".format(lam, est(p_l)),
                       "{:.1f}".format(vm) if np.isfinite(vm) else "—",
                       str(int(m.nobs)), ver])

            ci, se = u.ci_params, u.ci_bse
            for nom in ci.index:
                if nom == DEP_A: continue
                b = -float(ci[nom]); s = float(se[nom])
                t_ = b / s if s else np.nan
                c8.append([pais, mid, nom,
                           "{:.4f}{}".format(b, est(pval(t_, gl))),
                           "{:.4f}".format(s),
                           "{:.2f}".format(t_) if np.isfinite(t_) else "—"])

            r = np.asarray(m.resid, float)
            try: bg = float(acorr_breusch_godfrey(m, nlags=2)[3])
            except Exception: bg = np.nan
            try: lb = float(acorr_ljungbox(r, lags=[2],
                            return_df=True)["lb_pvalue"].iloc[0])
            except Exception: lb = np.nan
            try: bp = float(het_breuschpagan(r, m.model.exog)[1])
            except Exception: bp = np.nan
            c10.append([pais, mid, str(int(m.nobs)), str(gl),
                        "{:.3f}".format(bg) if np.isfinite(bg) else "—",
                        "{:.3f}".format(lb) if np.isfinite(lb) else "—",
                        "{:.3f}".format(bp) if np.isfinite(bp) else "—",
                        "{:.3f}".format(float(jarque_bera(r)[1])),
                        "{:.2f}".format(float(durbin_watson(r))),
                        "{:.3f}".format(float(m.rsquared_adj))])
        except Exception as e:
            c7.append([pais, mid, "—", "—", "—", "—", "—", "—",
                       "error {}: {}".format(type(e).__name__, e)[:45]])
    return c7, c8, c10


def familia_B(df):
    c9 = []
    for cod, pais, mid, regs in combinaciones():
        y, X = cargar(df, cod, DEP_B, regs)
        if y is None: continue
        Z = X.copy()
        Z["rho"] = y.shift(1)
        for c in X.columns:
            Z[c + "_L1"] = X[c].shift(1)
        F = ficticias(y.index)
        if F is not None:
            Z = pd.concat([Z, F], axis=1)
        dat = pd.concat([y, Z], axis=1).dropna()
        try:
            m = sm.OLS(dat.iloc[:, 0], sm.add_constant(dat.iloc[:, 1:])).fit(
                cov_type="HAC",
                cov_kwds={"maxlags": 2, "use_correction": True})
            rho = float(m.params.get("rho", 0.0))
            den = 1.0 - rho
            for c in X.columns:
                b0 = float(m.params.get(c, 0.0))
                b1 = float(m.params.get(c + "_L1", 0.0))
                net = (b0 + b1) / den if abs(den) > 1e-8 else np.nan
                c9.append([pais, mid, c,
                           "{:.4f}{}".format(b0, est(m.pvalues.get(c, np.nan))),
                           "{:.4f}{}".format(b1, est(m.pvalues.get(c + "_L1", np.nan))),
                           "{:.4f}".format(rho),
                           "{:.4f}".format(net) if np.isfinite(net) else "—",
                           "{:.3f}".format(m.rsquared_adj), str(int(m.nobs))])
        except Exception as e:
            c9.append([pais, mid, "error", str(e)[:40], "—", "—", "—", "—", "—"])
    return c9


def main():
    os.makedirs(OUTDIR, exist_ok=True)
    df = pd.read_csv(PANEL)
    df["country"] = df["country"].astype(str).str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    print("Panel: {} observaciones x {} variables".format(*df.shape))
    print("Especificacion S2: ficticia de impulso d2020\n")

    c7, c8, c10 = familia_A(df)
    c9 = familia_B(df)

    t7 = pd.DataFrame(c7, columns=["Pais", "Modelo", "Ordenes", "F limites",
                                   "I(0)/I(1) 5%", "ECT", "Vida media", "n",
                                   "Veredicto"])
    t8 = pd.DataFrame(c8, columns=["Pais", "Modelo", "Variable",
                                   "Coeficiente", "EE", "t"])
    t9 = pd.DataFrame(c9, columns=["Pais", "Modelo", "Variable",
                                   "Contemporaneo", "Rezagado", "rho",
                                   "Multiplicador neto", "R2 aj.", "n"])
    t10 = pd.DataFrame(c10, columns=["Pais", "Modelo", "n", "gl",
                                     "Breusch-Godfrey p", "Ljung-Box p",
                                     "Breusch-Pagan p", "Jarque-Bera p",
                                     "Durbin-Watson", "R2 aj."])

    for nom, obj in [("cuadro7_cointegracion", t7),
                     ("cuadro8_largo_plazo", t8),
                     ("cuadro9_corto_plazo", t9),
                     ("cuadro10_diagnosticos", t10)]:
        obj.to_csv(os.path.join(OUTDIR, nom + ".csv"), index=False)
        print("=" * 74); print(nom.upper()); print("=" * 74)
        print(obj.to_string(index=False)); print()

    val = t7[t7["Veredicto"] == "Cointegracion"].groupby("Pais").size()
    print("Modelos con cointegracion valida por pais:")
    print(val.to_string() if len(val) else "ninguno")
    print("\nCuatro cuadros guardados en {}".format(OUTDIR))


if __name__ == "__main__":
    main()

Panel: 68 observaciones x 146 variables
Especificacion S2: ficticia de impulso d2020

CUADRO7_COINTEGRACION
          Pais Modelo      Ordenes F limites I(0)/I(1) 5%        ECT Vida media  n            Veredicto
         China     M1 2, [2, 2, 2]    14.896  2.88 / 4.01  -0.0212**       32.4 32        Cointegracion
         China     M3 1, [1, 1, 2]     9.981  2.88 / 4.01  -0.0540**       12.5 32        Cointegracion
         China     M4 1, [2, 1, 2]    14.410  2.88 / 4.01  -0.0605**       11.1 30        Cointegracion
         China    M5b    2, [2, 1]    11.988  3.23 / 4.32  -0.0200**       34.3 27        Cointegracion
         China     M8 1, [1, 1, 1]    15.746  2.88 / 4.01 -0.0387***       17.5 32        Cointegracion
         China     M9    2, [2, 1]    12.020  3.23 / 4.32 -0.0180***       38.2 27        Cointegracion
Estados Unidos     M1 2, [2, 2, 1]     2.538  2.88 / 4.01     0.0173          — 32    Sin cointegracion
Estados Unidos     M3 2, [1, 2, 1]     3.175  2.88 / 4.01   

In [48]:
!python /content/estimacion_final_S2.py

python3: can't open file '/content/estimacion_final_S2.py': [Errno 2] No such file or directory


Script 2: maquetación en Word con formato APA
Celda 3, con %%writefile /content/cuadros_word_apa.py como primera línea. Genera los cuadros 4, 5, 7, 8, 9 y 10 en un solo documento.

In [49]:
# -*- coding: utf-8 -*-
"""
MAQUETACION EN WORD, FORMATO APA 7
Lee los CSV de /content/outputs y produce un documento con los cuadros
4, 5, 7, 8, 9 y 10, en horizontal, con notas y referencias.
"""
import os
import pandas as pd
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

OUTDIR = "/content/outputs"
DATADIR = "/content/data_clean"
SALIDA = os.path.join(OUTDIR, "Cuadros_resultados_APA.docx")
ANCHO_UTIL = 23.94          # cm disponibles en horizontal con margenes de 2 cm

ETIQUETAS = {
    "const": "Constante",
    "inv_gfcf_gdp": "Formacion bruta de capital fijo (% del PIB)",
    "trade_gdp": "Apertura comercial (% del PIB)",
    "inflation_cpi_pct": "Inflacion, precios al consumidor (%)",
    "credit_priv_gdp": "Credito al sector privado (% del PIB)",
    "bis_pvt_credit_gdp": "Credito privado, BIS (% del PIB)",
    "pub_debt_gdp": "Deuda publica (% del PIB)",
    "d_hh_debt_gdp": "Deuda de los hogares, primera diferencia",
    "corp_debt_gdp": "Deuda corporativa (% del PIB)",
    "ratio_ind_agr_rec": "Razon de productividad industria/agricultura",
    "IIC_alt": "Indice de intensidad de la acumulacion",
    "ln_gdp_pc_pwt": "PIB per capita real (logaritmo)",
    "gdp_growth_pct": "Tasa de crecimiento del PIB (%)",
}


# ---------------------------------------------------------------- utilidades
def _el(nombre):
    return OxmlElement(nombre)


def regla(fila, lado, sz=12):
    for celda in fila.cells:
        tcPr = celda._tc.get_or_add_tcPr()
        bordes = tcPr.find(qn("w:tcBorders"))
        if bordes is None:
            bordes = _el("w:tcBorders"); tcPr.append(bordes)
        b = bordes.find(qn("w:" + lado))
        if b is None:
            b = _el("w:" + lado); bordes.append(b)
        b.set(qn("w:val"), "single"); b.set(qn("w:sz"), str(sz))
        b.set(qn("w:space"), "0"); b.set(qn("w:color"), "000000")


def sombrear(fila, color="F0EFEA"):
    for celda in fila.cells:
        tcPr = celda._tc.get_or_add_tcPr()
        shd = _el("w:shd")
        shd.set(qn("w:val"), "clear"); shd.set(qn("w:color"), "auto")
        shd.set(qn("w:fill"), color)
        tcPr.append(shd)


def repetir_encabezado(fila):
    trPr = fila._tr.get_or_add_trPr()
    th = _el("w:tblHeader"); th.set(qn("w:val"), "true")
    trPr.append(th)


def layout_fijo(tabla):
    tblPr = tabla._tbl.tblPr
    lay = _el("w:tblLayout"); lay.set(qn("w:type"), "fixed")
    tblPr.append(lay)


def anchos_proporcionales(df):
    largos = []
    for col in df.columns:
        maximo = max([len(str(col))] + [len(str(v)) for v in df[col]])
        largos.append(max(maximo, 6))
    total = float(sum(largos))
    return [ANCHO_UTIL * l / total for l in largos]


def fijar_anchos(tabla, anchos):
    for j, ancho in enumerate(anchos):
        for fila in tabla.rows:
            fila.cells[j].width = Cm(ancho)


def texto(celda, valor, negrita=False, tam=8, centrado=False):
    celda.text = ""
    p = celda.paragraphs[0]
    p.paragraph_format.space_before = Pt(1)
    p.paragraph_format.space_after = Pt(1)
    if centrado:
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(str(valor))
    r.font.name = "Arial"; r.font.size = Pt(tam); r.bold = negrita


def agregar_cuadro(doc, numero, titulo, df, nota):
    p = doc.add_paragraph()
    p.paragraph_format.keep_with_next = True
    p.paragraph_format.space_after = Pt(0)
    r = p.add_run("Cuadro {}".format(numero))
    r.font.name = "Arial"; r.font.size = Pt(11); r.bold = True

    p = doc.add_paragraph()
    p.paragraph_format.keep_with_next = True
    p.paragraph_format.space_after = Pt(6)
    r = p.add_run(titulo)
    r.font.name = "Arial"; r.font.size = Pt(11); r.italic = True

    tabla = doc.add_table(rows=1, cols=len(df.columns))
    tabla.alignment = WD_TABLE_ALIGNMENT.CENTER
    layout_fijo(tabla)

    enc = tabla.rows[0]
    for j, col in enumerate(df.columns):
        texto(enc.cells[j], col, negrita=True, tam=8, centrado=True)
    sombrear(enc); regla(enc, "top", 12); regla(enc, "bottom", 8)
    repetir_encabezado(enc)

    for _, ren in df.iterrows():
        fila = tabla.add_row()
        for j, val in enumerate(ren):
            texto(fila.cells[j], "" if pd.isna(val) else val,
                  centrado=(j > 1))
    regla(tabla.rows[-1], "bottom", 12)
    fijar_anchos(tabla, anchos_proporcionales(df))

    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(4)
    r = p.add_run("Nota. ")
    r.font.name = "Arial"; r.font.size = Pt(9); r.italic = True
    r = p.add_run(nota)
    r.font.name = "Arial"; r.font.size = Pt(9)
    doc.add_paragraph()


def traducir(df, columna="Variable"):
    if columna in df.columns:
        df = df.copy()
        df[columna] = df[columna].map(lambda v: ETIQUETAS.get(v, v))
    return df


def leer(nombre):
    ruta = os.path.join(OUTDIR, nombre)
    if not os.path.exists(ruta):
        print("AVISO: no se encontro {}".format(ruta))
        return None
    return pd.read_csv(ruta).astype(str)


# ------------------------------------------------------------------ cuadro 4
def cuadro4():
    filas = [
        ["M1", "PIB per capita real (log)", "FBCF, apertura, inflacion",
         "Determinantes reales basicos", "34", "34"],
        ["M3", "PIB per capita real (log)", "Credito privado, FBCF, apertura",
         "Profundizacion financiera (fuente: Banco Mundial)", "34", "34"],
        ["M4", "PIB per capita real (log)", "Credito privado BIS, FBCF, apertura",
         "Robustez de la medida financiera", "32", "32"],
        ["M5b", "PIB per capita real (log)", "Deuda publica, apertura",
         "Endeudamiento publico (China; dos regresores por n = 27)", "29", "—"],
        ["M5", "PIB per capita real (log)", "Deuda publica, FBCF, apertura",
         "Endeudamiento publico (Estados Unidos)", "—", "34"],
        ["M6", "PIB per capita real (log)", "Deuda de hogares (dif.), FBCF, apertura",
         "Financiarizacion de los hogares", "—", "34"],
        ["M7", "PIB per capita real (log)", "Deuda corporativa, FBCF, apertura",
         "Endeudamiento empresarial", "—", "34"],
        ["M8", "PIB per capita real (log)", "Razon industria/agricultura, FBCF, apertura",
         "Cambio estructural sectorial", "33", "25"],
        ["M9", "PIB per capita real (log)", "Indice de intensidad de la acumulacion, apertura",
         "Sintesis inversora y financiera", "29", "34"],
    ]
    return pd.DataFrame(filas, columns=["Modelo", "Variable dependiente",
                                        "Regresores", "Hipotesis asociada",
                                        "n China", "n Estados Unidos"])


# ------------------------------------------------------------------ cuadro 5
def cuadro5():
    ruta = os.path.join(DATADIR, "bitacora_reparacion.csv")
    if not os.path.exists(ruta):
        print("AVISO: no se encontro la bitacora de reparacion")
        return None
    return pd.read_csv(ruta).astype(str)


# --------------------------------------------------------------------- main
def main():
    doc = Document()
    sec = doc.sections[0]
    sec.orientation = WD_ORIENT.LANDSCAPE
    sec.page_width, sec.page_height = Cm(27.94), Cm(21.59)
    for lado in ("top", "bottom", "left", "right"):
        setattr(sec, lado + "_margin", Cm(2))

    est = doc.styles["Normal"]
    est.font.name = "Arial"; est.font.size = Pt(11)

    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run("Resultados de la estimacion. China y Estados Unidos, "
                  "1990-2023")
    r.font.name = "Arial"; r.font.size = Pt(14); r.bold = True
    doc.add_paragraph()

    fuente = ("Elaboracion propia con datos de Penn World Table 10.01 "
              "(Feenstra, Inklaar y Timmer, 2015, https://doi.org/10.1257/"
              "aer.20130954), World Development Indicators del Banco Mundial "
              "(https://databank.worldbank.org/source/world-development-"
              "indicators), estadisticas de credito total del Bank for "
              "International Settlements (https://www.bis.org/statistics/"
              "totcredit.htm) y Global Debt Database del Fondo Monetario "
              "Internacional (Mbaye, Moreno-Badia y Chae, 2018, "
              "https://www.imf.org/-/media/files/publications/wp/2018/"
              "wp18111.pdf).")

    sig = " Niveles de significancia: *** p < .01, ** p < .05, * p < .10."

    agregar_cuadro(doc, 4,
        "Especificaciones estimadas, variables y correspondencia con las "
        "hipotesis de investigacion",
        cuadro4(),
        "Los tamanos de muestra corresponden a las observaciones disponibles "
        "antes de la generacion de rezagos y primeras diferencias. El guion "
        "indica que la especificacion no se estima para ese pais por "
        "indisponibilidad de la serie o por insuficiencia de grados de "
        "libertad. " + fuente)

    c5 = cuadro5()
    if c5 is not None:
        agregar_cuadro(doc, 5,
            "Bitacora de reconstruccion de series con cobertura incompleta",
            c5,
            "Cada registro documenta la serie afectada, el diagnostico de la "
            "falta de informacion y el procedimiento de reconstruccion "
            "aplicado de manera identica en ambos paises, con el fin de "
            "preservar la comparabilidad. " + fuente)

    c7 = leer("cuadro7_cointegracion.csv")
    if c7 is not None:
        agregar_cuadro(doc, 7,
            "Prueba de limites de cointegracion y termino de correccion de "
            "error, por pais y especificacion",
            c7,
            "Estimacion mediante un modelo autorregresivo de rezagos "
            "distribuidos en su forma de correccion de error no restringida. "
            "Los valores criticos corresponden al caso de constante no "
            "restringida y tendencia ausente en Pesaran, Shin y Smith (2001, "
            "https://doi.org/10.1002/jae.616). Se declara cointegracion "
            "unicamente cuando el estadistico F supera el limite superior al "
            "5 % y, simultaneamente, el termino de correccion de error resulta "
            "negativo y estadisticamente significativo. La vida media se "
            "expresa en anos y se calcula como ln(0.5)/ln(1 + lambda). Los "
            "ordenes de rezago se seleccionaron con el criterio de "
            "informacion de Akaike, con un orden minimo de uno para todos los "
            "regresores. Todas las especificaciones incluyen una variable "
            "ficticia de impulso para 2020." + sig + " " + fuente)

    c8 = leer("cuadro8_largo_plazo.csv")
    if c8 is not None:
        agregar_cuadro(doc, 8,
            "Coeficientes de largo plazo de la relacion cointegrante",
            traducir(c8),
            "La variable dependiente es el logaritmo del PIB per capita real. "
            "Los coeficientes se presentan con el signo correspondiente al "
            "efecto de largo plazo sobre la variable dependiente; es decir, "
            "invertido respecto de la normalizacion en la que el programa "
            "expresa el vector cointegrante. Los errores estandar se obtienen "
            "por el metodo delta. Las pruebas t emplean los grados de libertad "
            "de la ecuacion de correccion de error." + sig + " " + fuente)

    c9 = leer("cuadro9_corto_plazo.csv")
    if c9 is not None:
        agregar_cuadro(doc, 9,
            "Dinamica de corto plazo y multiplicadores netos de la tasa de "
            "crecimiento del producto",
            traducir(c9),
            "La variable dependiente es la tasa de crecimiento anual del PIB. "
            "Los errores estandar son consistentes ante heterocedasticidad y "
            "autocorrelacion segun el procedimiento de Newey y West, con dos "
            "rezagos. El coeficiente rho corresponde al primer rezago de la "
            "variable dependiente. El multiplicador neto se calcula como la "
            "suma de los coeficientes contemporaneo y rezagado dividida entre "
            "uno menos rho, e indica el efecto acumulado de un cambio "
            "permanente de una unidad en el regresor." + sig + " " + fuente)

    c10 = leer("cuadro10_diagnosticos.csv")
    if c10 is not None:
        agregar_cuadro(doc, 10,
            "Pruebas de diagnostico de los residuos de la ecuacion de "
            "correccion de error",
            c10,
            "Se reportan valores de probabilidad. Breusch-Godfrey y Ljung-Box "
            "contrastan la ausencia de autocorrelacion de orden dos; "
            "Breusch-Pagan, la homocedasticidad; Jarque-Bera, la normalidad. "
            "Valores superiores a .05 indican que no se rechaza la hipotesis "
            "nula y, por tanto, que el supuesto correspondiente se sostiene. "
            "Los diagnosticos se calculan sobre la ecuacion de correccion de "
            "error reconstruida y estimada por minimos cuadrados ordinarios, "
            "cuya equivalencia con la estimacion original se verifico con una "
            "diferencia nula en el termino de correccion de error. " + fuente)

    p = doc.add_paragraph()
    r = p.add_run("Referencias")
    r.font.name = "Arial"; r.font.size = Pt(12); r.bold = True

    refs = [
        "Bank for International Settlements. (s. f.). Credit to the private "
        "non-financial sector: Documentation. https://www.bis.org/statistics/"
        "totcredit/credpriv_doc.pdf",
        "Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2015). The next "
        "generation of the Penn World Table. American Economic Review, "
        "105(10), 3150-3182. https://doi.org/10.1257/aer.20130954",
        "Mbaye, S., Moreno-Badia, M., y Chae, K. (2018). Global debt database: "
        "Methodology and sources (Working Paper N.o 18/111). International "
        "Monetary Fund. https://www.imf.org/-/media/files/publications/wp/"
        "2018/wp18111.pdf",
        "Pesaran, M. H., Shin, Y., y Smith, R. J. (2001). Bounds testing "
        "approaches to the analysis of level relationships. Journal of Applied "
        "Econometrics, 16(3), 289-326. https://doi.org/10.1002/jae.616",
        "Stock, J. H., y Watson, M. W. (1993). A simple estimator of "
        "cointegrating vectors in higher order integrated systems. "
        "Econometrica, 61(4), 783-820. https://doi.org/10.2307/2951763",
    ]
    for ref in refs:
        p = doc.add_paragraph()
        p.paragraph_format.first_line_indent = Cm(-1.27)
        p.paragraph_format.left_indent = Cm(1.27)
        p.paragraph_format.space_after = Pt(6)
        r = p.add_run(ref)
        r.font.name = "Arial"; r.font.size = Pt(10)

    doc.save(SALIDA)
    print("Documento guardado en {}".format(SALIDA))


if __name__ == "__main__":
    main()

Documento guardado en /content/outputs/Cuadros_resultados_APA.docx


Celda 4:

In [50]:
!pip -q install python-docx
!python /content/cuadros_word_apa.py
from google.colab import files
files.download("/content/outputs/Cuadros_resultados_APA.docx")

python3: can't open file '/content/cuadros_word_apa.py': [Errno 2] No such file or directory


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [51]:
from docx import Document
d = Document("/content/outputs/Cuadros_resultados_APA.docx")

for i, p in enumerate(d.paragraphs):
    if p.text.strip():
        print("P{:03d} | {}".format(i, p.text.strip()[:300]))

print("\n" + "=" * 70)
print("TABLAS: {}".format(len(d.tables)))
print("=" * 70)
for k, t in enumerate(d.tables, 1):
    print("\n--- TABLA {} ({} filas x {} columnas) ---".format(
        k, len(t.rows), len(t.columns)))
    for fila in t.rows:
        print(" | ".join(c.text.strip() for c in fila.cells))

P000 | Resultados de la estimacion. China y Estados Unidos, 1990-2023
P002 | Cuadro 4
P003 | Especificaciones estimadas, variables y correspondencia con las hipotesis de investigacion
P004 | Nota. Los tamanos de muestra corresponden a las observaciones disponibles antes de la generacion de rezagos y primeras diferencias. El guion indica que la especificacion no se estima para ese pais por indisponibilidad de la serie o por insuficiencia de grados de libertad. Elaboracion propia con dato
P006 | Cuadro 5
P007 | Bitacora de reconstruccion de series con cobertura incompleta
P008 | Nota. Cada registro documenta la serie afectada, el diagnostico de la falta de informacion y el procedimiento de reconstruccion aplicado de manera identica en ambos paises, con el fin de preservar la comparabilidad. Elaboracion propia con datos de Penn World Table 10.01 (Feenstra, Inklaar y Timmer, 
P010 | Cuadro 7
P011 | Prueba de limites de cointegracion y termino de correccion de error, por pais y especificaci

Script de maquetación, versión 2
Completo, no hay que sustituir nada. Filtra el cuadro 8, añade el cuadro de anexo, alinea a la izquierda las columnas de texto largo y limita el ancho de la columna de justificaciones del cuadro 5, que era la que se podía desbordar.

Celda con %%writefile /content/cuadros_word_apa_v2.py:

In [52]:
# -*- coding: utf-8 -*-
"""
MAQUETACION EN WORD, FORMATO APA 7. Version 2.
Cambios: el cuadro 8 reporta solo relaciones de largo plazo identificadas
(modelos con cointegracion validada en el cuadro 7); las estimaciones no
identificadas pasan a un cuadro de anexo. Alineacion a la izquierda en
columnas de texto y limite al ancho de las columnas muy largas.
"""
import os
import pandas as pd
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

OUTDIR = "/content/outputs"
DATADIR = "/content/data_clean"
SALIDA = os.path.join(OUTDIR, "Cuadros_resultados_APA_v2.docx")
ANCHO_UTIL = 23.94
TOPE_CARACTERES = 55       # limite para el reparto proporcional de anchos

ETIQUETAS = {
    "const": "Constante",
    "inv_gfcf_gdp": "Formacion bruta de capital fijo (% del PIB)",
    "trade_gdp": "Apertura comercial (% del PIB)",
    "inflation_cpi_pct": "Inflacion, precios al consumidor (%)",
    "credit_priv_gdp": "Credito al sector privado (% del PIB)",
    "bis_pvt_credit_gdp": "Credito privado, BIS (% del PIB)",
    "pub_debt_gdp": "Deuda publica (% del PIB)",
    "d_hh_debt_gdp": "Deuda de los hogares, primera diferencia",
    "corp_debt_gdp": "Deuda corporativa (% del PIB)",
    "ratio_ind_agr_rec": "Razon de productividad industria/agricultura",
    "IIC_alt": "Indice de intensidad de la acumulacion",
}
COLS_TEXTO = {"Variable", "Regresores", "Hipotesis asociada", "Justificacion",
              "Decision", "Incidencia", "Veredicto", "Variable dependiente",
              "Pais", "Modelo", "Estatus"}


def _el(n):
    return OxmlElement(n)


def regla(fila, lado, sz=12):
    for celda in fila.cells:
        tcPr = celda._tc.get_or_add_tcPr()
        bordes = tcPr.find(qn("w:tcBorders"))
        if bordes is None:
            bordes = _el("w:tcBorders"); tcPr.append(bordes)
        b = bordes.find(qn("w:" + lado))
        if b is None:
            b = _el("w:" + lado); bordes.append(b)
        b.set(qn("w:val"), "single"); b.set(qn("w:sz"), str(sz))
        b.set(qn("w:space"), "0"); b.set(qn("w:color"), "000000")


def sombrear(fila, color="F0EFEA"):
    for celda in fila.cells:
        tcPr = celda._tc.get_or_add_tcPr()
        shd = _el("w:shd")
        shd.set(qn("w:val"), "clear"); shd.set(qn("w:color"), "auto")
        shd.set(qn("w:fill"), color)
        tcPr.append(shd)


def repetir_encabezado(fila):
    trPr = fila._tr.get_or_add_trPr()
    th = _el("w:tblHeader"); th.set(qn("w:val"), "true")
    trPr.append(th)


def layout_fijo(tabla):
    lay = _el("w:tblLayout"); lay.set(qn("w:type"), "fixed")
    tabla._tbl.tblPr.append(lay)


def anchos_proporcionales(df):
    largos = []
    for col in df.columns:
        maximo = max([len(str(col))] + [len(str(v)) for v in df[col]])
        largos.append(min(max(maximo, 6), TOPE_CARACTERES))
    total = float(sum(largos))
    return [ANCHO_UTIL * l / total for l in largos]


def fijar_anchos(tabla, anchos):
    for j, ancho in enumerate(anchos):
        for fila in tabla.rows:
            fila.cells[j].width = Cm(ancho)


def texto(celda, valor, negrita=False, tam=8, alineacion="izq"):
    celda.text = ""
    p = celda.paragraphs[0]
    p.paragraph_format.space_before = Pt(1)
    p.paragraph_format.space_after = Pt(1)
    if alineacion == "centro":
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(str(valor))
    r.font.name = "Arial"; r.font.size = Pt(tam); r.bold = negrita


def agregar_cuadro(doc, etiqueta, titulo, df, nota):
    p = doc.add_paragraph()
    p.paragraph_format.keep_with_next = True
    p.paragraph_format.space_after = Pt(0)
    r = p.add_run(etiqueta)
    r.font.name = "Arial"; r.font.size = Pt(11); r.bold = True

    p = doc.add_paragraph()
    p.paragraph_format.keep_with_next = True
    p.paragraph_format.space_after = Pt(6)
    r = p.add_run(titulo)
    r.font.name = "Arial"; r.font.size = Pt(11); r.italic = True

    alineaciones = ["izq" if c in COLS_TEXTO else "centro" for c in df.columns]

    tabla = doc.add_table(rows=1, cols=len(df.columns))
    tabla.alignment = WD_TABLE_ALIGNMENT.CENTER
    layout_fijo(tabla)

    enc = tabla.rows[0]
    for j, col in enumerate(df.columns):
        texto(enc.cells[j], col, negrita=True, alineacion="centro")
    sombrear(enc); regla(enc, "top", 12); regla(enc, "bottom", 8)
    repetir_encabezado(enc)

    for _, ren in df.iterrows():
        fila = tabla.add_row()
        for j, val in enumerate(ren):
            texto(fila.cells[j], "" if pd.isna(val) else val,
                  alineacion=alineaciones[j])
    regla(tabla.rows[-1], "bottom", 12)
    fijar_anchos(tabla, anchos_proporcionales(df))

    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(4)
    r = p.add_run("Nota. ")
    r.font.name = "Arial"; r.font.size = Pt(9); r.italic = True
    r = p.add_run(nota)
    r.font.name = "Arial"; r.font.size = Pt(9)
    doc.add_paragraph()


def traducir(df):
    if "Variable" in df.columns:
        df = df.copy()
        df["Variable"] = df["Variable"].map(lambda v: ETIQUETAS.get(v, v))
    return df


def leer(nombre):
    ruta = os.path.join(OUTDIR, nombre)
    if not os.path.exists(ruta):
        print("AVISO: no se encontro {}".format(ruta)); return None
    return pd.read_csv(ruta).astype(str)


def cuadro4():
    filas = [
        ["M1", "PIB per capita real (log)", "FBCF, apertura, inflacion",
         "Determinantes reales basicos", "34", "34"],
        ["M3", "PIB per capita real (log)", "Credito privado, FBCF, apertura",
         "Profundizacion financiera (Banco Mundial)", "34", "34"],
        ["M4", "PIB per capita real (log)", "Credito privado BIS, FBCF, apertura",
         "Robustez de la medida financiera", "32", "32"],
        ["M5b", "PIB per capita real (log)", "Deuda publica, apertura",
         "Endeudamiento publico (China; dos regresores por n = 27)", "29", "—"],
        ["M5", "PIB per capita real (log)", "Deuda publica, FBCF, apertura",
         "Endeudamiento publico (Estados Unidos)", "—", "34"],
        ["M6", "PIB per capita real (log)", "Deuda de hogares (dif.), FBCF, apertura",
         "Financiarizacion de los hogares", "—", "34"],
        ["M7", "PIB per capita real (log)", "Deuda corporativa, FBCF, apertura",
         "Endeudamiento empresarial", "—", "34"],
        ["M8", "PIB per capita real (log)", "Razon industria/agricultura, FBCF, apertura",
         "Cambio estructural sectorial", "33", "25"],
        ["M9", "PIB per capita real (log)", "Indice de intensidad de la acumulacion, apertura",
         "Sintesis inversora y financiera", "29", "34"],
    ]
    return pd.DataFrame(filas, columns=["Modelo", "Variable dependiente",
                                        "Regresores", "Hipotesis asociada",
                                        "n China", "n Estados Unidos"])


def cuadro5():
    ruta = os.path.join(DATADIR, "bitacora_reparacion.csv")
    if not os.path.exists(ruta):
        print("AVISO: no se encontro la bitacora"); return None
    return pd.read_csv(ruta).astype(str)


def partir_cuadro8(c8, c7):
    """Separa las relaciones identificadas de las que no lo estan."""
    if c7 is None:
        return c8, None
    validos = set(zip(c7.loc[c7["Veredicto"] == "Cointegracion", "Pais"],
                      c7.loc[c7["Veredicto"] == "Cointegracion", "Modelo"]))
    clave = list(zip(c8["Pais"], c8["Modelo"]))
    marca = pd.Series([k in validos for k in clave], index=c8.index)
    ident = c8[marca].reset_index(drop=True)
    noid = c8[~marca].copy().reset_index(drop=True)
    if len(noid):
        noid["Estatus"] = "No identificado"
    return ident, (noid if len(noid) else None)


def main():
    doc = Document()
    sec = doc.sections[0]
    sec.orientation = WD_ORIENT.LANDSCAPE
    sec.page_width, sec.page_height = Cm(27.94), Cm(21.59)
    for lado in ("top", "bottom", "left", "right"):
        setattr(sec, lado + "_margin", Cm(2))
    est = doc.styles["Normal"]
    est.font.name = "Arial"; est.font.size = Pt(11)

    p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run("Resultados de la estimacion. China y Estados Unidos, "
                  "1990-2023")
    r.font.name = "Arial"; r.font.size = Pt(14); r.bold = True
    doc.add_paragraph()

    fuente = ("Elaboracion propia con datos de Penn World Table 10.01 "
              "(Feenstra, Inklaar y Timmer, 2015, https://doi.org/10.1257/"
              "aer.20130954), World Development Indicators del Banco Mundial "
              "(https://databank.worldbank.org/source/world-development-"
              "indicators), estadisticas de credito total del Bank for "
              "International Settlements (https://www.bis.org/statistics/"
              "totcredit.htm) y Global Debt Database del Fondo Monetario "
              "Internacional (Mbaye, Moreno-Badia y Chae, 2018, "
              "https://www.imf.org/-/media/files/publications/wp/2018/"
              "wp18111.pdf).")
    sig = " Niveles de significancia: *** p < .01, ** p < .05, * p < .10."

    agregar_cuadro(doc, "Cuadro 4",
        "Especificaciones estimadas, variables y correspondencia con las "
        "hipotesis de investigacion", cuadro4(),
        "Los tamanos de muestra corresponden a las observaciones disponibles "
        "antes de la generacion de rezagos y primeras diferencias. El guion "
        "indica que la especificacion no se estima para ese pais por "
        "indisponibilidad de la serie o por insuficiencia de grados de "
        "libertad. " + fuente)

    c5 = cuadro5()
    if c5 is not None:
        agregar_cuadro(doc, "Cuadro 5",
            "Bitacora de reconstruccion de series con cobertura incompleta",
            c5,
            "Cada registro documenta la serie afectada, el diagnostico de la "
            "falta de informacion y el procedimiento de reconstruccion "
            "aplicado de manera identica en ambos paises, con el fin de "
            "preservar la comparabilidad. " + fuente)

    c7 = leer("cuadro7_cointegracion.csv")
    if c7 is not None:
        agregar_cuadro(doc, "Cuadro 7",
            "Prueba de limites de cointegracion y termino de correccion de "
            "error, por pais y especificacion", c7,
            "Estimacion mediante un modelo autorregresivo de rezagos "
            "distribuidos en su forma de correccion de error no restringida. "
            "Los valores criticos corresponden al caso de constante no "
            "restringida y tendencia ausente en Pesaran, Shin y Smith (2001, "
            "https://doi.org/10.1002/jae.616). Se declara cointegracion "
            "unicamente cuando el estadistico F supera el limite superior al "
            "5 % y, de manera simultanea, el termino de correccion de error "
            "resulta negativo y estadisticamente significativo. La vida media "
            "se expresa en anos y se calcula como ln(0.5)/ln(1 + lambda); "
            "carece de interpretacion cuando el termino de correccion de error "
            "no es negativo. Los ordenes de rezago se seleccionaron con el "
            "criterio de informacion de Akaike, con un orden minimo de uno "
            "para todos los regresores, de modo que la variable focal de cada "
            "hipotesis permanece en la especificacion. Todas las ecuaciones "
            "incluyen una variable ficticia de impulso para 2020." + sig
            + " " + fuente)

    c8 = leer("cuadro8_largo_plazo.csv")
    if c8 is not None:
        ident, noid = partir_cuadro8(c8, c7)
        agregar_cuadro(doc, "Cuadro 8",
            "Coeficientes de largo plazo de las relaciones cointegrantes "
            "identificadas", traducir(ident),
            "La variable dependiente es el logaritmo del PIB per capita real. "
            "Solo se reportan las especificaciones que satisfacen de manera "
            "conjunta los dos criterios del cuadro 7. Ninguna especificacion "
            "estimada para Estados Unidos los satisface; en consecuencia, sus "
            "coeficientes de largo plazo no estan identificados y se presentan "
            "unicamente en el cuadro de anexo A1. Los coeficientes se expresan "
            "con el signo del efecto de largo plazo sobre la variable "
            "dependiente, esto es, invertidos respecto de la normalizacion en "
            "que el vector cointegrante se expresa computacionalmente. Los "
            "errores estandar se obtienen por el metodo delta y las pruebas t "
            "emplean los grados de libertad de la ecuacion de correccion de "
            "error." + sig + " " + fuente)

        if noid is not None:
            agregar_cuadro(doc, "Cuadro A1",
                "Estimaciones de largo plazo no identificadas. "
                "Estados Unidos, 1990-2023", traducir(noid),
                "Se presentan por transparencia y no admiten interpretacion "
                "sustantiva. El coeficiente de largo plazo equivale al "
                "cociente entre el parametro de nivel del regresor y el "
                "termino de correccion de error, con signo invertido; cuando "
                "este ultimo tiende a cero, como ocurre en todas estas "
                "especificaciones segun el cuadro 7, el cociente no esta "
                "definido y los errores estandar se vuelven arbitrariamente "
                "grandes. La ausencia de un mecanismo de correccion de error "
                "en el caso estadounidense constituye en si misma un resultado "
                "del analisis, no una limitacion de la estimacion. " + fuente)

    c9 = leer("cuadro9_corto_plazo.csv")
    if c9 is not None:
        agregar_cuadro(doc, "Cuadro 9",
            "Dinamica de corto plazo y multiplicadores netos de la tasa de "
            "crecimiento del producto", traducir(c9),
            "La variable dependiente es la tasa de crecimiento anual del PIB, "
            "estacionaria en niveles, por lo que la especificacion no requiere "
            "prueba de cointegracion. Los errores estandar son consistentes "
            "ante heterocedasticidad y autocorrelacion segun el procedimiento "
            "de Newey y West, con dos rezagos. El coeficiente rho corresponde "
            "al primer rezago de la variable dependiente. El multiplicador "
            "neto se calcula como la suma de los coeficientes contemporaneo y "
            "rezagado dividida entre uno menos rho, e indica el efecto "
            "acumulado de una variacion permanente de una unidad en el "
            "regresor." + sig + " " + fuente)

    c10 = leer("cuadro10_diagnosticos.csv")
    if c10 is not None:
        agregar_cuadro(doc, "Cuadro 10",
            "Pruebas de diagnostico de los residuos de la ecuacion de "
            "correccion de error", c10,
            "Se reportan valores de probabilidad. Breusch-Godfrey y Ljung-Box "
            "contrastan la ausencia de autocorrelacion de orden dos; "
            "Breusch-Pagan, la homocedasticidad; Jarque-Bera, la normalidad. "
            "Valores superiores a .05 indican que no se rechaza la hipotesis "
            "nula y que el supuesto correspondiente se sostiene. Los "
            "diagnosticos se calculan sobre la ecuacion de correccion de error "
            "reconstruida y estimada por minimos cuadrados ordinarios, cuya "
            "equivalencia con la estimacion original se verifico obteniendo "
            "una diferencia nula en el termino de correccion de error en las "
            "quince especificaciones. " + fuente)

    p = doc.add_paragraph()
    r = p.add_run("Referencias")
    r.font.name = "Arial"; r.font.size = Pt(12); r.bold = True

    refs = [
        "Bank for International Settlements. (s. f.). Credit to the private "
        "non-financial sector: Documentation. https://www.bis.org/statistics/"
        "totcredit/credpriv_doc.pdf",
        "Feenstra, R. C., Inklaar, R., y Timmer, M. P. (2015). The next "
        "generation of the Penn World Table. American Economic Review, "
        "105(10), 3150-3182. https://doi.org/10.1257/aer.20130954",
        "Mbaye, S., Moreno-Badia, M., y Chae, K. (2018). Global debt database: "
        "Methodology and sources (Working Paper N.o 18/111). International "
        "Monetary Fund. https://www.imf.org/-/media/files/publications/wp/"
        "2018/wp18111.pdf",
        "Newey, W. K., y West, K. D. (1987). A simple, positive semi-definite, "
        "heteroskedasticity and autocorrelation consistent covariance matrix. "
        "Econometrica, 55(3), 703-708. https://doi.org/10.2307/1913610",
        "Pesaran, M. H., Shin, Y., y Smith, R. J. (2001). Bounds testing "
        "approaches to the analysis of level relationships. Journal of Applied "
        "Econometrics, 16(3), 289-326. https://doi.org/10.1002/jae.616",
        "Stock, J. H., y Watson, M. W. (1993). A simple estimator of "
        "cointegrating vectors in higher order integrated systems. "
        "Econometrica, 61(4), 783-820. https://doi.org/10.2307/2951763",
    ]
    for ref in refs:
        p = doc.add_paragraph()
        p.paragraph_format.first_line_indent = Cm(-1.27)
        p.paragraph_format.left_indent = Cm(1.27)
        p.paragraph_format.space_after = Pt(6)
        r = p.add_run(ref)
        r.font.name = "Arial"; r.font.size = Pt(10)

    doc.save(SALIDA)
    print("Documento guardado en {}".format(SALIDA))


if __name__ == "__main__":
    main()

Documento guardado en /content/outputs/Cuadros_resultados_APA_v2.docx


In [53]:
!python /content/cuadros_word_apa_v2.py
from google.colab import files
files.download("/content/outputs/Cuadros_resultados_APA_v2.docx")

python3: can't open file '/content/cuadros_word_apa_v2.py': [Errno 2] No such file or directory


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [54]:
from docx import Document

RUTA = "/content/outputs/Cuadros_resultados_APA_v2.docx"
d = Document(RUTA)

print("PARRAFOS")
print("=" * 70)
for i, p in enumerate(d.paragraphs):
    if p.text.strip():
        print("P{:03d} | {}".format(i, p.text.strip()))

print("\nTABLAS: {}".format(len(d.tables)))
print("=" * 70)
for k, t in enumerate(d.tables, 1):
    print("\n--- TABLA {} ({} filas x {} columnas) ---".format(
        k, len(t.rows), len(t.columns)))
    for fila in t.rows:
        print(" | ".join(c.text.strip() for c in fila.cells))

PARRAFOS
P000 | Resultados de la estimacion. China y Estados Unidos, 1990-2023
P002 | Cuadro 4
P003 | Especificaciones estimadas, variables y correspondencia con las hipotesis de investigacion
P004 | Nota. Los tamanos de muestra corresponden a las observaciones disponibles antes de la generacion de rezagos y primeras diferencias. El guion indica que la especificacion no se estima para ese pais por indisponibilidad de la serie o por insuficiencia de grados de libertad. Elaboracion propia con datos de Penn World Table 10.01 (Feenstra, Inklaar y Timmer, 2015, https://doi.org/10.1257/aer.20130954), World Development Indicators del Banco Mundial (https://databank.worldbank.org/source/world-development-indicators), estadisticas de credito total del Bank for International Settlements (https://www.bis.org/statistics/totcredit.htm) y Global Debt Database del Fondo Monetario Internacional (Mbaye, Moreno-Badia y Chae, 2018, https://www.imf.org/-/media/files/publications/wp/2018/wp18111.pdf).
P006

In [57]:
import subprocess, glob, os

print("Scripts en /content:")
for f in sorted(glob.glob("/content/*.py")):
    print("  ", os.path.basename(f), os.path.getsize(f), "bytes")

destino = "/content/cuadros_word_apa_v2.py"
print("\nExiste el archivo esperado:", os.path.exists(destino))

if os.path.exists(destino):
    r = subprocess.run(["python", destino], capture_output=True, text=True)
    print("Codigo de salida:", r.returncode)
    print("\n--- SALIDA ---\n", r.stdout)
    print("\n--- ERROR ---\n", r.stderr)

Scripts en /content:
   construir_panel_analisis.py 1 bytes
   generar_cuadros_apa.py 33918 bytes

Existe el archivo esperado: False


In [58]:
import subprocess, os, glob

cand = "/content/generar_cuadros_apa.py"
txt = open(cand, encoding="utf-8").read()

print("Marcadores del generador v2:")
for m in ["Cuadro A1", "No identificado", "Newey", "cuadro7_cointegracion"]:
    print("  {:<24} {}".format(m, m in txt))
print("\nPrimeras lineas:")
print("\n".join(txt.splitlines()[:12]))

print("\nArchivos en /content/outputs:")
for f in sorted(glob.glob("/content/outputs/*")):
    print("  ", os.path.basename(f), os.path.getsize(f), "bytes")

if "Cuadro A1" in txt:
    r = subprocess.run(["python", cand], capture_output=True, text=True)
    print("\nCodigo de salida:", r.returncode)
    print("--- SALIDA ---\n", r.stdout)
    print("--- ERROR ---\n", r.stderr[-3000:])
else:
    print("\nEste archivo NO es el generador v2. Te lo reescribo completo.")

Marcadores del generador v2:
  Cuadro A1                False
  No identificado          False
  Newey                    False
  cuadro7_cointegracion    False

Primeras lineas:
# -*- coding: utf-8 -*-
"""
GENERADOR DE CUADROS PREVIOS AL ANALISIS ESTADISTICO  (formato APA 7)
Proyecto: modelos de crecimiento comparados China - Estados Unidos, 1990-2025

Genera cuatro cuadros a partir del panel maestro y los exporta a un documento
de Word con formato de tabla APA (numero de tabla en negrita, titulo en
cursiva, solo reglas horizontales, nota al pie):

    Tabla 1  Cobertura y disponibilidad de datos por variable y pais

Archivos en /content/outputs:
   A_bounds_ect.csv 1219 bytes
   A_bounds_ect_v3.csv 1831 bytes
   A_diagnosticos.csv 429 bytes
   A_diagnosticos_v3.csv 639 bytes
   A_diagnosticos_validos.csv 1066 bytes
   A_largo_plazo.csv 1552 bytes
   A_largo_plazo_v3.csv 2526 bytes
   A_validacion_ect.csv 847 bytes
   B_coeficientes_v3.csv 4893 bytes
   B_corto_plazo.csv 5089 bytes
  

In [59]:
# -*- coding: utf-8 -*-
"""Retoques finales sobre el documento de cuadros ya generado."""

import glob, os
from docx import Document

ORIGEN = "/content/outputs/Cuadros_resultados_APA_v2.docx"
DESTINO = "/content/outputs/Cuadros_resultados_APA_v3.docx"

doc = Document(ORIGEN)


def escribir(celda, texto):
    """Reemplaza el texto conservando el formato de la primera corrida."""
    p = celda.paragraphs[0]
    if p.runs:
        p.runs[0].text = texto
        for r in p.runs[1:]:
            r.text = ""
    else:
        p.add_run(texto)


# ---------------------------------------------------------------- 1. vida media
GUION = "\u2014"
arreglos = 0
for t in doc.tables:
    enc = [c.text.strip() for c in t.rows[0].cells]
    if "Vida media" in enc and "Veredicto" in enc:
        i_vm, i_ver = enc.index("Vida media"), enc.index("Veredicto")
        for fila in t.rows[1:]:
            if fila.cells[i_ver].text.strip() != "Cointegracion":
                if fila.cells[i_vm].text.strip() not in ("", GUION, "-"):
                    escribir(fila.cells[i_vm], GUION)
                    arreglos += 1
        break
print("Vidas medias suprimidas en el cuadro 7: {}".format(arreglos))


# ------------------------------------------------------- 2. reubicar el anexo A1
body = doc.element.body
hijos = list(body)


def buscar(texto):
    for i, el in enumerate(hijos):
        if el.tag.endswith("}p"):
            t = "".join(n.text or "" for n in el.iter() if n.tag.endswith("}t"))
            if t.strip() == texto:
                return i
    return None


i_a1, i_ref = buscar("Cuadro A1"), buscar("Referencias")
if i_a1 is None or i_ref is None:
    print("AVISO: marcadores no localizados; el anexo no se movio")
elif i_a1 > i_ref:
    print("El anexo ya estaba al final")
else:
    bloque = hijos[i_a1:i_a1 + 5]
    if not any(el.tag.endswith("}tbl") for el in bloque):
        print("AVISO: estructura inesperada; el anexo no se movio")
    else:
        ancla = hijos[i_ref]
        for el in bloque:
            body.remove(el)
        for el in bloque:
            ancla.addprevious(el)
        print("Anexo A1 reubicado antes de las referencias")

doc.save(DESTINO)
print("\nGuardado: {} ({:,} bytes)".format(DESTINO, os.path.getsize(DESTINO)))

# ------------------------------------------------------------ 3. verificacion
v = Document(DESTINO)
orden = []
for p in v.paragraphs:
    s = p.text.strip()
    if s.startswith("Cuadro") or s == "Referencias":
        orden.append(s)
print("Orden de los cuadros:", " > ".join(orden))
print("Tablas:", len(v.tables))

from google.colab import files
files.download(DESTINO)

Vidas medias suprimidas en el cuadro 7: 3
Anexo A1 reubicado antes de las referencias

Guardado: /content/outputs/Cuadros_resultados_APA_v3.docx (52,053 bytes)
Orden de los cuadros: Cuadro 4 > Cuadro 5 > Cuadro 7 > Cuadro 8 > Cuadro 9 > Cuadro 10 > Cuadro A1 > Referencias
Tablas: 7


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
from google.colab import drive
drive.mount("/content/drive")

import shutil, os
BASE = "/content/drive/MyDrive/tesis_china_eeuu"
for sub in ["scripts", "data_clean", "outputs"]:
    os.makedirs(os.path.join(BASE, sub), exist_ok=True)

for f in glob.glob("/content/*.py"):
    if os.path.getsize(f) > 100:
        shutil.copy2(f, os.path.join(BASE, "scripts"))
for f in glob.glob("/content/data_clean/*") + glob.glob("/content/outputs/*"):
    dest = "data_clean" if "/data_clean/" in f else "outputs"
    shutil.copy2(f, os.path.join(BASE, dest))

print("Respaldo en", BASE)
for r, _, fs in os.walk(BASE):
    for f in fs:
        print("  ", os.path.relpath(os.path.join(r, f), BASE))

Mounted at /content/drive
Respaldo en /content/drive/MyDrive/tesis_china_eeuu
   scripts/generar_cuadros_apa.py
   data_clean/china_us_panel_analisis_v4_1990_2023.csv
   data_clean/china_us_panel_analisis_v3_1990_2023.csv
   data_clean/cobertura_series_nuevas.csv
   data_clean/china_us_panel_1990_2025.csv
   data_clean/china_us_panel_analisis_v1_1990_2023.csv
   data_clean/bitacora_reparacion.csv
   data_clean/diccionario_variables_v1.csv
   data_clean/generar_cuadros_apa.py
   data_clean/china_us_super_panel_1990_2025.csv
   data_clean/china_us_panel_analisis_v2_1990_2023.csv
   data_clean/china_us_super_panel_reparado.csv
   data_clean/china_us_panel_analisis_v5_1990_2023.csv
   outputs/B_coeficientes_v3.csv
   outputs/cuadro10_diagnosticos.csv
   outputs/A_diagnosticos_v3.csv
   outputs/cuadro8_largo_plazo.csv
   outputs/cuadro3b_segunda_ronda.csv
   outputs/A_bounds_ect.csv
   outputs/cuadro6b_vif_por_modelo.csv
   outputs/Cuadros_resultados_APA.docx
   outputs/B_corto_plazo.csv
  

In [62]:
# -*- coding: utf-8 -*-
import os
from docx import Document

LOCAL = "/content/outputs/Cuadros_resultados_APA_v3.docx"
DRIVE = "/content/drive/MyDrive/tesis_china_eeuu/outputs/Cuadros_resultados_APA_v3.docx"

print("INTEGRIDAD")
print("=" * 70)
for etiq, ruta in [("Local", LOCAL), ("Drive", DRIVE)]:
    if os.path.exists(ruta):
        print("{:<6} OK  {:,} bytes".format(etiq, os.path.getsize(ruta)))
    else:
        print("{:<6} AUSENTE".format(etiq))

RUTA = LOCAL if os.path.exists(LOCAL) else DRIVE
if not os.path.exists(RUTA):
    raise SystemExit("El documento no esta en ninguna de las dos rutas.")

d = Document(RUTA)
print("\nLeido desde:", RUTA)
print("Tablas: {} | Parrafos con texto: {}".format(
    len(d.tables), sum(1 for p in d.paragraphs if p.text.strip())))
print("Filas por tabla:", [len(t.rows) for t in d.tables])

print("\n\nPARRAFOS")
print("=" * 70)
for i, p in enumerate(d.paragraphs):
    if p.text.strip():
        print("P{:03d} | {}".format(i, p.text.strip()))

print("\n\nTABLAS")
print("=" * 70)
for k, t in enumerate(d.tables, 1):
    print("\n--- TABLA {} ({} filas x {} columnas) ---".format(
        k, len(t.rows), len(t.columns)))
    for fila in t.rows:
        print(" | ".join(c.text.strip() for c in fila.cells))

vacias = [(k, j) for k, t in enumerate(d.tables, 1)
          for j, f in enumerate(t.rows)
          for c in f.cells if not c.text.strip()]
print("\n\nCeldas vacias detectadas:", len(vacias))

INTEGRIDAD
Local  OK  52,053 bytes
Drive  OK  52,053 bytes

Leido desde: /content/outputs/Cuadros_resultados_APA_v3.docx
Tablas: 7 | Parrafos con texto: 29
Filas por tabla: [10, 9, 15, 23, 40, 15, 32]


PARRAFOS
P000 | Resultados de la estimacion. China y Estados Unidos, 1990-2023
P002 | Cuadro 4
P003 | Especificaciones estimadas, variables y correspondencia con las hipotesis de investigacion
P004 | Nota. Los tamanos de muestra corresponden a las observaciones disponibles antes de la generacion de rezagos y primeras diferencias. El guion indica que la especificacion no se estima para ese pais por indisponibilidad de la serie o por insuficiencia de grados de libertad. Elaboracion propia con datos de Penn World Table 10.01 (Feenstra, Inklaar y Timmer, 2015, https://doi.org/10.1257/aer.20130954), World Development Indicators del Banco Mundial (https://databank.worldbank.org/source/world-development-indicators), estadisticas de credito total del Bank for International Settlements (https://

In [63]:
# -*- coding: utf-8 -*-
import shutil, os
from docx import Document

RUTA = "/content/outputs/Cuadros_resultados_APA_v3.docx"
DRIVE = "/content/drive/MyDrive/tesis_china_eeuu/outputs/Cuadros_resultados_APA_v3.docx"

VIEJO = ("carece de interpretacion cuando el termino de correccion de error "
         "no es negativo")
NUEVO = ("solo se reporta en las especificaciones que satisfacen los dos "
         "criterios de cointegracion, pues carece de interpretacion cuando el "
         "termino de correccion de error es positivo o no difiere de cero de "
         "manera estadisticamente significativa")

doc = Document(RUTA)
hechos = 0
for p in doc.paragraphs:
    if VIEJO in p.text:
        nuevo_txt = p.text.replace(VIEJO, NUEVO)
        if p.runs:
            p.runs[0].text = nuevo_txt
            for r in p.runs[1:]:
                r.text = ""
        else:
            p.add_run(nuevo_txt)
        hechos += 1

if hechos:
    doc.save(RUTA)
    shutil.copy2(RUTA, DRIVE)
    print("Nota corregida en {} parrafo(s). Respaldo actualizado.".format(hechos))
    print("Tamano: {:,} bytes".format(os.path.getsize(RUTA)))
    v = Document(RUTA)
    for p in v.paragraphs:
        if p.text.strip().startswith("Nota.") and "vida media" in p.text.lower():
            print("\n" + p.text.strip())
            break
else:
    print("No se encontro la frase; el documento no se modifico.")

Nota corregida en 1 parrafo(s). Respaldo actualizado.
Tamano: 52,095 bytes

Nota. Estimacion mediante un modelo autorregresivo de rezagos distribuidos en su forma de correccion de error no restringida. Los valores criticos corresponden al caso de constante no restringida y tendencia ausente en Pesaran, Shin y Smith (2001, https://doi.org/10.1002/jae.616). Se declara cointegracion unicamente cuando el estadistico F supera el limite superior al 5 % y, de manera simultanea, el termino de correccion de error resulta negativo y estadisticamente significativo. La vida media se expresa en anos y se calcula como ln(0.5)/ln(1 + lambda); solo se reporta en las especificaciones que satisfacen los dos criterios de cointegracion, pues carece de interpretacion cuando el termino de correccion de error es positivo o no difiere de cero de manera estadisticamente significativa. Los ordenes de rezago se seleccionaron con el criterio de informacion de Akaike, con un orden minimo de uno para todos los regr